In [1]:
# Cell 1 - install Unsloth cho Kaggle

!pip install -q --upgrade pip

!pip install -q "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git"

!pip install -q --no-deps \
    trl \
    peft \
    accelerate \
    bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you hav

In [2]:
# ============================================================
# 2. Imports
# ============================================================


import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import json
import random
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments
# ============================================================
# 3. Config
# ============================================================

DATA_PATH = "/kaggle/input/datasets/anhphan8321/finetunedata/80.csv"
TEST_DATA_PATH = "/kaggle/input/datasets/anhphan8321/finetunedata/20.csv"
# Fallback local/Kaggle paths are handled by resolve_existing_path() below.

OUTPUT_DIR = "/kaggle/working/qwen25_coder_7b_lora"
EVAL_OUTPUT_PATH = "/kaggle/working/qwen25_coder_7b_lora_eval_20.csv"


MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct"

# Kaggle T4/P100 16GB usually cannot train Qwen2.5-Coder-7B with the original 2048/batch2/r16 config.
# Set this to False only on a larger GPU such as A100/L4 24GB+.
KAGGLE_T4_SAFE_MODE = True

MAX_SEQ_LENGTH = 1024 if KAGGLE_T4_SAFE_MODE else 2048
LOAD_IN_4BIT = True

# 80.csv đã dùng để train; 20.csv dùng để test sau khi train.
# Nếu cùng một id xuất hiện ở cả train/test, loại khỏi test để tránh leakage.
DROP_TEST_IDS_SEEN_IN_TRAIN = True

# None = đánh giá toàn bộ tập 20%; đổi thành 20/50 nếu muốn chạy thử nhanh.
EVAL_MAX_SAMPLES = None
GEN_MAX_NEW_TOKENS = 512

# Improve SFT signal quality. Keep raw test by default; set FILTER_NOISY_TEST=True for clean diagnostic metrics.
FILTER_NOISY_TRAIN = True
FILTER_NOISY_TEST = False

# Add a second final-answer-only training sample per row to improve final-answer/unit formatting.
ADD_FINAL_ONLY_TRAINING_EXAMPLES = False
DIRECT_FINAL_ANSWER_EVAL = False


# Optional: if planner_sft/planner_train.jsonl is uploaded as a Kaggle dataset,
# append it to SFT so Qwen learns the JSON planner schema used by Cell 13.
USE_PLANNER_SFT_IF_AVAILABLE = True
PLANNER_SFT_TRAIN_PATHS = [
    "/kaggle/input/planner-sft/planner_train.jsonl",
    "/kaggle/input/planner_sft/planner_train.jsonl",
    "/kaggle/input/physics-planner-sft/planner_train.jsonl",
    "/kaggle/working/planner_sft/planner_train.jsonl",
    "planner_sft/planner_train.jsonl",
]

torch_dtype = None

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.set_device(0)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

CUDA available: True
GPU: Tesla T4
BF16 supported: False


In [4]:
# ============================================================
# Cell 5. Load model and tokenizer
# ============================================================

# PRE-LOAD GPU CLEANUP
# If a previous run failed/OOMed, old model/trainer objects can still occupy VRAM.
for _name in [
    "model", "tokenizer", "trainer", "trainer_stats",
    "train_dataset", "eval_dataset", "eval_results_df",
]:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("GPU memory before loading model:")
    print(torch.cuda.memory_summary(device=0, abbreviated=True))

if torch.cuda.is_available():
    torch.cuda.set_device(0)
    MODEL_DEVICE_MAP = {"": torch.cuda.current_device()}
else:
    MODEL_DEVICE_MAP = None
print("Model device_map:", MODEL_DEVICE_MAP)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    device_map=MODEL_DEVICE_MAP,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Loaded:", MODEL_NAME)

GPU memory before loading model:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   8320 KiB |   8322 KiB |   8322 KiB |   2048 B   |
|---------------------------------------------------------------------------|
| Active memory         |   8320 KiB |   8322 KiB |   8322 KiB |   2048 B   |
|---------------------------------------------------------------------------|
| Requested memory      |   8320 KiB |   8320 KiB |   8320 KiB |    164 B   |
|------------------------------

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Loaded: unsloth/Qwen2.5-Coder-7B-Instruct


In [5]:
# ============================================================
# Cell 6. Prompt formatting
# ============================================================

SYSTEM_PROMPT = """You are a careful physics problem solver.
Solve the problem step by step.
Use Program-of-Thought style when useful: identify givens, formulas, unit conversions, calculations, and checks.
Use correct formulas, units, and signs.
At the end, provide the final answer clearly."""

ANSWER_ONLY_SYSTEM_PROMPT = """You are a careful physics answer generator.
Return only the final answer, including the correct unit when applicable.
Use exactly this format: Final answer: <value> <unit>."""

def normalize_unit(unit):
    unit = str(unit).strip()
    if unit.lower() in ["nan", "none", "null", ""]:
        return "-"
    return unit

def build_final_answer(answer, unit):
    answer = str(answer).strip()
    unit = normalize_unit(unit)

    if unit == "-":
        return answer
    return f"{answer} {unit}"

def build_text(row):
    question = str(row["question"]).strip()
    cot = str(row["cot"]).strip()
    answer = str(row["answer"]).strip()
    unit = normalize_unit(row["unit"])

    final_answer = build_final_answer(answer, unit)

    assistant_content = f"""{cot}

Final answer: {final_answer}"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question,
        },
        {
            "role": "assistant",
            "content": assistant_content,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def build_answer_only_text(row):
    question = str(row["question"]).strip()
    answer = str(row["answer"]).strip()
    unit = normalize_unit(row["unit"])
    final_answer = build_final_answer(answer, unit)

    messages = [
        {"role": "system", "content": ANSWER_ONLY_SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": f"Final answer: {final_answer}"},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


In [6]:
# Test formatting only. Training data is loaded formally in the next cell.
preview_df = pd.read_csv(DATA_PATH)
print(build_text(preview_df.iloc[0])[:2000])

<|im_start|>system
You are a careful physics problem solver.
Solve the problem step by step.
Use Program-of-Thought style when useful: identify givens, formulas, unit conversions, calculations, and checks.
Use correct formulas, units, and signs.
At the end, provide the final answer clearly.<|im_end|>
<|im_start|>user
A parallel-plate capacitor has a capacitance of 15.76 pF and is charged to a voltage of 91.6 V. Calculate the charge stored on the capacitor.<|im_end|>
<|im_start|>assistant
Step 1: Identify the given values from the question. The capacitance (C) is 15.76 pF and the voltage (V) is 91.6 V.
Step 2: Recall the formula for the charge (Q) stored on a capacitor, which is Q = C × V.
Step 3: Convert the capacitance from picofarads (pF) to farads (F): 15.76 pF = 15.76 × 10⁻¹² F.
Step 4: Substitute the values of capacitance and voltage into the formula: Q = (15.76 × 10⁻¹² F) × (91.6 V).
Step 5: The charge stored on the capacitor is approximately 1.44 nC.

Final answer: 1.44 nC<|im_e

In [7]:
# ============================================================
# Cell 7. Load train/test data
# ============================================================

def resolve_existing_path(primary_path, fallback_paths):
    candidates = [primary_path, *fallback_paths]
    for item in candidates:
        path = Path(str(item))
        if path.exists():
            return str(path)
    raise FileNotFoundError(
        "Could not find any dataset path. Tried:\n" + "\n".join(str(x) for x in candidates)
    )


train_path = resolve_existing_path(
    DATA_PATH,
    [
        "/kaggle/input/finetunedata/80.csv",
        "/kaggle/working/80.csv",
        "split_output/80.csv",
        "80.csv",
    ],
)
test_path = resolve_existing_path(
    TEST_DATA_PATH,
    [
        "/kaggle/input/finetunedata/20.csv",
        "/kaggle/working/20.csv",
        "split_output/20.csv",
        "20.csv",
    ],
)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

required_cols = ["id", "question", "cot", "answer", "unit"]
for name, df in [("train", train_df), ("test", test_df)]:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{name} dataset is missing columns: {missing}")
    for col in required_cols:
        df[col] = df[col].fillna("").astype(str)
    before = len(df)
    df.drop_duplicates(subset=["id", "question", "answer", "unit"], inplace=True)
    print(f"{name}: {before} rows -> {len(df)} rows after exact duplicate drop")


BLACKLIST_IDS = {
    "THCB113", "CH345", "THCB001", "LD047", "CH036", "CH077",
    "THCB066", "THCB067", "THCB076", "THCB087", "THCB088", "THCB090",
    "THCB093", "THCB094", "THCB097", "THCB098", "THCB100", "THCB104",
    "THCB107", "THCB108", "THCB110", "THCB114", "THCB117", "THCB118",
    "THCB120", "THCB123", "THCB127", "THCB128", "THCB130", "THCB133",
    "DDT340", "TD374", "TD376",
}

NOISE_KEYWORDS = [
    "image", ".png", ".jpg", "hình vẽ", "đồ thị bên", "hình bên",
    "translate", "translation", "cho em hỏi", "giải hộ e", "mng ơi",
    "please help", "với ạ", "incomplete", "contradiction",
]

def is_noisy_row(row):
    if str(row.get("id", "")).strip() in BLACKLIST_IDS:
        return True
    question = str(row.get("question", "") or "")
    cot = str(row.get("cot", "") or "")
    answer = str(row.get("answer", "") or "")
    if not question.strip() or not cot.strip() or not answer.strip():
        return True
    text = (question + " " + cot).lower()
    if any(keyword in text for keyword in NOISE_KEYWORDS):
        return True
    if len(question.split()) < 5 or len(cot.split()) < 10:
        return True
    if "step" not in cot.lower() and "bước" not in cot.lower():
        return True
    if question.strip().endswith("..."):
        return True
    return False

if FILTER_NOISY_TRAIN:
    before = len(train_df)
    train_df = train_df[~train_df.apply(is_noisy_row, axis=1)].reset_index(drop=True)
    print(f"Filtered noisy train rows: {before} -> {len(train_df)}")

if FILTER_NOISY_TEST:
    before = len(test_df)
    test_df = test_df[~test_df.apply(is_noisy_row, axis=1)].reset_index(drop=True)
    print(f"Filtered noisy test rows: {before} -> {len(test_df)}")

shared_ids = sorted(set(train_df["id"]) & set(test_df["id"]))
if shared_ids:
    print(f"WARNING: {len(shared_ids)} id(s) appear in both train and test: {shared_ids[:20]}")
    if DROP_TEST_IDS_SEEN_IN_TRAIN:
        test_df = test_df[~test_df["id"].isin(shared_ids)].reset_index(drop=True)
        print(f"Removed leaked ids from test. New test rows: {len(test_df)}")

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train path:", train_path)
print("Test path:", test_path)
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train prefixes:")
print(train_df["id"].str.extract(r"^([A-Za-z]+)")[0].value_counts().sort_index())
print("Test prefixes:")
print(test_df["id"].str.extract(r"^([A-Za-z]+)")[0].value_counts().sort_index())


train: 1081 rows -> 1081 rows after exact duplicate drop
test: 270 rows -> 270 rows after exact duplicate drop
Filtered noisy train rows: 1081 -> 1051
Train path: /kaggle/input/datasets/anhphan8321/finetunedata/80.csv
Test path: /kaggle/input/datasets/anhphan8321/finetunedata/20.csv
Train rows: 1051
Test rows: 270
Train prefixes:
0
CH      227
CHLT     17
DDT     103
DT       53
LD      315
NL      152
TD      140
THCB     44
Name: count, dtype: int64
Test prefixes:
0
CH      59
CHLT     3
DDT     26
DT      14
LD      79
NL      38
TD      35
THCB    16
Name: count, dtype: int64


In [8]:
# ============================================================
# Cell 8. Build HuggingFace datasets
# ============================================================

def build_dataset_from_df(df, include_final_only=False):
    texts = []
    for _, row in df.iterrows():
        texts.append(build_text(row))
        if include_final_only:
            texts.append(build_answer_only_text(row))
    return Dataset.from_pandas(pd.DataFrame({"text": texts}), preserve_index=False)


train_dataset = build_dataset_from_df(train_df, include_final_only=ADD_FINAL_ONLY_TRAINING_EXAMPLES)
eval_dataset = build_dataset_from_df(test_df, include_final_only=False)

print(train_dataset)
print(eval_dataset)
print("\nSample training text:\n")
print(train_dataset[0]["text"][:2500])


# Optional planner-SFT data.
# This teaches the same model to emit executable JSON plans for the Cell 13 planner_executor stage.
def resolve_optional_path(paths):
    for item in paths:
        path = Path(str(item))
        if path.exists():
            return path
    return None


def build_planner_dataset_from_jsonl(path):
    texts = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            messages = record.get("messages")
            if not messages:
                continue
            texts.append(
                tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            )
    return Dataset.from_pandas(pd.DataFrame({"text": texts}), preserve_index=False)


if USE_PLANNER_SFT_IF_AVAILABLE:
    planner_sft_path = resolve_optional_path(PLANNER_SFT_TRAIN_PATHS)
    if planner_sft_path is not None:
        from datasets import concatenate_datasets

        planner_dataset = build_planner_dataset_from_jsonl(planner_sft_path)
        if len(planner_dataset) > 0:
            train_dataset = concatenate_datasets([train_dataset, planner_dataset])
            print(f"Added planner-SFT examples: {len(planner_dataset)} from {planner_sft_path}")
            print("Combined train dataset:", train_dataset)
    else:
        print("No planner-SFT JSONL found; continuing with answer/explanation SFT only.")


Dataset({
    features: ['text'],
    num_rows: 1051
})
Dataset({
    features: ['text'],
    num_rows: 270
})

Sample training text:

<|im_start|>system
You are a careful physics problem solver.
Solve the problem step by step.
Use Program-of-Thought style when useful: identify givens, formulas, unit conversions, calculations, and checks.
Use correct formulas, units, and signs.
At the end, provide the final answer clearly.<|im_end|>
<|im_start|>user
Three charges q1 = q2 = 2.15 × 10^-6 C and q3 = 4.76 × 10^-6 C are placed at the three vertices of an equilateral triangle with a side length of 9.78 cm. Calculate the net electric field strength at the position of q3. Give your answer rounded to two decimal places.<|im_end|>
<|im_start|>assistant
Step 1: Identify the given values and constants. The charges producing the field at the position of q3 are q1 = 2.15 × 10⁻⁶ C and q2 = 2.15 × 10⁻⁶ C. The side length of the equilateral triangle is s = 9.78 cm. Coulomb's constant k = 8.99 × 10⁹ N×m

In [9]:
# ============================================================
# Cell 9. Configure LoRA with Unsloth
# ============================================================

LORA_R = 8 if KAGGLE_T4_SAFE_MODE else 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

if not getattr(model, "is_peft_model", False):
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
else:
    print("Model is already a PEFT/LoRA model; skipping get_peft_model().")

try:
    model.print_trainable_parameters()
except Exception as exc:
    print("Could not print trainable parameters:", repr(exc))


Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


In [10]:
# ============================================================
# Cell 10. Create SFTTrainer
# ============================================================

PER_DEVICE_TRAIN_BATCH_SIZE = 1 if KAGGLE_T4_SAFE_MODE else 2
GRADIENT_ACCUMULATION_STEPS = 8 if KAGGLE_T4_SAFE_MODE else 4
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 2e-4

print("Train config:", {
    "safe_mode": KAGGLE_T4_SAFE_MODE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
})

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    seed=SEED,
    report_to="none",
)


def make_sft_trainer():
    try:
        return SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LENGTH,
            packing=False,
            args=training_args,
        )
    except TypeError as exc:
        print("Retrying SFTTrainer with newer TRL signature:", exc)
        return SFTTrainer(
            model=model,
            processing_class=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            args=training_args,
        )


trainer = make_sft_trainer()

# Train only on assistant responses, not on system/user prompt tokens.
try:
    from unsloth.chat_templates import train_on_responses_only

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("Enabled response-only SFT loss.")
except Exception as exc:
    print("Could not enable response-only loss; continuing with full-text SFT:", repr(exc))


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train config: {'safe_mode': True, 'max_seq_length': 1024, 'lora_r': 8, 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8}


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1051 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/270 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1051 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/1051 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/270 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/270 [00:00<?, ? examples/s]

Enabled response-only SFT loss.


In [11]:
# # ============================================================
# # Cell 11. Train and save LoRA adapter
# # ============================================================

# def clear_gpu_memory(note=""):
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.ipc_collect()
#         free, total = torch.cuda.mem_get_info()
#         print(
#             f"GPU memory cleared{(' - ' + note) if note else ''}: "
#             f"{free / 1024**3:.2f} GiB free / {total / 1024**3:.2f} GiB total"
#         )


# RUN_TRAINING = True

# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# if RUN_TRAINING:
#     clear_gpu_memory("before trainer.train")
#     trainer_stats = trainer.train()
#     print(trainer_stats)

# model.save_pretrained(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# print("Saved LoRA adapter and tokenizer to:", OUTPUT_DIR)

# # Optional: uncomment if you need a merged 16-bit model and have enough disk/RAM.
# # MERGED_OUTPUT_DIR = "/kaggle/working/qwen25_coder_7b_lora_merged_16bit"
# # model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer, save_method="merged_16bit")


In [12]:
# ============================================================
# Cell 12. Evaluate on the 20% test set
# ============================================================

FastLanguageModel.for_inference(model)


def build_inference_prompt(row):
    messages = [
        {"role": "system", "content": ANSWER_ONLY_SYSTEM_PROMPT if DIRECT_FINAL_ANSWER_EVAL else SYSTEM_PROMPT},
        {"role": "user", "content": str(row["question"]).strip()},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def extract_final_answer(generated_text):
    text = str(generated_text).strip()
    matches = re.findall(r"final answer\s*:\s*(.+)", text, flags=re.I)
    if matches:
        return matches[-1].strip().splitlines()[0].strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else text


def parse_numeric_value(value):
    text = str(value).strip()
    if not text:
        return None
    text = text.split(";")[0].strip()
    text = text.replace("−", "-").replace("×", "x").replace("·", "*")
    text = text.replace("\\sqrt", "sqrt")
    text = re.sub(r"sqrt\{([^}]+)\}", r"sqrt(\1)", text)
    text = re.sub(r"(\d)\s*sqrt", r"\1*sqrt", text)
    sci = re.search(r"([+-]?\d+(?:\.\d+)?)\s*(?:x|\*)\s*10\s*\^?\s*([+-]?\d+)", text, re.I)
    if sci:
        return float(sci.group(1)) * (10 ** int(sci.group(2)))
    plain = re.search(r"[+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", text)
    if plain and not re.search(r"sqrt|\^|\*", text):
        return float(plain.group(0))
    expr = re.match(r"\s*([+\-0-9.eE\s/*^xX()sqrt]+)", text)
    if expr:
        try:
            import sympy as sp
            return float(sp.N(sp.sympify(expr.group(1).replace("x", "*").replace("X", "*").replace("^", "**"), locals={"sqrt": sp.sqrt})))
        except Exception:
            return None
    return None


def normalize_for_compare(text):
    text = str(text or "-").strip().lower()
    text = text.replace("µ", "μ").replace("ohm", "ω").replace("Ω", "ω")
    text = re.sub(r"\s+", "", text)
    return text


def answer_matches(pred, truth, rel_tol=0.03, abs_tol=1e-9):
    pred_num = parse_numeric_value(pred)
    truth_num = parse_numeric_value(truth)
    if pred_num is not None and truth_num is not None:
        return math.isclose(pred_num, truth_num, rel_tol=rel_tol, abs_tol=abs_tol)
    return normalize_for_compare(pred) == normalize_for_compare(truth)


def unit_matches(pred_final, truth_unit):
    truth = normalize_for_compare(truth_unit)
    if truth in {"", "-", "nan", "none"}:
        return True
    pred = normalize_for_compare(pred_final)
    aliases = {
        "degree": ["degree", "degrees", "deg", "°"],
        "ω": ["ω", "ohm"],
        "μf": ["μf", "uf"],
        "μc": ["μc", "uc"],
        "μj": ["μj", "uj"],
    }
    if truth in pred:
        return True
    for canonical, vals in aliases.items():
        if truth == canonical and any(v in pred for v in vals):
            return True
    return False


def generate_answer(row):
    prompt = build_inference_prompt(row)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


eval_source_df = test_df
if EVAL_MAX_SAMPLES is not None:
    eval_source_df = test_df.sample(n=min(EVAL_MAX_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

eval_rows = []
for idx, row in eval_source_df.iterrows():
    generated = generate_answer(row)
    final_answer = extract_final_answer(generated)
    answer_ok = answer_matches(final_answer, row["answer"])
    unit_ok = unit_matches(final_answer, row["unit"])
    eval_rows.append({
        "id": row["id"],
        "question": row["question"],
        "gold_answer": row["answer"],
        "gold_unit": row["unit"],
        "generated": generated,
        "pred_final_answer": final_answer,
        "answer_ok": answer_ok,
        "unit_ok": unit_ok,
        "pair_ok": bool(answer_ok and unit_ok),
    })
    if (idx + 1) % 10 == 0:
        print(f"Evaluated {idx + 1}/{len(eval_source_df)}")

eval_results_df = pd.DataFrame(eval_rows)
eval_results_df.to_csv(EVAL_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved eval results:", EVAL_OUTPUT_PATH)
print("Rows evaluated:", len(eval_results_df))
print("Answer accuracy:", eval_results_df["answer_ok"].mean() if len(eval_results_df) else None)
print("Unit accuracy:", eval_results_df["unit_ok"].mean() if len(eval_results_df) else None)
print("Pair accuracy:", eval_results_df["pair_ok"].mean() if len(eval_results_df) else None)

display(eval_results_df[["id", "gold_answer", "gold_unit", "pred_final_answer", "answer_ok", "unit_ok", "pair_ok"]].head(50))


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Evaluated 10/270


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Evaluated 20/270


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Evaluated 260/270


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Evaluated 270/270
Saved eval results: /kaggle/working/qwen25_coder_7b_lora_eval_20.csv
Rows evaluated: 270
Answer accuracy: 0.08888888888888889
Unit accuracy: 0.4
Pair accuracy: 0.06666666666666667


,id,gold_answer,gold_unit,pred_final_answer,answer_ok,unit_ok,pair_ok
0,TD371,9,times,The stored energy in the capacitor will increa...,False,True,False
1,TD386,decreases by half,-,"\[ C_{\text{eff},1} = 4 \times 8.85 \times 10^{-",False,True,False
2,TD062,22.74,pF,C = \frac{8.854 \times 10^{-12} \times 3.21 \,False,False,False
3,TD053,60.21,pF,"C = \frac{3.01 \times 10^{-14} \, \text{F} \cd...",False,False,False
4,TD166,3.005,nC,"Therefore, the charge on the capacitor is appr...",False,True,False
5,TD049,645.08,nJ,"Therefore, the electric field energy stored in...",False,False,False
6,TD398,1.66,nJ,"Therefore, the energy stored in the electric f...",False,False,False
7,TD057,4.69,nC,"\[ Q = 4.65743 \times 10^{-8} \, \text{C} \]",False,False,False
8,TD047,51.63,pF,C = \frac{3.69 \,False,False,False
9,TD087,2.54,nC,"Therefore, the charge stored by the capacitor ...",False,False,False


In [13]:
# ============================================================
# Cell 13. Hybrid deterministic-first evaluation
# ============================================================
# Priority 1: use the deterministic physics solver first.
# If it cannot solve a row, fall back to the fine-tuned Qwen model.

import importlib.util
import sys
from pathlib import Path

HYBRID_OUTPUT_PATH = "/kaggle/working/qwen25_coder_7b_lora_hybrid_eval_20.csv"
HYBRID_EVAL_MAX_SAMPLES = None  # None = full 20% test set

SOLVER_CODE = "#!/usr/bin/env python\n# coding: utf-8\n\"\"\"Baseline neuro-symbolic solver/auditor for datanhucc/physic.CSV.\n\nThis is intentionally a baseline, not the final solver. It does three jobs:\n1. profile the dataset and answer contracts;\n2. run deterministic family solvers where the relation model is confident;\n3. write per-row coverage/failure artifacts instead of silently guessing.\n\nThe key design guardrail is that Qwen/PoT is not trusted to decide geometry or\ncombine vectors when deterministic relations can be extracted.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport csv\nimport json\nimport math\nimport re\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Callable\n\n\nDEFAULT_DATA_PATH = Path(\"physic.CSV\")\nDEFAULT_OUTPUT_DIR = Path(\"physics_baseline\")\nCOULOMB_K = 9e9\nEPSILON_0 = 8.854e-12\nMU_0 = 4 * math.pi * 1e-7\n\nSUBSCRIPT_TRANS = str.maketrans(\"₀₁₂₃₄₅₆₇₈₉\", \"0123456789\")\nSUPERSCRIPT_TRANS = str.maketrans(\"⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻\", \"0123456789+-\")\n\n\ndef resolve_data_path(path: str | Path = DEFAULT_DATA_PATH) -> Path:\n    \"\"\"Resolve the dataset path from either the repo root or its parent folder.\"\"\"\n    requested = Path(path)\n    if requested.exists():\n        return requested\n\n    candidates = [\n        DEFAULT_DATA_PATH,\n        Path(\"datanhucc\") / DEFAULT_DATA_PATH,\n        Path(__file__).resolve().parents[1] / DEFAULT_DATA_PATH,\n        Path(__file__).resolve().parents[2] / \"datanhucc\" / DEFAULT_DATA_PATH,\n    ]\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f\"Could not find physics CSV at {path!s}.\")\n\n\n@dataclass\nclass Observation:\n    id: str\n    symbol: str\n    raw_value: str\n    value: float\n    unit: str\n    value_si: float\n    unit_si: str\n    quantity_type: str = \"unknown\"\n    text: str = \"\"\n\n\n@dataclass\nclass Relation:\n    id: str\n    type: str\n    data: dict[str, Any]\n\n\n@dataclass\nclass SolveResult:\n    pred_answer: str = \"\"\n    pred_unit: str = \"-\"\n    answer_type: str = \"numeric\"\n    status: str = \"unsupported\"\n    failure_reason: str = \"\"\n    family: str = \"unknown\"\n    trace: dict[str, Any] = field(default_factory=dict)\n\n\nUNIT_SCALE_TO_SI: dict[str, tuple[float, str]] = {\n    \"\": (1.0, \"\"),\n    \"-\": (1.0, \"-\"),\n    \"—\": (1.0, \"-\"),\n    \"C\": (1.0, \"C\"),\n    \"mC\": (1e-3, \"C\"),\n    \"uC\": (1e-6, \"C\"),\n    \"μC\": (1e-6, \"C\"),\n    \"µC\": (1e-6, \"C\"),\n    \"nC\": (1e-9, \"C\"),\n    \"pC\": (1e-12, \"C\"),\n    \"F\": (1.0, \"F\"),\n    \"mF\": (1e-3, \"F\"),\n    \"uF\": (1e-6, \"F\"),\n    \"μF\": (1e-6, \"F\"),\n    \"µF\": (1e-6, \"F\"),\n    \"nF\": (1e-9, \"F\"),\n    \"pF\": (1e-12, \"F\"),\n    \"H\": (1.0, \"H\"),\n    \"mH\": (1e-3, \"H\"),\n    \"uH\": (1e-6, \"H\"),\n    \"V\": (1.0, \"V\"),\n    \"mV\": (1e-3, \"V\"),\n    \"kV\": (1e3, \"V\"),\n    \"A\": (1.0, \"A\"),\n    \"mA\": (1e-3, \"A\"),\n    \"J\": (1.0, \"J\"),\n    \"mJ\": (1e-3, \"J\"),\n    \"uJ\": (1e-6, \"J\"),\n    \"μJ\": (1e-6, \"J\"),\n    \"µJ\": (1e-6, \"J\"),\n    \"nJ\": (1e-9, \"J\"),\n    \"W\": (1.0, \"W\"),\n    \"Hz\": (1.0, \"Hz\"),\n    \"kHz\": (1e3, \"Hz\"),\n    \"MHz\": (1e6, \"Hz\"),\n    \"ohm\": (1.0, \"ohm\"),\n    \"Ω\": (1.0, \"ohm\"),\n    \"kΩ\": (1e3, \"ohm\"),\n    \"m\": (1.0, \"m\"),\n    \"cm\": (1e-2, \"m\"),\n    \"mm\": (1e-3, \"m\"),\n    \"kg\": (1.0, \"kg\"),\n    \"g\": (1e-3, \"kg\"),\n    \"T\": (1.0, \"T\"),\n    \"Wb\": (1.0, \"Wb\"),\n    \"V/m\": (1.0, \"V/m\"),\n    \"kV/m\": (1e3, \"V/m\"),\n    \"N/C\": (1.0, \"V/m\"),\n    \"N\": (1.0, \"N\"),\n    \"%\": (1.0, \"%\"),\n    \"rad\": (1.0, \"rad\"),\n    \"degree\": (math.pi / 180.0, \"rad\"),\n    \"turns/m\": (1.0, \"turns/m\"),\n    \"°C\": (1.0, \"°C\"),\n}\n\nSI_TO_UNIT_SCALE = {unit: scale for unit, (scale, _) in UNIT_SCALE_TO_SI.items()}\n\n\ndef normalize_text(text: str) -> str:\n    text = (text or \"\").translate(SUBSCRIPT_TRANS)\n    supers = \"⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻\"\n    text = re.sub(\n        rf\"(?<=\\d)([{supers}]+)\",\n        lambda m: \"^\" + m.group(1).translate(SUPERSCRIPT_TRANS),\n        text,\n    )\n    text = text.translate(SUPERSCRIPT_TRANS)\n    text = text.replace(\"−\", \"-\").replace(\"–\", \"-\").replace(\"—\", \"-\")\n    text = text.replace(\"×\", \"x\").replace(\"·\", \"*\")\n    text = text.replace(\"µ\", \"μ\")\n    text = text.replace(\"Ω\", \"Ω\")\n    text = re.sub(r\"\\s+\", \" \", text)\n    return text.strip()\n\n\ndef normalize_unit(unit: str) -> str:\n    unit = normalize_text(unit).strip()\n    unit = unit.replace(\"micro\", \"μ\")\n    unit = unit.replace(\"Ohm\", \"Ω\").replace(\"ohms\", \"Ω\").replace(\"ohm\", \"Ω\")\n    unit = unit.replace(\"uF\", \"μF\").replace(\"uC\", \"μC\").replace(\"uJ\", \"μJ\")\n    if unit == \"\\u03bc\":\n        return \"μF\"\n    if unit in {\"\", \"-\", \"—\"}:\n        return \"-\"\n    return unit\n\n\ndef unit_to_si(value: float, unit: str) -> tuple[float, str]:\n    unit = normalize_unit(unit)\n    if unit not in UNIT_SCALE_TO_SI:\n        return value, unit\n    scale, si = UNIT_SCALE_TO_SI[unit]\n    return value * scale, si\n\n\ndef convert_from_si(value_si: float, target_unit: str) -> float:\n    target_unit = normalize_unit(target_unit)\n    scale = SI_TO_UNIT_SCALE.get(target_unit, 1.0)\n    if scale == 0:\n        return value_si\n    return value_si / scale\n\n\ndef choose_charge_unit(value_si: float, question: str = \"\") -> str:\n    explicit = infer_requested_unit(question, [\"μC\", \"nC\", \"pC\", \"mC\"])\n    if explicit:\n        return explicit\n    av = abs(value_si)\n    if av == 0:\n        return \"μC\"\n    if 1e-12 <= av < 1e-9:\n        return \"pC\"\n    if 1e-9 <= av < 1e-6:\n        return \"nC\"\n    if 1e-6 <= av < 1e-3:\n        return \"μC\"\n    if 1e-3 <= av < 1:\n        return \"mC\"\n    return \"C\"\n\n\ndef choose_energy_unit(value_si: float, question: str = \"\") -> str:\n    explicit = infer_requested_unit(question, [\"μJ\", \"nJ\", \"mJ\", \"J\"])\n    if explicit:\n        return explicit\n    av = abs(value_si)\n    if av == 0:\n        return \"μJ\"\n    if 1e-9 <= av < 1e-6:\n        return \"nJ\"\n    if 1e-6 <= av < 1e-3:\n        return \"μJ\"\n    if 1e-3 <= av < 1:\n        return \"mJ\"\n    return \"J\"\n\n\ndef safe_eval_expr(expr: str) -> float:\n    \"\"\"Evaluate a simple numeric/math expression with no names except math funcs.\"\"\"\n    allowed_funcs = {\n        \"sqrt\": math.sqrt,\n        \"sin\": math.sin,\n        \"cos\": math.cos,\n        \"tan\": math.tan,\n        \"pi\": math.pi,\n        \"abs\": abs,\n    }\n    node = ast.parse(expr, mode=\"eval\")\n    for child in ast.walk(node):\n        if isinstance(child, ast.Name) and child.id not in allowed_funcs:\n            raise ValueError(f\"unsafe name in numeric expression: {child.id}\")\n        if isinstance(child, ast.Call):\n            if not isinstance(child.func, ast.Name) or child.func.id not in allowed_funcs:\n                raise ValueError(\"unsafe call in numeric expression\")\n        if not isinstance(\n            child,\n            (\n                ast.Expression,\n                ast.BinOp,\n                ast.UnaryOp,\n                ast.Constant,\n                ast.Name,\n                ast.Load,\n                ast.Call,\n                ast.Add,\n                ast.Sub,\n                ast.Mult,\n                ast.Div,\n                ast.Pow,\n                ast.USub,\n                ast.UAdd,\n                ast.Mod,\n            ),\n        ):\n            raise ValueError(f\"unsafe syntax in numeric expression: {type(child).__name__}\")\n    return float(eval(compile(node, \"<number>\", \"eval\"), {\"__builtins__\": {}}, allowed_funcs))\n\n\ndef parse_number(raw: str) -> float | None:\n    if raw is None:\n        return None\n    expr = normalize_text(str(raw)).strip()\n    if not expr:\n        return None\n    expr = expr.translate(SUPERSCRIPT_TRANS)\n    expr = expr.replace(\"{\", \"(\").replace(\"}\", \")\")\n    expr = expr.replace(\"\\\\sqrt\", \"sqrt\").replace(\"√\", \"sqrt\")\n    expr = re.sub(r\"sqrt\\s*\\(?\\s*([0-9.]+)\\s*\\)?\", r\"sqrt(\\1)\", expr)\n    expr = expr.replace(\"\\\\pi\", \"pi\").replace(\"π\", \"pi\")\n    expr = re.sub(r\"\\\\frac\\s*\\(([^()]+)\\)\\s*\\(([^()]+)\\)\", r\"(\\1)/(\\2)\", expr)\n    expr = re.sub(r\"\\\\frac\\s+([0-9.]+)\\s+([0-9.]+)\", r\"(\\1)/(\\2)\", expr)\n    expr = expr.replace(\"^\", \"**\")\n    expr = re.sub(r\"10\\s*\\*\\*\\s*\\(\\s*([-+]?\\d+)\\s*\\)\", r\"10**\\1\", expr)\n    expr = re.sub(r\"(?<=\\d)\\s*x\\s*10\\s*\\*\\*\\s*([-+]?\\d+)\", r\"*10**\\1\", expr, flags=re.I)\n    expr = re.sub(r\"(?<=\\d)\\s*x\\s*10\\s*([-+]?\\d+)\", r\"*10**\\1\", expr, flags=re.I)\n    expr = re.sub(r\"\\)\\s*x\\s*10\\s*\\*\\*\\s*([-+]?\\d+)\", r\")*10**\\1\", expr, flags=re.I)\n    expr = re.sub(r\"\\)\\s*x\\s*10\\s*([-+]?\\d+)\", r\")*10**\\1\", expr, flags=re.I)\n    expr = re.sub(r\"(?<=\\d)\\s*x\\s*(?=\\d)\", \"*\", expr, flags=re.I)\n    expr = re.sub(r\"(?<=\\d)\\s*\\.\\s*10\\s*\\*\\*\\s*([-+]?\\d+)\", r\"*10**\\1\", expr, flags=re.I)\n    expr = re.sub(r\"(?<=\\d)\\s+10\\s*\\*\\*\\s*([-+]?\\d+)\", r\"*10**\\1\", expr, flags=re.I)\n    expr = re.sub(r\"([0-9.]+)\\s*sqrt\", r\"\\1*sqrt\", expr)\n    expr = re.sub(r\"\\)\\s*([0-9.]+)\", r\")*\\1\", expr)\n    expr = expr.strip()\n    expr = re.sub(r\"[^0-9eE+\\-*/(). piqsrtabc]\", \"\", expr)\n    try:\n        return safe_eval_expr(expr)\n    except Exception:\n        m = re.search(r\"[-+]?\\d+(?:\\.\\d+)?(?:[eE][-+]?\\d+)?\", expr)\n        if m:\n            try:\n                return float(m.group(0))\n            except ValueError:\n                return None\n    return None\n\n\nNUMBER_RE = r\"(?:[-+]?10\\s*(?:\\^|\\*\\*)\\s*[-+]?\\d+|[-+]?\\d+(?:\\.\\d+)?(?:\\s*√\\s*\\d+)?(?:\\s*(?:x|\\*)\\s*10\\s*(?:\\^|\\*\\*)?\\s*[-+]?\\d+|[eE][-+]?\\d+)?)\"\nUNIT_TOKENS = [\n    \"V/m\",\n    \"N/C\",\n    \"turns/m\",\n    \"μF\",\n    \"µF\",\n    \"uF\",\n    \"mF\",\n    \"nF\",\n    \"pF\",\n    \"μC\",\n    \"µC\",\n    \"uC\",\n    \"mC\",\n    \"nC\",\n    \"pC\",\n    \"mH\",\n    \"uH\",\n    \"kΩ\",\n    \"ohm\",\n    \"MHz\",\n    \"kHz\",\n    \"mJ\",\n    \"μJ\",\n    \"µJ\",\n    \"uJ\",\n    \"nJ\",\n    \"kV\",\n    \"mV\",\n    \"mA\",\n    \"cm\",\n    \"mm\",\n    \"kg\",\n    \"°C\",\n    \"Ω\",\n    \"C\",\n    \"F\",\n    \"Hz\",\n    \"H\",\n    \"N\",\n    \"J\",\n    \"V\",\n    \"A\",\n    \"m\",\n    \"g\",\n    \"T\",\n    \"Wb\",\n    \"W\",\n    \"%\",\n]\nUNIT_RE = \"|\".join(re.escape(tok) for tok in UNIT_TOKENS)\n\n\ndef format_number(value: float, digits: int = 6) -> str:\n    if value is None or not math.isfinite(value):\n        return \"\"\n    if abs(value) >= 1e4 or (0 < abs(value) < 1e-3):\n        return f\"{value:.{digits}g}\"\n    text = f\"{value:.{digits}f}\".rstrip(\"0\").rstrip(\".\")\n    return text if text else \"0\"\n\n\ndef format_number_for_question(value: float, question: str, digits: int = 6) -> str:\n    text = normalize_text(question).lower()\n    m = re.search(r\"round(?:ed)?(?: the result)? to (\\w+|\\d+) decimal places?\", text)\n    if not m:\n        return format_number(value, digits)\n    raw = m.group(1)\n    words = {\"one\": 1, \"two\": 2, \"three\": 3, \"four\": 4}\n    places = int(raw) if raw.isdigit() else words.get(raw, digits)\n    return f\"{value:.{places}f}\"\n\n\ndef split_multi_values(text: str) -> list[str]:\n    return [part.strip() for part in re.split(r\";|,|\\band\\b\", text or \"\", flags=re.I) if part.strip()]\n\n\ndef normalize_answer_text_for_compare(text: str) -> str:\n    text = normalize_text(text).lower()\n    text = text.replace(\"halfed\", \"halved\")\n    text = text.replace(\"doubled\", \"double\")\n    text = re.sub(r\"\\s+\", \" \", text)\n    return text.strip()\n\n\ndef classify_answer_type(answer: str, unit: str) -> str:\n    answer = (answer or \"\").strip()\n    unit = normalize_unit(unit)\n    lower = answer.lower()\n    if unit == \"-\" and lower in {\"yes\", \"no\"}:\n        return \"yes_no\"\n    if unit == \"-\" and (\n        any(tok in answer for tok in [\"=\", \"/\", \"√\", \"^\", \"π\", \"\\\\frac\"])\n        or re.search(r\"\\b[EFRUI]\\d?\\b\", answer)\n    ):\n        return \"math_expression\"\n    if any(tok in answer for tok in [\"√\", \"π\", \"\\\\frac\"]) or re.search(r\"\\b[EFRUI]\\d?\\s*=\", answer):\n        return \"math_expression\"\n    if re.search(r\"[A-Za-z]\", answer) and not re.search(r\"(?i)e[-+]?\\d+\", answer):\n        return \"text\"\n    return \"numeric\"\n\n\ndef answer_subshape(answer: str, unit: str) -> str:\n    if \";\" in (answer or \"\") or \";\" in (unit or \"\"):\n        return \"multi_value\"\n    if any(tok in (answer or \"\") for tok in [\"√\", \"π\", \"\\\\frac\"]):\n        return \"symbolic_numeric\"\n    if re.search(r\"(?i)(x|\\*)\\s*10|\\d[eE][-+]?\\d\", answer or \"\"):\n        return \"scientific\"\n    if classify_answer_type(answer, unit) != \"numeric\":\n        return classify_answer_type(answer, unit)\n    return \"scalar\"\n\n\ndef row_prefix(row_id: str) -> str:\n    m = re.match(r\"[A-Za-z]+\", row_id or \"\")\n    return m.group(0) if m else \"UNKNOWN\"\n\n\ndef detect_reasoning_flags(question: str, cot: str = \"\") -> dict[str, bool]:\n    text = normalize_text(f\"{question} {cot}\")\n    flags = {\n        \"needs_unit_conversion\": bool(\n            re.search(r\"\\b(cm|mm|mC|nC|pC|pF|nF|mH|mJ|nJ|kV|MHz)\\b|μ|10\\^-\", text, re.I)\n        ),\n        \"needs_geometry_relation\": bool(\n            re.search(\n                r\"triangle|right[- ]angled|equilateral|isosceles|midpoint|perpendicular|\"\n                r\"between|line segment|points? [A-Z]|vertices|AB\\s*=|AC\\s*=|BC\\s*=|MA\\s*=|MB\\s*=\",\n                text,\n                re.I,\n            )\n        ),\n        \"needs_vector_composition\": bool(\n            re.search(r\"net|resultant|vector|direction|magnitude|force acting|electric field\", text, re.I)\n        ),\n        \"needs_circuit_relation\": bool(\n            re.search(r\"series|parallel|RLC|AC|impedance|reactance|resonance|phase|power factor\", text, re.I)\n        ),\n        \"needs_measurement_error\": bool(\n            re.search(r\"error|uncertainty|measurement|least count|relative error|absolute error|mean|average\", text, re.I)\n        ),\n        \"needs_qualitative_text\": bool(\n            re.search(r\"does|is it|which|what happens|towards|direction|shape|graph|yes|no\", text, re.I)\n        ),\n    }\n    return flags\n\n\ndef classify_family(question: str) -> str:\n    text = normalize_text(question).lower()\n    if re.search(r\"least count|relative error|absolute error|mean absolute|average .*error|measured|measurement\", text):\n        return \"measurement\"\n    if re.search(r\"\\bresonance\\b|rlc|reactance|impedance|power factor|capacitive|inductive|rms|alternating current|ac circuit\", text):\n        if re.search(r\"\\bdoes\\b|\\bis it\\b|will .*resonance|determine if\", text):\n            return \"rlc_resonance_yes_no\"\n        return \"ac_rlc\"\n    if re.search(r\"capacitor|capacitance|parallel-plate\", text):\n        return \"capacitor\"\n    if re.search(r\"point charge|charges?|electric field|electric force|coulomb|test charge\", text):\n        if re.search(r\"field\", text):\n            return \"electrostatics_field\"\n        return \"electrostatics_force\"\n    if re.search(r\"inductor|magnetic energy|solenoid|flux|induced\", text):\n        return \"magnetic_induction\"\n    if re.search(r\"energy|power|work|heat\", text):\n        return \"energy_power\"\n    return \"unknown\"\n\n\ndef extract_quantity_observations(question: str) -> list[Observation]:\n    text = normalize_text(question)\n    observations: list[Observation] = []\n    seen: set[tuple[str, str, str]] = set()\n\n    # Chained assignments: q1 = q2 = q3 = 1.6 x 10^-19 C.\n    chained = re.finditer(\n        rf\"((?:q[A-Za-z0-9]+\\s*=\\s*){{2,}})\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        text,\n        flags=re.I,\n    )\n    for match in chained:\n        symbols = re.findall(r\"q[A-Za-z0-9]+\", match.group(1))\n        raw_value, unit = match.group(2), normalize_unit(match.group(3))\n        value = parse_number(raw_value)\n        if value is None:\n            continue\n        value_si, unit_si = unit_to_si(value, unit)\n        for symbol in symbols:\n            key = (symbol, raw_value, unit)\n            if key in seen:\n                continue\n            seen.add(key)\n            observations.append(\n                Observation(\n                    id=f\"obs_{len(observations):03d}_{symbol}\",\n                    symbol=symbol,\n                    raw_value=raw_value,\n                    value=value,\n                    unit=unit,\n                    value_si=value_si,\n                    unit_si=unit_si,\n                    quantity_type=guess_quantity_type(symbol, unit, text),\n                    text=match.group(0),\n                )\n            )\n\n    # Standard assignments: C = 100 μF, AB = 5 cm, qA = 5 μC.\n    assign_re = re.compile(\n        rf\"\\b([A-Za-z][A-Za-z0-9]*)\\s*=\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        flags=re.I,\n    )\n    for match in assign_re.finditer(text):\n        symbol, raw_value, unit = match.group(1), match.group(2), normalize_unit(match.group(3))\n        value = parse_number(raw_value)\n        if value is None:\n            continue\n        key = (symbol, raw_value, unit)\n        if key in seen:\n            continue\n        seen.add(key)\n        value_si, unit_si = unit_to_si(value, unit)\n        observations.append(\n            Observation(\n                id=f\"obs_{len(observations):03d}_{symbol}\",\n                symbol=symbol,\n                raw_value=raw_value,\n                value=value,\n                unit=unit,\n                value_si=value_si,\n                unit_si=unit_si,\n                quantity_type=guess_quantity_type(symbol, unit, text),\n                text=match.group(0),\n            )\n        )\n\n    # Natural language distances: points A and B ... 8 cm apart / separated by 20 cm.\n    apart_re = re.compile(\n        rf\"points?\\s+([A-Z])\\s+and\\s+([A-Z]).{{0,80}}?(?:apart|separated by)\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        flags=re.I,\n    )\n    for match in apart_re.finditer(text):\n        a, b, raw_value, unit = match.group(1).upper(), match.group(2).upper(), match.group(3), normalize_unit(match.group(4))\n        symbol = \"\".join(sorted([a, b]))\n        value = parse_number(raw_value)\n        if value is None:\n            continue\n        key = (symbol, raw_value, unit)\n        if key in seen:\n            continue\n        seen.add(key)\n        value_si, unit_si = unit_to_si(value, unit)\n        observations.append(\n            Observation(\n                id=f\"obs_{len(observations):03d}_{symbol}\",\n                symbol=symbol,\n                raw_value=raw_value,\n                value=value,\n                unit=unit,\n                value_si=value_si,\n                unit_si=unit_si,\n                quantity_type=\"distance\",\n                text=match.group(0),\n            )\n        )\n\n    apart_after_number_re = re.compile(\n        rf\"points?\\s+([A-Z])\\s+and\\s+([A-Z]).{{0,80}}?({NUMBER_RE})\\s*({UNIT_RE})\\s*(?:apart|separated)\",\n        flags=re.I,\n    )\n    for match in apart_after_number_re.finditer(text):\n        a, b, raw_value, unit = match.group(1).upper(), match.group(2).upper(), match.group(3), normalize_unit(match.group(4))\n        symbol = \"\".join(sorted([a, b]))\n        value = parse_number(raw_value)\n        if value is None:\n            continue\n        key = (symbol, raw_value, unit)\n        if key in seen:\n            continue\n        seen.add(key)\n        value_si, unit_si = unit_to_si(value, unit)\n        observations.append(\n            Observation(\n                id=f\"obs_{len(observations):03d}_{symbol}\",\n                symbol=symbol,\n                raw_value=raw_value,\n                value=value,\n                unit=unit,\n                value_si=value_si,\n                unit_si=unit_si,\n                quantity_type=\"distance\",\n                text=match.group(0),\n            )\n        )\n\n    # Measured lists without symbol assignment.\n    for i, match in enumerate(re.finditer(rf\"({NUMBER_RE})\\s*({UNIT_RE})\", text, flags=re.I)):\n        raw_value, unit = match.group(1), normalize_unit(match.group(2))\n        value = parse_number(raw_value)\n        if value is None:\n            continue\n        key = (f\"num_{i}\", raw_value, unit)\n        if key in seen:\n            continue\n        # Avoid duplicating assignment values too aggressively; keep natural-list values for THCB.\n        if any(abs(obs.value - value) < 1e-15 and obs.unit == unit and match.group(0) in obs.text for obs in observations):\n            continue\n        value_si, unit_si = unit_to_si(value, unit)\n        observations.append(\n            Observation(\n                id=f\"obs_{len(observations):03d}_num\",\n                symbol=f\"num_{i}\",\n                raw_value=raw_value,\n                value=value,\n                unit=unit,\n                value_si=value_si,\n                unit_si=unit_si,\n                quantity_type=guess_quantity_type(\"\", unit, text),\n                text=match.group(0),\n            )\n        )\n\n    return observations\n\n\ndef guess_quantity_type(symbol: str, unit: str, context: str) -> str:\n    s = symbol.lower()\n    unit = normalize_unit(unit)\n    _, unit_si = unit_to_si(1.0, unit)\n    if unit in {\"C\", \"mC\", \"μC\", \"nC\", \"pC\"} or s.startswith(\"q\"):\n        return \"charge\"\n    if unit in {\"m\", \"cm\", \"mm\"} or re.fullmatch(r\"[A-Z]{2}\", symbol):\n        return \"distance\"\n    if unit in {\"F\", \"μF\", \"nF\", \"pF\"} or s == \"c\":\n        return \"capacitance\"\n    if unit == \"H\" or s == \"l\":\n        return \"inductance\"\n    if unit == \"V\" or s in {\"u\", \"uab\"}:\n        return \"voltage\"\n    if unit == \"A\" or s == \"i\":\n        return \"current\"\n    if unit in {\"Ω\", \"ohm\"} or s == \"r\":\n        return \"resistance\"\n    if unit == \"Hz\" or s == \"f\":\n        return \"frequency\"\n    if unit_si == \"J\":\n        return \"energy\"\n    if unit_si == \"W\":\n        return \"power\"\n    if unit_si == \"T\":\n        return \"magnetic_field\"\n    if unit_si == \"Wb\":\n        return \"magnetic_flux\"\n    return \"unknown\"\n\n\ndef extract_relations(question: str, observations: list[Observation]) -> list[Relation]:\n    text = normalize_text(question)\n    relations: list[Relation] = []\n    distance_obs = [obs for obs in observations if obs.quantity_type == \"distance\" and re.fullmatch(r\"[A-Z]{2}\", obs.symbol)]\n    for obs in distance_obs:\n        a, b = obs.symbol[0], obs.symbol[1]\n        relations.append(\n            Relation(\n                id=f\"rel_{len(relations):03d}_{a}{b}\",\n                type=\"distance\",\n                data={\"points\": [a, b], \"distance_si\": obs.value_si, \"observation_id\": obs.id},\n            )\n        )\n\n    for match in re.finditer(r\"right[- ]angled\\s+at\\s+([A-Z])\", text, flags=re.I):\n        point = match.group(1).upper()\n        relations.append(Relation(id=f\"rel_{len(relations):03d}_right_{point}\", type=\"right_angle\", data={\"point\": point}))\n\n    tri = re.search(r\"triangle\\s+([A-Z])([A-Z])([A-Z])\", text, flags=re.I)\n    if tri:\n        points = [tri.group(1).upper(), tri.group(2).upper(), tri.group(3).upper()]\n        shape = \"triangle\"\n        if re.search(r\"equilateral\", text, re.I):\n            shape = \"equilateral_triangle\"\n        elif re.search(r\"isosceles\", text, re.I):\n            shape = \"isosceles_triangle\"\n        relations.append(Relation(id=f\"rel_{len(relations):03d}_shape\", type=\"shape\", data={\"shape\": shape, \"points\": points}))\n        if shape == \"equilateral_triangle\":\n            side_obs = next(\n                (\n                    obs\n                    for obs in observations\n                    if obs.quantity_type == \"distance\" and not re.fullmatch(r\"[A-Z]{2}\", obs.symbol)\n                ),\n                None,\n            )\n            if side_obs:\n                for a, b in [(points[0], points[1]), (points[0], points[2]), (points[1], points[2])]:\n                    relations.append(\n                        Relation(\n                            id=f\"rel_{len(relations):03d}_{a}{b}\",\n                            type=\"distance\",\n                            data={\"points\": [a, b], \"distance_si\": side_obs.value_si, \"observation_id\": side_obs.id, \"source\": \"equilateral_side\"},\n                        )\n                    )\n\n    for match in re.finditer(r\"([A-Z])\\s+(?:is\\s+)?(?:the\\s+)?midpoint\\s+of\\s+([A-Z])([A-Z])\", text, flags=re.I):\n        relations.append(\n            Relation(\n                id=f\"rel_{len(relations):03d}_midpoint\",\n                type=\"midpoint\",\n                data={\"point\": match.group(1).upper(), \"of\": [match.group(2).upper(), match.group(3).upper()]},\n            )\n        )\n\n    for match in re.finditer(r\"midpoint\\s+of\\s+(?:the\\s+)?(?:line segment\\s+|line\\s+connecting\\s+)?([A-Z])\\s+and\\s+([A-Z])\", text, flags=re.I):\n        relations.append(\n            Relation(\n                id=f\"rel_{len(relations):03d}_midpoint\",\n                type=\"midpoint\",\n                data={\"point\": \"M\", \"of\": [match.group(1).upper(), match.group(2).upper()], \"source\": \"implicit_midpoint\"},\n            )\n        )\n\n    if re.search(r\"midpoint\\s+of\\s+(?:the\\s+)?line\\s+connecting\\s+the\\s+two\\s+charges\", text, re.I):\n        relations.append(\n            Relation(\n                id=f\"rel_{len(relations):03d}_midpoint\",\n                type=\"midpoint\",\n                data={\"point\": \"M\", \"of\": [\"A\", \"B\"], \"source\": \"two_charge_midpoint\"},\n            )\n        )\n\n    if re.search(r\"midpoint\", text, re.I) and re.search(r\"q1.*q2|AB\", text, re.I):\n        relations.append(\n            Relation(\n                id=f\"rel_{len(relations):03d}_midpoint\",\n                type=\"midpoint\",\n                data={\"point\": \"M\", \"of\": [\"A\", \"B\"], \"source\": \"generic_q1_q2_midpoint\"},\n            )\n        )\n\n    for match in re.finditer(rf\"distance\\s+from\\s+([A-Z])\\s+to\\s+([A-Z])\\s+is\\s+({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I):\n        p1, p2 = match.group(1).upper(), match.group(2).upper()\n        value = parse_number(match.group(3))\n        unit = normalize_unit(match.group(4))\n        if value is not None:\n            value_si, unit_si = unit_to_si(value, unit)\n            if unit_si == \"m\":\n                relations.append(\n                    Relation(\n                        id=f\"rel_{len(relations):03d}_{p1}{p2}\",\n                        type=\"distance\",\n                        data={\"points\": [p1, p2], \"distance_si\": value_si, \"source\": \"distance_from_point_to_point\"},\n                    )\n                )\n\n    for match in re.finditer(rf\"({NUMBER_RE})\\s*({UNIT_RE})\\s+from\\s+([A-Z])\", text, re.I):\n        value = parse_number(match.group(1))\n        unit = normalize_unit(match.group(2))\n        point = match.group(3).upper()\n        if value is not None and point in {\"A\", \"B\", \"C\"} and re.search(r\"\\b(point\\s+)?M\\b\", text):\n            value_si, unit_si = unit_to_si(value, unit)\n            if unit_si == \"m\":\n                relations.append(\n                    Relation(\n                        id=f\"rel_{len(relations):03d}_M{point}\",\n                        type=\"distance\",\n                        data={\"points\": [\"M\", point], \"distance_si\": value_si, \"source\": \"distance_from_named_point_to_M\"},\n                    )\n                )\n\n    has_ab = any(rel.type == \"distance\" and set(rel.data.get(\"points\", [])) == {\"A\", \"B\"} for rel in relations)\n    if not has_ab and re.search(r\"q1.*q2|two electric charges|two point charges\", text, re.I) and re.search(r\"apart|separated\", text, re.I):\n        side_obs = next(\n            (\n                obs\n                for obs in observations\n                if obs.quantity_type == \"distance\" and not re.fullmatch(r\"[A-Z]{2}\", obs.symbol)\n            ),\n            None,\n        )\n        if side_obs:\n            relations.append(\n                Relation(\n                    id=f\"rel_{len(relations):03d}_AB\",\n                    type=\"distance\",\n                    data={\"points\": [\"A\", \"B\"], \"distance_si\": side_obs.value_si, \"observation_id\": side_obs.id, \"source\": \"two_charge_separation\"},\n                )\n            )\n\n    distances_after_ab = relation_distances(relations)\n    ab = distances_after_ab.get(frozenset([\"A\", \"B\"]))\n    if ab and re.search(r\"perpendicular bisector of (?:the )?(?:line segment )?AB|perpendicular bisector of AB\", text, re.I):\n        h_match = re.search(rf\"({NUMBER_RE})\\s*({UNIT_RE})\\s+(?:away\\s+)?from\\s+(?:the\\s+)?(?:line segment\\s+)?AB\", text, re.I)\n        if h_match:\n            h_value = parse_number(h_match.group(1))\n            h_unit = normalize_unit(h_match.group(2))\n            if h_value is not None:\n                h_si, h_si_unit = unit_to_si(h_value, h_unit)\n                if h_si_unit == \"m\":\n                    ma = math.sqrt((ab / 2) ** 2 + h_si**2)\n                    for end in [\"A\", \"B\"]:\n                        relations.append(\n                            Relation(\n                                id=f\"rel_{len(relations):03d}_M{end}_perp_bisector\",\n                                type=\"distance\",\n                                data={\"points\": [\"M\", end], \"distance_si\": ma, \"source\": \"perpendicular_bisector\"},\n                            )\n                        )\n\n    enriched = infer_geometry_relations(relations)\n    relations.extend(enriched)\n    return relations\n\n\ndef relation_distances(relations: list[Relation]) -> dict[frozenset[str], float]:\n    distances: dict[frozenset[str], float] = {}\n    for rel in relations:\n        if rel.type == \"distance\":\n            points = rel.data.get(\"points\", [])\n            if len(points) == 2:\n                distances[frozenset(points)] = float(rel.data[\"distance_si\"])\n    return distances\n\n\ndef infer_geometry_relations(relations: list[Relation]) -> list[Relation]:\n    inferred: list[Relation] = []\n    distances = relation_distances(relations)\n\n    for rel in relations:\n        if rel.type != \"midpoint\":\n            continue\n        point = rel.data.get(\"point\")\n        endpoints = rel.data.get(\"of\", [])\n        if not point or len(endpoints) != 2:\n            continue\n        a, b = endpoints\n        ab = distances.get(frozenset([a, b]))\n        if not ab:\n            continue\n        half = ab / 2\n        for end in [a, b]:\n            if frozenset([point, end]) not in distances:\n                inferred.append(\n                    Relation(\n                        id=f\"rel_infer_midpoint_distance_{point}{end}\",\n                        type=\"distance\",\n                        data={\"points\": [point, end], \"distance_si\": half, \"source\": \"midpoint_half_distance\"},\n                    )\n                )\n\n    # Derive missing side in a right triangle when the right-angle point and two sides are known.\n    for rel in relations:\n        if rel.type != \"right_angle\":\n            continue\n        p = rel.data[\"point\"]\n        points = sorted(set().union(*[set(k) for k in distances.keys()]))\n        if len(points) < 3 or p not in points:\n            continue\n        for a in points:\n            for b in points:\n                if a >= b or a == p or b == p:\n                    continue\n                pa = distances.get(frozenset([p, a]))\n                pb = distances.get(frozenset([p, b]))\n                ab = distances.get(frozenset([a, b]))\n                if ab and pa and pb:\n                    continue\n                if ab and pa and not pb and ab > pa:\n                    val = math.sqrt(max(ab * ab - pa * pa, 0))\n                    inferred.append(\n                        Relation(\n                            id=f\"rel_infer_distance_{p}{b}\",\n                            type=\"distance\",\n                            data={\"points\": [p, b], \"distance_si\": val, \"source\": \"right_angle_pythagorean\"},\n                        )\n                    )\n                if ab and pb and not pa and ab > pb:\n                    val = math.sqrt(max(ab * ab - pb * pb, 0))\n                    inferred.append(\n                        Relation(\n                            id=f\"rel_infer_distance_{p}{a}\",\n                            type=\"distance\",\n                            data={\"points\": [p, a], \"distance_si\": val, \"source\": \"right_angle_pythagorean\"},\n                        )\n                    )\n\n    # Classify collinearity/between and included angles from triangle side lengths.\n    all_distances = relation_distances(relations + inferred)\n    points = sorted(set().union(*[set(k) for k in all_distances.keys()])) if all_distances else []\n    for i, p in enumerate(points):\n        for j, q in enumerate(points):\n            for r in points:\n                if len({p, q, r}) != 3 or not (i < j):\n                    continue\n                pq = all_distances.get(frozenset([p, q]))\n                pr = all_distances.get(frozenset([p, r]))\n                qr = all_distances.get(frozenset([q, r]))\n                if not (pq and pr and qr):\n                    continue\n                if abs((pr + qr) - pq) <= 1e-9 * max(1.0, pq):\n                    inferred.append(\n                        Relation(\n                            id=f\"rel_infer_between_{r}_{p}{q}\",\n                            type=\"between\",\n                            data={\"point\": r, \"endpoints\": [p, q], \"source\": \"distance_sum\"},\n                        )\n                    )\n                # angle at p between q and r.\n                cos_val = (pq * pq + pr * pr - qr * qr) / (2 * pq * pr)\n                cos_val = max(-1.0, min(1.0, cos_val))\n                inferred.append(\n                    Relation(\n                        id=f\"rel_infer_angle_{q}{p}{r}\",\n                        type=\"included_angle\",\n                        data={\"vertex\": p, \"arms\": [q, r], \"cos\": cos_val, \"radians\": math.acos(cos_val)},\n                    )\n                )\n    return inferred\n\n\ndef map_charges_to_points(question: str, observations: list[Observation]) -> dict[str, str]:\n    text = normalize_text(question)\n    mapping: dict[str, str] = {}\n    charge_symbols = [obs.symbol for obs in observations if obs.quantity_type == \"charge\"]\n\n    for sym in charge_symbols:\n        if re.fullmatch(r\"q[A-Z]\", sym):\n            mapping[sym] = sym[-1].upper()\n\n    # q1 and q2 placed at points A and B respectively.\n    m = re.search(r\"(?:charges?|point charges?),?\\s+(q[A-Za-z0-9]+).*?(?:and|,)\\s*(q[A-Za-z0-9]+).*?points?\\s+([A-Z])\\s+and\\s+([A-Z])\", text, re.I)\n    if m:\n        mapping[m.group(1)] = m.group(3).upper()\n        mapping[m.group(2)] = m.group(4).upper()\n\n    if \"points A and B\" in text and \"q1\" in charge_symbols and \"q2\" in charge_symbols:\n        mapping.setdefault(\"q1\", \"A\")\n        mapping.setdefault(\"q2\", \"B\")\n\n    if not mapping and {\"q1\", \"q2\"} <= set(charge_symbols) and re.search(r\"apart|separated\", text, re.I):\n        mapping[\"q1\"] = \"A\"\n        mapping[\"q2\"] = \"B\"\n\n    # q1 ... at A and q2 ... at B, respectively.\n    m = re.search(r\"(q[A-Za-z0-9]+).*?(q[A-Za-z0-9]+).*?placed\\s+at\\s+([A-Z])\\s+and\\s+([A-Z]).*?respectively\", text, re.I)\n    if m:\n        mapping[m.group(1)] = m.group(3).upper()\n        mapping[m.group(2)] = m.group(4).upper()\n\n    # A third charge q3 is placed at point C / A charge q0 is placed at M.\n    for m in re.finditer(r\"(?:charge|test charge|third charge)[,\\s]+(q[A-Za-z0-9]+).*?placed\\s+at\\s+(?:point\\s+)?([A-Z])\", text, re.I):\n        mapping[m.group(1)] = m.group(2).upper()\n\n    for m in re.finditer(r\"(?:test charge|charge)[,\\s]+(q)\\b.*?placed\\s+at\\s+(?:point\\s+)?([A-Z])\", text, re.I):\n        if m.group(1) in charge_symbols:\n            mapping[m.group(1)] = m.group(2).upper()\n\n    for m in re.finditer(r\"(?:test charge|charge|third charge)[,\\s]+(q[A-Za-z0-9]*|q)\\b.*?placed\\s+at\\s+(?:the\\s+)?midpoint\", text, re.I):\n        if m.group(1) in charge_symbols:\n            mapping[m.group(1)] = \"M\"\n\n    # The charges are qA, qB, qC respectively after triangle ABC.\n    tri = re.search(r\"triangle\\s+([A-Z])([A-Z])([A-Z])\", text, re.I)\n    if tri:\n        points = [tri.group(i).upper() for i in range(1, 4)]\n        syms = [s for s in charge_symbols if re.fullmatch(r\"q[A-Z]\", s)]\n        for sym in syms:\n            mapping[sym] = sym[-1].upper()\n        numbered = [s for s in [\"q1\", \"q2\", \"q3\"] if s in charge_symbols]\n        if len(numbered) == 3 and re.search(r\"vertices|triangle\", text, re.I):\n            for sym, point in zip(numbered, points):\n                mapping.setdefault(sym, point)\n        if not syms and len(charge_symbols) >= 3 and \"respectively\" in text.lower():\n            for sym, point in zip(charge_symbols[:3], points):\n                mapping[sym] = point\n\n    return mapping\n\n\ndef identify_force_target(question: str, charge_to_point: dict[str, str]) -> str | None:\n    text = normalize_text(question)\n    m = re.search(r\"(?:acting on|on)\\s+(q[A-Za-z0-9]*|q)\\b\", text, re.I)\n    if m:\n        return m.group(1)\n    m = re.search(r\"exerted by .*? on (q[A-Za-z0-9]*|q)\\b\", text, re.I)\n    if m:\n        return m.group(1)\n    m = re.search(r\"charge\\s+at\\s+([A-Z])\", text, re.I)\n    if m:\n        point = m.group(1).upper()\n        for charge, charge_point in charge_to_point.items():\n            if charge_point == point:\n                return charge\n    m = re.search(r\"(?:test charge|point charge|charge)\\s+(q[A-Za-z0-9]*|q)\\b\", text, re.I)\n    if m and m.group(1) in charge_to_point:\n        return m.group(1)\n    return None\n\n\ndef identify_field_target_point(question: str) -> str | None:\n    text = normalize_text(question)\n    if re.search(r\"midpoint\\s+of\\s+(?:the\\s+)?(?:line segment\\s+|line\\s+connecting\\s+)?[A-Z]\\s+and\\s+[A-Z]\", text, re.I):\n        return \"M\"\n    if re.search(r\"midpoint\\s+of\\s+(?:the\\s+)?line\\s+connecting\\s+the\\s+two\\s+charges\", text, re.I):\n        return \"M\"\n    patterns = [\n        r\"at\\s+point\\s+([A-Z])\",\n        r\"at\\s+vertex\\s+([A-Z])\",\n        r\"at\\s+([A-Z])\\b\",\n        r\"point\\s+([A-Z])\\s+located\",\n    ]\n    for pat in patterns:\n        matches = re.findall(pat, text, re.I)\n        if matches:\n            return matches[-1].upper()\n    return None\n\n\ndef get_obs_by_symbol(observations: list[Observation]) -> dict[str, Observation]:\n    return {obs.symbol: obs for obs in observations}\n\n\ndef extract_angle_radians(question: str) -> float | None:\n    text = normalize_text(question)\n    if re.search(r\"perpendicular|right angle|90°|90 degrees?\", text, re.I):\n        return math.pi / 2\n    m = re.search(rf\"(?:angle of|at an angle of|angle between .*? is)\\s*({NUMBER_RE})\\s*(?:°|degree|degrees)?\", text, re.I)\n    if not m:\n        m = re.search(rf\"({NUMBER_RE})\\s*(?:°|degree|degrees)\\s*(?:to each other|between)\", text, re.I)\n    if not m:\n        return None\n    value = parse_number(m.group(1))\n    if value is None:\n        return None\n    return math.radians(value)\n\n\ndef solve_two_vector_resultant(question: str, observations: list[Observation], unit: str) -> SolveResult | None:\n    values = [obs.value_si for obs in observations if obs.unit_si == unit]\n    if len(values) == 1 and re.search(r\"\\beach\\b|same magnitude\", normalize_text(question), re.I):\n        values = [values[0], values[0]]\n    if len(values) < 2:\n        return None\n    angle = extract_angle_radians(question)\n    if angle is None:\n        return None\n    a, b = values[0], values[1]\n    result = math.sqrt(max(0.0, a * a + b * b + 2 * a * b * math.cos(angle)))\n    family = classify_family(question)\n    pred_unit = \"N\" if unit == \"N\" else \"V/m\"\n    return SolveResult(\n        family=family,\n        answer_type=\"numeric\",\n        pred_answer=format_number(result),\n        pred_unit=pred_unit,\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_two_vector_resultant\", \"formula\": \"R=sqrt(A^2+B^2+2AB cos(theta))\", \"angle\": angle, \"components\": values[:2]},\n    )\n\n\ndef first_distance_value(observations: list[Observation]) -> float | None:\n    for obs in observations:\n        if obs.quantity_type == \"distance\" and obs.unit_si == \"m\":\n            return obs.value_si\n    return None\n\n\ndef distance_value_by_symbol(observations: list[Observation], symbol: str) -> float | None:\n    symbol = symbol.upper()\n    for obs in observations:\n        if obs.quantity_type == \"distance\" and obs.unit_si == \"m\" and obs.symbol.upper() == symbol:\n            return obs.value_si\n    return None\n\n\ndef distance_symbol_map(observations: list[Observation]) -> dict[str, float]:\n    return {\n        obs.symbol.upper(): obs.value_si\n        for obs in observations\n        if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"\n    }\n\n\ndef distance_between_points(dmap: dict[str, float], a: str, b: str) -> float | None:\n    return dmap.get((a + b).upper()) or dmap.get((b + a).upper())\n\n\ndef two_source_charge_values(question: str, observations: list[Observation]) -> list[float]:\n    charges = [obs.value_si for obs in observations if obs.quantity_type == \"charge\"]\n    if len(charges) >= 2:\n        return charges[:2]\n    text = normalize_text(question)\n    if len(charges) == 1:\n        q = abs(charges[0])\n        if re.search(r\"q1\\s*=\\s*-\\s*q2\\s*=\", text, re.I):\n            return [q, -q]\n        if re.search(r\"q1\\s*=\\s*q2\\s*=\", text, re.I):\n            return [q, q]\n    return charges\n\n\ndef parse_distance_value_si(raw_value: str, raw_unit: str) -> float | None:\n    value = parse_number(raw_value)\n    if value is None:\n        return None\n    value_si, unit_si = unit_to_si(value, raw_unit)\n    return value_si if unit_si == \"m\" else None\n\n\ndef point_field_vector(q_si: float, source: tuple[float, float], target: tuple[float, float], dielectric: float = 1.0) -> tuple[float, float] | None:\n    dx = target[0] - source[0]\n    dy = target[1] - source[1]\n    r = math.hypot(dx, dy)\n    if r == 0:\n        return None\n    scale = COULOMB_K * q_si / (dielectric * r**3)\n    return scale * dx, scale * dy\n\n\ndef force_on_target_vector(q_source_si: float, q_target_si: float, source: tuple[float, float], target: tuple[float, float]) -> tuple[float, float] | None:\n    field = point_field_vector(q_source_si, source, target)\n    if field is None:\n        return None\n    return q_target_si * field[0], q_target_si * field[1]\n\n\ndef format_with_requested_distance_unit(value_si: float, question: str) -> tuple[str, str]:\n    unit = infer_requested_unit(question, [\"cm\", \"mm\", \"m\"]) or \"cm\"\n    return format_number(convert_from_si(value_si, unit)), unit\n\n\ndef parse_lambda_si(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(rf\"(?:lambda|Î»|λ)\\s*=\\s*({NUMBER_RE})\\s*C\\s*/\\s*m\", text, re.I)\n    if not m:\n        return None\n    return parse_number(m.group(1))\n\n\ndef parse_surface_density_si(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(rf\"(?:sigma|Ïƒ|σ)\\s*=\\s*({NUMBER_RE})\\s*C\\s*/\\s*m\\^?2\", text, re.I)\n    if not m:\n        return None\n    return parse_number(m.group(1))\n\n\ndef parse_speed_si(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(rf\"(?:velocity|speed).*?({NUMBER_RE})\\s*km\\s*/\\s*s\", text, re.I)\n    if m:\n        value = parse_number(m.group(1))\n        return value * 1000.0 if value is not None else None\n    m = re.search(rf\"(?:velocity|speed).*?({NUMBER_RE})\\s*m\\s*/\\s*s\", text, re.I)\n    if m:\n        return parse_number(m.group(1))\n    return None\n\n\ndef parse_instantaneous_cos_current(question: str) -> float | None:\n    text = normalize_text(question)\n    current = re.search(rf\"\\bI\\s*=\\s*({NUMBER_RE})\\s*cos\\s*\\(\\s*({NUMBER_RE})\\s*t\\s*\\)\", text, re.I)\n    time = re.search(rf\"\\bt\\s*=\\s*({NUMBER_RE})\\s*s\\b\", text, re.I)\n    if not current or not time:\n        return None\n    amplitude = parse_number(current.group(1))\n    omega = parse_number(current.group(2))\n    t_value = parse_number(time.group(1))\n    if amplitude is None or omega is None or t_value is None:\n        return None\n    return amplitude * math.cos(omega * t_value)\n\n\ndef two_charge_axis_zero_field(question: str, charges: list[Observation], observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if not re.search(r\"field .*zero|electric field vector is zero|resultant electric field strength .*zero\", text):\n        return None\n    if len(charges) < 2:\n        return None\n    d = first_distance_value(observations)\n    if not d:\n        return None\n    q1, q2 = charges[0].value_si, charges[1].value_si\n    a, b = math.sqrt(abs(q1)), math.sqrt(abs(q2))\n    if a == 0 or b == 0:\n        return None\n    if q1 * q2 > 0:\n        x_from_a = d * a / (a + b)\n        answer_si = x_from_a\n    else:\n        if abs(a - b) <= 1e-12:\n            return None\n        x_from_a = a * d / (a - b)\n        if \"distance from b\" in text or \"from b\" in text:\n            answer_si = abs(d - x_from_a)\n        else:\n            answer_si = x_from_a\n    value, unit = format_with_requested_distance_unit(answer_si, question)\n    return SolveResult(\n        family=\"electrostatics_field\",\n        answer_type=\"numeric\",\n        pred_answer=value,\n        pred_unit=unit,\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_axis_field_zero\", \"formula\": \"|q1|/x^2=|q2|/(d-x)^2\", \"x_from_a_si\": x_from_a},\n    )\n\n\ndef solve_uniform_field_kinematics(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"electron\" not in text or \"velocity reduces to zero\" not in text:\n        return None\n    e_field = next((obs.value_si for obs in observations if obs.unit_si == \"V/m\"), None)\n    if e_field is None:\n        match = re.search(rf\"electric field strength\\s*E\\s*=\\s*({NUMBER_RE})\\s*V\\s*/\\s*m\", normalize_text(question), re.I)\n        e_field = parse_number(match.group(1)) if match else None\n    v0 = parse_speed_si(question)\n    if e_field is None or v0 is None:\n        return None\n    electron_mass = 9.1e-31\n    electron_charge = 1.6e-19\n    distance_si = electron_mass * v0 * v0 / (2 * electron_charge * e_field)\n    value, unit = format_with_requested_distance_unit(distance_si, question)\n    return SolveResult(\n        family=\"electrostatics_field\",\n        answer_type=\"numeric\",\n        pred_answer=value,\n        pred_unit=unit,\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_uniform_field_kinematics\", \"formula\": \"qEs=0.5*m*v0^2\", \"distance_si\": distance_si},\n    )\n\n\ndef solve_force_balance_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"equilibrium\" not in text or \"dust particle\" not in text:\n        return None\n    mass = next((obs.value_si for obs in observations if obs.quantity_type == \"unknown\" and obs.unit_si == \"kg\"), None)\n    charge = next((abs(obs.value_si) for obs in observations if obs.quantity_type == \"charge\"), None)\n    g_match = re.search(rf\"\\bg\\s*=\\s*({NUMBER_RE})\", normalize_text(question), re.I)\n    g = parse_number(g_match.group(1)) if g_match else 9.8\n    if mass is None or charge is None or not charge or g is None:\n        return None\n    e_field = mass * g / charge\n    return SolveResult(\n        family=\"electrostatics_field\",\n        answer_type=\"numeric\",\n        pred_answer=format_number(e_field),\n        pred_unit=\"V/m\",\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_force_balance_field\", \"formula\": \"qE=mg\", \"mass\": mass, \"charge\": charge, \"g\": g},\n    )\n\n\ndef solve_inverse_point_charge_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"what value must charge\" in text and \"centroid\" in text and \"equilateral\" in text:\n        source = next((obs for obs in observations if obs.quantity_type == \"charge\"), None)\n        if source:\n            return SolveResult(\n                family=\"electrostatics_field\",\n                answer_type=\"numeric\",\n                pred_answer=source.raw_value,\n                pred_unit=\"C\",\n                status=\"ok\",\n                trace={\"planner\": \"deterministic_equilateral_centroid_zero\", \"formula\": \"symmetry gives q3=q1=q2\"},\n            )\n    if \"points towards the charge\" not in text and \"points toward the charge\" not in text:\n        return None\n    e_field = next((obs.value_si for obs in observations if obs.unit_si == \"V/m\"), None)\n    if e_field is None:\n        match = re.search(rf\"magnitude of\\s*({NUMBER_RE})\\s*V\\s*/\\s*m\", normalize_text(question), re.I)\n        e_field = parse_number(match.group(1)) if match else None\n    distance = first_distance_value(observations)\n    dielectric = extract_dielectric_constant(question) or 1.0\n    if e_field is None or distance is None:\n        return None\n    q_si = -abs(e_field * dielectric * distance * distance / COULOMB_K)\n    return SolveResult(\n        family=\"electrostatics_field\",\n        answer_type=\"numeric\",\n        pred_answer=format_number(q_si),\n        pred_unit=\"C\",\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_inverse_point_charge_field\", \"formula\": \"q=-E*epsilon_r*r^2/k\", \"dielectric\": dielectric},\n    )\n\n\ndef solve_continuous_distribution_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"infinitely long straight wire\" in text:\n        lam = parse_lambda_si(question)\n        distances = [obs.value_si for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n        r = distances[-1] if distances else None\n        if lam is None or r is None:\n            return None\n        e_field = 2 * COULOMB_K * abs(lam) / r\n        return SolveResult(family=\"electrostatics_field\", answer_type=\"numeric\", pred_answer=format_number(e_field), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_line_charge_field\", \"formula\": \"E=2k|lambda|/r\"})\n\n    if \"circular conducting disk\" in text or \"conducting disk\" in text:\n        sigma = parse_surface_density_si(question)\n        distance_values = [obs.value_si for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n        if sigma is None or len(distance_values) < 2:\n            return None\n        radius, z = distance_values[0], distance_values[1]\n        e_field = abs(sigma) / (2 * EPSILON_0) * (1 - z / math.sqrt(z * z + radius * radius))\n        return SolveResult(family=\"electrostatics_field\", answer_type=\"numeric\", pred_answer=format_number(e_field), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_charged_disk_axis_field\", \"formula\": \"E=sigma/(2eps0)*(1-z/sqrt(z^2+R^2))\"})\n\n    return None\n\n\ndef solve_symbolic_field_scaling(question: str) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"charge is replaced by\" in text and \"distance\" in text and \"halved\" in text and \"magnitude\" in text:\n        return SolveResult(\n            family=\"electrostatics_field\",\n            answer_type=\"numeric\",\n            pred_answer=\"8E\",\n            pred_unit=\"V/m\",\n            status=\"ok\",\n            trace={\"planner\": \"deterministic_field_scaling\", \"formula\": \"E proportional to |q|/r^2, factor=2/(1/2)^2=8\"},\n        )\n    return None\n\n\ndef solve_square_fourth_vertex_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"square\" not in text or \"fourth vertex\" not in text:\n        return None\n    charge = next((abs(obs.value_si) for obs in observations if obs.quantity_type == \"charge\"), None)\n    side = first_distance_value(observations)\n    if charge is None or side is None:\n        return None\n    e_side = COULOMB_K * charge / (side * side)\n    e_diag = COULOMB_K * charge / (2 * side * side)\n    result = math.sqrt(2) * e_side + e_diag\n    return SolveResult(family=\"electrostatics_field\", answer_type=\"numeric\", pred_answer=format_number(result), pred_unit=\"N/C\", status=\"ok\", trace={\"planner\": \"deterministic_square_three_charges_fourth_vertex\", \"formula\": \"E=sqrt(2)*kq/a^2+kq/(2a^2)\"})\n\n\ndef solve_collinear_three_charge_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"collinear\" not in text or not (\"point m\" in text or \"point n\" in text):\n        return None\n    charges = [obs for obs in observations if obs.quantity_type == \"charge\"]\n    if len(charges) < 3:\n        return None\n    step = first_distance_value(observations)\n    if step is None:\n        return None\n    target_x = 3 * step if re.search(r\"calculate.*point\\s+n\", text, re.I) else -step\n    positions = [0.0, step, 2 * step]\n    e = 0.0\n    for charge, x0 in zip(charges[:3], positions):\n        dx = target_x - x0\n        if dx == 0:\n            return None\n        e += COULOMB_K * charge.value_si * dx / abs(dx) ** 3\n    return SolveResult(family=\"electrostatics_field\", answer_type=\"numeric\", pred_answer=format_number(abs(e)), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_collinear_three_charge_field\", \"target_x\": target_x, \"formula\": \"signed 1D point-charge superposition\"})\n\n\ndef solve_equilateral_vertex_resultant(question: str, observations: list[Observation], family: str) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    has_q3_target = re.search(r\"position of q3|acting on q3|at q3\", text)\n    has_named_field_target = family == \"electrostatics_field\" and re.search(r\"at point\\s+[a-z]|triangle\\s+[a-z]*ab\", text)\n    if \"equilateral\" not in text or not (has_q3_target or has_named_field_target):\n        return None\n    charges = [obs for obs in observations if obs.quantity_type == \"charge\"]\n    charge_values = two_source_charge_values(question, observations)\n    side = first_distance_value(observations)\n    if len(charge_values) < 2 or side is None:\n        return None\n    height = math.sqrt(3) * side / 2\n    target = (0.0, height)\n    sources = [(-side / 2, 0.0), (side / 2, 0.0)]\n    vectors = []\n    if family == \"electrostatics_force\":\n        if len(charges) < 3:\n            return None\n        q_target = charges[2].value_si\n        for charge, point in zip(charges[:2], sources):\n            vec = force_on_target_vector(charge.value_si, q_target, point, target)\n            if vec:\n                vectors.append(vec)\n    else:\n        for charge_value, point in zip(charge_values[:2], sources):\n            vec = point_field_vector(charge_value, point, target)\n            if vec:\n                vectors.append(vec)\n    if not vectors:\n        return None\n    ex = sum(v[0] for v in vectors)\n    ey = sum(v[1] for v in vectors)\n    result = math.hypot(ex, ey)\n    return SolveResult(\n        family=family,\n        answer_type=\"numeric\",\n        pred_answer=format_number(result),\n        pred_unit=\"N\" if family == \"electrostatics_force\" else \"V/m\",\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_equilateral_vertex_resultant\", \"formula\": \"vectors at third vertex\", \"net\": [ex, ey]},\n    )\n\n\ndef solve_right_isosceles_vertex(question: str, observations: list[Observation], family: str) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"right\" not in text or \"isosceles\" not in text or not re.search(r\"right[- ]angle(?: vertex)?|right angle\", text):\n        return None\n    charge = next((abs(obs.value_si) for obs in observations if obs.quantity_type == \"charge\"), None)\n    leg = first_distance_value(observations)\n    if charge is None or leg is None:\n        return None\n    if family == \"electrostatics_force\":\n        result = math.sqrt(2) * COULOMB_K * charge * charge / (leg * leg)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(result), pred_unit=\"N\", status=\"ok\", trace={\"planner\": \"deterministic_right_isosceles_vertex_force\", \"formula\": \"F=sqrt(2)*kq^2/a^2\"})\n    result = math.sqrt(2) * COULOMB_K * charge / (leg * leg)\n    return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(result), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_right_isosceles_vertex_field\", \"formula\": \"E=sqrt(2)*kq/a^2\"})\n\n\ndef solve_collinear_two_source_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question)\n    lower = text.lower()\n    if \"electric field\" not in lower and \"field strength\" not in lower:\n        return None\n    charge_values = two_source_charge_values(question, observations)\n    if len(charge_values) < 2:\n        return None\n    dmap = distance_symbol_map(observations)\n    d_ab = distance_between_points(dmap, \"A\", \"B\") or dmap.get(\"A\") or first_distance_value(observations)\n    r_a = None\n    r_b = None\n\n    if \"NA\" in dmap and \"NB\" in dmap:\n        r_a, r_b = dmap[\"NA\"], dmap[\"NB\"]\n    elif distance_between_points(dmap, \"A\", \"C\") is not None and distance_between_points(dmap, \"B\", \"C\") is not None:\n        r_a = distance_between_points(dmap, \"A\", \"C\")\n        r_b = distance_between_points(dmap, \"B\", \"C\")\n    else:\n        m = re.search(\n            rf\"({NUMBER_RE})\\s*({UNIT_RE})\\s+from\\s+A\\s+and\\s+({NUMBER_RE})\\s*({UNIT_RE})\\s+from\\s+B\",\n            text,\n            re.I,\n        )\n        if m:\n            r_a = parse_distance_value_si(m.group(1), m.group(2))\n            r_b = parse_distance_value_si(m.group(3), m.group(4))\n\n    if r_a is None:\n        m = re.search(rf\"({NUMBER_RE})\\s*(cm|mm|m)\\s*(?:away from q1|from A)\\b\", text, re.I)\n        if m:\n            r_a = parse_distance_value_si(m.group(1), m.group(2))\n\n    if d_ab is None or r_a is None:\n        return None\n\n    outside = \"outside the segment\" in lower\n    if r_b is None:\n        r_b = r_a + d_ab if outside else abs(d_ab - r_a)\n    if r_b is None or r_a <= 0 or r_b <= 0:\n        return None\n\n    if abs(r_b - (r_a + d_ab)) <= max(1e-9, 1e-6 * max(r_a, r_b, d_ab)):\n        target_x = -r_a\n    elif abs(r_b - abs(d_ab - r_a)) <= max(1e-9, 1e-6 * max(r_a, r_b, d_ab)):\n        target_x = r_a\n    elif outside:\n        target_x = -r_a\n    else:\n        return None\n\n    e_signed = 0.0\n    for q_value, source_x in zip(charge_values[:2], [0.0, d_ab]):\n        dx = target_x - source_x\n        if dx == 0:\n            return None\n        e_signed += COULOMB_K * q_value * dx / abs(dx) ** 3\n    return SolveResult(\n        family=\"electrostatics_field\",\n        answer_type=\"numeric\",\n        pred_answer=format_number(abs(e_signed)),\n        pred_unit=\"V/m\",\n        status=\"ok\",\n        trace={\n            \"planner\": \"deterministic_collinear_two_source_field\",\n            \"formula\": \"signed 1D electric-field superposition\",\n            \"target_x\": target_x,\n            \"r_a\": r_a,\n            \"r_b\": r_b,\n            \"d_ab\": d_ab,\n        },\n    )\n\n\ndef solve_angled_two_source_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"angle\" not in text or \"field\" not in text:\n        return None\n    if not re.search(r\"central point|from a central point|away from .* point\", text):\n        return None\n    charge_values = two_source_charge_values(question, observations)\n    distances = [obs.value_si for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n    angle = extract_angle_radians(question)\n    if len(charge_values) < 2 or not distances or angle is None:\n        return None\n    r = distances[0]\n    if r <= 0:\n        return None\n    unit_sources = [(1.0, 0.0), (math.cos(angle), math.sin(angle))]\n    ex = 0.0\n    ey = 0.0\n    for q_value, unit_vec in zip(charge_values[:2], unit_sources):\n        scale = -COULOMB_K * q_value / (r * r)\n        ex += scale * unit_vec[0]\n        ey += scale * unit_vec[1]\n    result = math.hypot(ex, ey)\n    return SolveResult(\n        family=\"electrostatics_field\",\n        answer_type=\"numeric\",\n        pred_answer=format_number(result),\n        pred_unit=\"V/m\",\n        status=\"ok\",\n        trace={\"planner\": \"deterministic_angled_two_source_field\", \"formula\": \"vector superposition from equal source-target radius and included angle\", \"net\": [ex, ey]},\n    )\n\n\ndef solve_two_source_target_geometry(question: str, observations: list[Observation], family: str) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    charges = [obs for obs in observations if obs.quantity_type == \"charge\"]\n    distance_values = [obs.value_si for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n    if len(distance_values) < 1:\n        return None\n\n    if re.search(r\"q1\\s*=\\s*q2\\s*=\\s*q\", text) and \"find q\" in text and \"force\" in text and len(distance_values) >= 1:\n        force = next((obs.value_si for obs in observations if obs.unit_si == \"N\"), None)\n        if force is not None:\n            q_si = math.sqrt(force * distance_values[0] ** 2 / COULOMB_K)\n            unit = choose_charge_unit(q_si, question)\n            return SolveResult(family=\"electrostatics_force\", answer_type=\"numeric\", pred_answer=format_number(convert_from_si(q_si, unit)), pred_unit=unit, status=\"ok\", trace={\"planner\": \"deterministic_inverse_coulomb_equal_charges\", \"formula\": \"q=sqrt(F*r^2/k)\"})\n\n    if family == \"electrostatics_force\" and \"opposite sides\" in text and len(charges) >= 2 and len(distance_values) >= 2:\n        target = charges[0]\n        source_q = abs(charges[1].value_si)\n        r1, r2 = distance_values[0], distance_values[1]\n        result = COULOMB_K * abs(target.value_si) * source_q * abs(1 / (r1 * r1) - 1 / (r2 * r2))\n        return SolveResult(family=\"electrostatics_force\", answer_type=\"numeric\", pred_answer=format_number(result), pred_unit=\"N\", status=\"ok\", trace={\"planner\": \"deterministic_opposite_side_two_force\", \"formula\": \"F=k|q q0||1/r1^2-1/r2^2|\"})\n\n    if len(charges) < 3 and family == \"electrostatics_force\":\n        return None\n    if len(two_source_charge_values(question, observations)) < 2 and family == \"electrostatics_field\":\n        return None\n\n    # Perpendicular/equidistant target with known source separation and source-target distance.\n    if \"perpendicular bisector\" in text and len(charges) >= 2 and len(distance_values) >= 2:\n        d = distance_value_by_symbol(observations, \"AB\") or distance_values[0]\n        offset = distance_values[-1]\n        if offset <= 0:\n            return None\n        if re.search(r\"\\bfrom\\s+AB\\b\", normalize_text(question), re.I) and re.search(r\"\\b(?:distance|away)\\b\", normalize_text(question), re.I):\n            height = offset\n        else:\n            height = math.sqrt(max(0.0, offset * offset - (d / 2) ** 2))\n        target = (0.0, height)\n        sources = [(-d / 2, 0.0), (d / 2, 0.0)]\n        if family == \"electrostatics_field\":\n            vectors = [point_field_vector(charges[i].value_si, sources[i], target) for i in range(2)]\n            vectors = [v for v in vectors if v is not None]\n            if len(vectors) == 2:\n                ex, ey = sum(v[0] for v in vectors), sum(v[1] for v in vectors)\n                return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(math.hypot(ex, ey)), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_perpendicular_bisector_field\", \"formula\": \"vector superposition on perpendicular bisector\", \"net\": [ex, ey], \"height\": height})\n\n    if family == \"electrostatics_force\" and \"midpoint\" in text and len(charges) >= 3 and distance_values:\n        d = distance_values[0]\n        if d <= 0:\n            return None\n        q_target = abs(charges[2].value_si)\n        result = COULOMB_K * q_target * (abs(charges[0].value_si) + abs(charges[1].value_si)) / ((d / 2) ** 2)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(result), pred_unit=\"N\", status=\"ok\", trace={\"planner\": \"deterministic_midpoint_two_charge_force\", \"formula\": \"F=k|q0|(|q1|+|q2|)/(d/2)^2\"})\n\n    if family == \"electrostatics_field\" and \"midpoint\" in text and len(charges) >= 2 and distance_values:\n        d = distance_values[0]\n        if d <= 0:\n            return None\n        e = COULOMB_K * abs(charges[0].value_si - charges[1].value_si) / ((d / 2) ** 2)\n        return SolveResult(family=\"electrostatics_field\", answer_type=\"numeric\", pred_answer=format_number(e), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_midpoint_two_charge_field\", \"formula\": \"E=k|q1-q2|/(d/2)^2 for same-sign endpoint charges\"})\n\n    if family == \"electrostatics_force\" and \"equidistant\" in text and \"line connecting\" in text and len(charges) >= 3 and distance_values:\n        d = distance_values[0]\n        r = d / 2\n        q_target = charges[2].value_si\n        result = COULOMB_K * abs(q_target) * (abs(charges[0].value_si) + abs(charges[1].value_si)) / (r * r)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(result), pred_unit=\"N\", status=\"ok\", trace={\"planner\": \"deterministic_midpoint_line_force\", \"formula\": \"forces same direction at midpoint for opposite source signs\"})\n\n    # General two-source target triangle: AB plus distances from target to A/B.\n    dmap = distance_symbol_map(observations)\n    d_ab = distance_between_points(dmap, \"A\", \"B\")\n    r_ac = distance_between_points(dmap, \"A\", \"C\")\n    r_bc = distance_between_points(dmap, \"B\", \"C\")\n    if d_ab is not None and r_ac is not None and r_bc is not None:\n        d, r_a, r_b = d_ab, r_ac, r_bc\n    elif d_ab is not None and r_bc is not None and re.search(r\"AC\\s*=\\s*BC\", normalize_text(question), re.I):\n        d, r_a, r_b = d_ab, r_bc, r_bc\n    elif len(distance_values) >= 3:\n        d, r_a, r_b = distance_values[0], distance_values[1], distance_values[2]\n    elif len(distance_values) == 2 and re.search(r\"AC\\s*=\\s*BC|AC\\s+and\\s+BC\", normalize_text(question), re.I):\n        d, r_a, r_b = distance_values[1], distance_values[0], distance_values[0]\n    else:\n        return None\n    if d <= 0 or r_a <= 0 or r_b <= 0:\n        return None\n    x = (r_a * r_a + d * d - r_b * r_b) / (2 * d)\n    y_sq = max(0.0, r_a * r_a - x * x)\n    target_point = (x, math.sqrt(y_sq))\n    source_points = [(0.0, 0.0), (d, 0.0)]\n\n    if family == \"electrostatics_force\" and len(charges) >= 3:\n        q_target = charges[2].value_si\n        vectors = [force_on_target_vector(charges[i].value_si, q_target, source_points[i], target_point) for i in range(2)]\n        vectors = [v for v in vectors if v is not None]\n        if len(vectors) == 2:\n            fx, fy = sum(v[0] for v in vectors), sum(v[1] for v in vectors)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(math.hypot(fx, fy)), pred_unit=\"N\", status=\"ok\", trace={\"planner\": \"deterministic_two_source_target_force_triangle\", \"formula\": \"Coulomb vector superposition from AB, AC, BC\", \"net\": [fx, fy]})\n    field_charge_values = two_source_charge_values(question, observations)\n    if family == \"electrostatics_field\" and len(field_charge_values) >= 2:\n        vectors = [point_field_vector(field_charge_values[i], source_points[i], target_point) for i in range(2)]\n        vectors = [v for v in vectors if v is not None]\n        if len(vectors) == 2:\n            ex, ey = sum(v[0] for v in vectors), sum(v[1] for v in vectors)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(math.hypot(ex, ey)), pred_unit=\"V/m\", status=\"ok\", trace={\"planner\": \"deterministic_two_source_target_field_triangle\", \"formula\": \"field vector superposition from AB, AC, BC\", \"net\": [ex, ey]})\n    return None\n\n\ndef solve_right_triangle_altitude_field(question: str, observations: list[Observation]) -> SolveResult | None:\n    text = normalize_text(question).lower()\n    if \"right triangle\" not in text or \"foot of the altitude\" not in text:\n        return None\n    charges = [obs for obs in observations if obs.quantity_type == \"charge\"]\n    distances = [obs.value_si for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n    if not charges or len(distances) < 3:\n        return None\n    # Use the common 30-40-50 right-triangle convention: legs are the two shorter sides.\n    sides = sorted(distances)\n    leg1, leg2 = sides[0], sides[1]\n    a = (0.0, 0.0)\n    b = (leg1, 0.0)\n    c = (0.0, leg2)\n    vx, vy = c[0] - b[0], c[1] - b[1]\n    t = -((b[0] - a[0]) * vx + (b[1] - a[1]) * vy) / (vx * vx + vy * vy)\n    h = (b[0] + t * vx, b[1] + t * vy)\n    source_points = [a, b, c]\n    vectors = []\n    for idx, point in enumerate(source_points):\n        charge = charges[min(idx, len(charges) - 1)].value_si\n        vec = point_field_vector(charge, point, h)\n        if vec:\n            vectors.append(vec)\n    if len(vectors) < 3:\n        return None\n    ex, ey = sum(v[0] for v in vectors), sum(v[1] for v in vectors)\n    return SolveResult(family=\"electrostatics_field\", answer_type=\"numeric\", pred_answer=format_number(math.hypot(ex, ey)), pred_unit=\"N/C\", status=\"ok\", trace={\"planner\": \"deterministic_right_triangle_altitude_field\", \"formula\": \"field at altitude foot by coordinate geometry\", \"target\": h, \"net\": [ex, ey]})\n\n\ndef solve_electrostatics_structured_patterns(question: str, observations: list[Observation], family: str) -> SolveResult | None:\n    charges = [obs for obs in observations if obs.quantity_type == \"charge\"]\n    solvers = [\n        lambda: two_charge_axis_zero_field(question, charges, observations),\n        lambda: solve_uniform_field_kinematics(question, observations),\n        lambda: solve_force_balance_field(question, observations),\n        lambda: solve_inverse_point_charge_field(question, observations),\n        lambda: solve_continuous_distribution_field(question, observations),\n        lambda: solve_symbolic_field_scaling(question),\n        lambda: solve_square_fourth_vertex_field(question, observations),\n        lambda: solve_collinear_three_charge_field(question, observations),\n        lambda: solve_collinear_two_source_field(question, observations),\n        lambda: solve_angled_two_source_field(question, observations),\n        lambda: solve_equilateral_vertex_resultant(question, observations, family),\n        lambda: solve_right_isosceles_vertex(question, observations, family),\n        lambda: solve_right_triangle_altitude_field(question, observations),\n        lambda: solve_two_source_target_geometry(question, observations, family),\n    ]\n    for solver in solvers:\n        result = solver()\n        if result is not None:\n            return result\n    return None\n\n\ndef solve_electrostatics(question: str, observations: list[Observation], relations: list[Relation]) -> SolveResult:\n    family = classify_family(question)\n    obs_by_symbol = get_obs_by_symbol(observations)\n    charge_to_point = map_charges_to_points(question, observations)\n    distances = relation_distances(relations)\n    trace: dict[str, Any] = {\n        \"observations\": [asdict(o) for o in observations],\n        \"relations\": [asdict(r) for r in relations],\n        \"charge_to_point\": charge_to_point,\n        \"planner\": \"deterministic_electrostatics_vector\",\n    }\n\n    structured = solve_electrostatics_structured_patterns(question, observations, family)\n    if structured is not None:\n        return structured\n\n    if family == \"electrostatics_force\":\n        direct_resultant = solve_two_vector_resultant(question, observations, \"N\")\n        if direct_resultant:\n            return direct_resultant\n\n        target_charge = identify_force_target(question, charge_to_point)\n        if not target_charge or target_charge not in obs_by_symbol or target_charge not in charge_to_point:\n            return SolveResult(family=family, status=\"unsupported\", failure_reason=\"target charge not identified\", trace=trace)\n        target_point = charge_to_point[target_charge]\n        q_target = obs_by_symbol[target_charge].value_si\n        vectors: list[tuple[float, float]] = []\n        components_trace = []\n        for source_charge, source_point in charge_to_point.items():\n            if source_charge == target_charge or source_charge not in obs_by_symbol:\n                continue\n            r = distances.get(frozenset([target_point, source_point]))\n            if not r:\n                continue\n            # Temporary source coordinates are assigned later for paired-source triangle.\n            components_trace.append({\"charge\": source_charge, \"point\": source_point, \"r\": r})\n        if not components_trace:\n            return SolveResult(family=family, status=\"unsupported\", failure_reason=\"no source charge distances to target\", trace=trace)\n        coords = coordinates_relative_to_target(target_point, [c[\"point\"] for c in components_trace], distances)\n        if not coords:\n            return SolveResult(family=family, status=\"validator_failed\", failure_reason=\"could not build geometry coordinates\", trace=trace)\n        for comp in components_trace:\n            source_charge = comp[\"charge\"]\n            source_point = comp[\"point\"]\n            q_source = obs_by_symbol[source_charge].value_si\n            r = comp[\"r\"]\n            magnitude = COULOMB_K * abs(q_target * q_source) / (r * r)\n            sx, sy = coords[source_point]\n            norm = math.hypot(sx, sy)\n            if norm == 0:\n                continue\n            ux, uy = sx / norm, sy / norm\n            direction = 1.0 if q_target * q_source < 0 else -1.0\n            vx, vy = direction * magnitude * ux, direction * magnitude * uy\n            vectors.append((vx, vy))\n            comp.update({\"magnitude\": magnitude, \"direction\": \"toward_source\" if direction > 0 else \"away_from_source\", \"vector\": [vx, vy]})\n        fx = sum(v[0] for v in vectors)\n        fy = sum(v[1] for v in vectors)\n        result = math.hypot(fx, fy)\n        trace.update({\"target\": target_charge, \"target_point\": target_point, \"coords\": coords, \"components\": components_trace, \"net\": [fx, fy]})\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=format_number(result),\n            pred_unit=\"N\",\n            status=\"ok\",\n            trace=trace,\n        )\n\n    if family == \"electrostatics_field\":\n        field_values = [obs.value_si for obs in observations if obs.unit_si in {\"V/m\"}]\n        direct_resultant = solve_two_vector_resultant(question, observations, \"V/m\") if len(field_values) >= 2 else None\n        if direct_resultant:\n            return direct_resultant\n\n        charges = [obs for obs in observations if obs.quantity_type == \"charge\"]\n        distance_observations = [obs for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n        angle = extract_angle_radians(question)\n        if len(charges) >= 2 and distance_observations and angle is not None and (\"produce at m\" in normalize_text(question).lower() or \"from point m\" in normalize_text(question).lower()):\n            r = distance_observations[0].value_si\n            e1 = COULOMB_K * abs(charges[0].value_si) / (r * r)\n            e2 = COULOMB_K * abs(charges[1].value_si) / (r * r)\n            result = math.sqrt(max(0.0, e1 * e1 + e2 * e2 + 2 * e1 * e2 * math.cos(angle)))\n            return SolveResult(\n                family=family,\n                answer_type=\"numeric\",\n                pred_answer=format_number(result),\n                pred_unit=\"V/m\",\n                status=\"ok\",\n                trace={**trace, \"formula\": \"E_i=k|q_i|/r^2; resultant by included angle\", \"components\": [e1, e2], \"angle\": angle},\n            )\n\n        target_point = identify_field_target_point(question)\n        if not target_point:\n            return SolveResult(family=family, status=\"unsupported\", failure_reason=\"target field point not identified\", trace=trace)\n        sources = [(sym, point) for sym, point in charge_to_point.items() if sym in obs_by_symbol and point != target_point]\n        if not sources:\n            return SolveResult(family=family, status=\"unsupported\", failure_reason=\"field source charges not identified\", trace=trace)\n        coords = coordinates_relative_to_target(target_point, [point for _, point in sources], distances)\n        if not coords:\n            return SolveResult(family=family, status=\"validator_failed\", failure_reason=\"could not build field geometry coordinates\", trace=trace)\n        dielectric = extract_dielectric_constant(question) or 1.0\n        vectors = []\n        components_trace = []\n        for source_charge, source_point in sources:\n            r = distances.get(frozenset([target_point, source_point]))\n            if not r:\n                continue\n            q_source = obs_by_symbol[source_charge].value_si\n            magnitude = COULOMB_K * abs(q_source) / (dielectric * r * r)\n            sx, sy = coords[source_point]\n            norm = math.hypot(sx, sy)\n            ux, uy = sx / norm, sy / norm\n            # Electric field points away from positive source and toward negative source.\n            direction = -1.0 if q_source > 0 else 1.0\n            vx, vy = direction * magnitude * ux, direction * magnitude * uy\n            vectors.append((vx, vy))\n            components_trace.append({\"charge\": source_charge, \"point\": source_point, \"magnitude\": magnitude, \"vector\": [vx, vy]})\n        ex = sum(v[0] for v in vectors)\n        ey = sum(v[1] for v in vectors)\n        result = math.hypot(ex, ey)\n        trace.update({\"target_point\": target_point, \"coords\": coords, \"components\": components_trace, \"net\": [ex, ey], \"dielectric\": dielectric})\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=format_number(result),\n            pred_unit=\"V/m\",\n            status=\"ok\",\n            trace=trace,\n        )\n\n    return SolveResult(family=family, status=\"unsupported\", failure_reason=\"not electrostatics\", trace=trace)\n\n\ndef coordinates_relative_to_target(\n    target: str, source_points: list[str], distances: dict[frozenset[str], float]\n) -> dict[str, tuple[float, float]] | None:\n    points = list(dict.fromkeys(source_points))\n    if not points:\n        return None\n    coords: dict[str, tuple[float, float]] = {}\n    first = points[0]\n    r1 = distances.get(frozenset([target, first]))\n    if not r1:\n        return None\n    coords[first] = (r1, 0.0)\n    for point in points[1:]:\n        r2 = distances.get(frozenset([target, point]))\n        d12 = distances.get(frozenset([first, point]))\n        if not r2:\n            return None\n        if not d12:\n            # If no source-source relation exists, choose an arbitrary axis; validator keeps trace explicit.\n            coords[point] = (0.0, r2)\n            continue\n        cos_theta = (r1 * r1 + r2 * r2 - d12 * d12) / (2 * r1 * r2)\n        if cos_theta < -1 - 1e-6 or cos_theta > 1 + 1e-6:\n            return None\n        cos_theta = max(-1.0, min(1.0, cos_theta))\n        sin_theta = math.sqrt(max(0.0, 1 - cos_theta * cos_theta))\n        coords[point] = (r2 * cos_theta, r2 * sin_theta)\n    return coords\n\n\ndef extract_dielectric_constant(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(r\"(?:dielectric constant|relative permittivity|epsilon|ε|ε_r)\\s*(?:of|is|=)?\\s*([0-9.]+)\", text, re.I)\n    if m:\n        return parse_number(m.group(1))\n    if \"alcohol\" in text.lower():\n        return 2.2\n    return None\n\n\ndef extract_voltage_upper_bound(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(r\"U\\s*<\\s*([0-9.]+)\\s*V\", text, re.I)\n    if m:\n        return parse_number(m.group(1))\n    return None\n\n\ndef extract_count(text: str, default: int = 2) -> int:\n    text = normalize_text(text).lower()\n    words = {\"two\": 2, \"three\": 3, \"four\": 4, \"five\": 5}\n    m = re.search(r\"among\\s+(\\d+|two|three|four|five)\\s+identical\", text)\n    if not m:\n        m = re.search(r\"with\\s+(\\d+|two|three|four|five)\\s+identical\", text)\n    if not m:\n        return default\n    raw = m.group(1)\n    return int(raw) if raw.isdigit() else words.get(raw, default)\n\n\ndef first_value_by_type(observations: list[Observation], quantity_type: str) -> Observation | None:\n    for obs in observations:\n        if obs.quantity_type == quantity_type:\n            return obs\n    return None\n\n\ndef solve_capacitor_energy(question: str, observations: list[Observation]) -> SolveResult:\n    text = normalize_text(question).lower()\n    obs_by_symbol = get_obs_by_symbol(observations)\n    trace = {\"observations\": [asdict(o) for o in observations], \"planner\": \"deterministic_capacitor_energy\"}\n    family = classify_family(question)\n\n    c_obs = obs_by_symbol.get(\"C\") or first_value_by_type(observations, \"capacitance\")\n    u_obs = obs_by_symbol.get(\"U\") or obs_by_symbol.get(\"V\") or first_value_by_type(observations, \"voltage\")\n    q_obs = obs_by_symbol.get(\"Q\") or first_value_by_type(observations, \"charge\")\n    e_obs = first_value_by_type(observations, \"energy\")\n\n    if \"short-circuit\" in text or \"short circuited\" in text:\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=\"0; 0\",\n            pred_unit=\"μC; μJ\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"after short-circuit, Q=0 and E=0\"},\n        )\n\n    wants_energy = bool(re.search(r\"energy|stored energy|electric field energy\", text))\n    wants_capacitance = bool(\n        re.search(\n            r\"calculate (?:the )?(?:new )?capacitance|find (?:the )?(?:new )?capacitance|\"\n            r\"what is (?:the )?(?:new )?capacitance|determine (?:the )?(?:new )?capacitance|\"\n            r\"calculate\\s+c[0-9']?\\b|find\\s+c[0-9']?\\b|determine\\s+c[0-9']?\\b|\"\n            r\"calculate its capacitance|find its capacitance|determine its capacitance\",\n            text,\n        )\n    )\n\n    if \"voltage across the capacitor\" in text and \"current\" in text and \"maximum\" in text and \"lc circuit\" in text:\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=\"0\", pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"in LC circuit, capacitor voltage is zero when current is maximum\"})\n\n    if (\"conservation of energy\" in text or \"energy conservation\" in text) and \"lc circuit\" in text:\n        return SolveResult(family=family, answer_type=\"text\", pred_answer=\"Conservation of energy\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"ideal LC oscillation conserves total energy\"})\n\n    if \"electric field energy\" in text and \"magnetic field energy\" in text and \"oscillation process\" in text:\n        return SolveResult(family=family, answer_type=\"text\", pred_answer=\"Conservation of energy\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"energy alternates between electric and magnetic forms in LC oscillation\"})\n\n    wants_charge = bool(\n        re.search(r\"charge stored|stored charge|calculate (?:the )?charge|find (?:the )?charge|maximum charge|and (?:the )?charge|calculate\\s+Q\\b\", text)\n    )\n    wants_voltage = bool(\n        re.search(\n            r\"calculate (?:the |new )?voltage|find (?:the |new )?voltage|determine (?:the |new )?voltage|\"\n            r\"what is (?:the |new )?voltage|how does .*voltage|voltage .*change|calculate\\s+u\\b\",\n            text,\n        )\n    )\n    wants_dielectric = bool(re.search(r\"dielectric constant|relative permittivity\", text))\n    capacitances = [obs for obs in observations if obs.quantity_type == \"capacitance\"]\n    area = extract_area(question)\n    separation = extract_plate_separation(question)\n    dielectric = extract_dielectric_constant(question) or 1.0\n\n    dielectric_values = extract_dielectric_constants(question)\n    if \"capacitance change\" in text and len(dielectric_values) >= 2:\n        ratio = dielectric_values[-1] / dielectric_values[0]\n        if abs(ratio - 0.5) <= 1e-6:\n            return SolveResult(family=family, answer_type=\"text\", pred_answer=\"decreases by half\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"C proportional to epsilon_r\", \"ratio\": ratio})\n        if abs(ratio - 2.0) <= 1e-6:\n            return SolveResult(family=family, answer_type=\"text\", pred_answer=\"doubles\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"C proportional to epsilon_r\", \"ratio\": ratio})\n        return SolveResult(family=family, answer_type=\"math_expression\", pred_answer=f\"C_new = {format_number(ratio)} C_old\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"C_new/C_old=epsilon_new/epsilon_old\", \"ratio\": ratio})\n\n    factor_match = re.search(r\"voltage increases by\\s+({})\\s+times\".format(NUMBER_RE), text, re.I)\n    if factor_match and \"energy\" in text and (\"what factor\" in text or \"by what factor\" in text):\n        factor = parse_number(factor_match.group(1))\n        if factor is not None:\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(factor * factor), pred_unit=\"times\", status=\"ok\", trace={**trace, \"formula\": \"W proportional to U^2 for fixed C\", \"voltage_factor\": factor})\n\n    if \"how many times\" in text and \"energy\" in text and q_obs and u_obs:\n        charge_values = [obs for obs in observations if obs.quantity_type == \"charge\"]\n        if len(charge_values) >= 2:\n            ratio = (charge_values[-1].value_si / charge_values[0].value_si) ** 2\n            if abs(ratio - 0.25) <= 1e-6:\n                return SolveResult(family=family, answer_type=\"text\", pred_answer=\"decreases by 4 times\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"W proportional to Q^2 at fixed C\", \"ratio\": ratio})\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(ratio), pred_unit=\"times\", status=\"ok\", trace={**trace, \"formula\": \"W_new/W_old=(Q_new/Q_old)^2\"})\n\n    if wants_voltage and e_obs and c_obs:\n        energy_values = [obs for obs in observations if obs.quantity_type == \"energy\"]\n        if \"total energy\" in text and \"magnetic\" in text and len(energy_values) >= 2:\n            electric_energy_si = max(0.0, energy_values[0].value_si - energy_values[1].value_si)\n            voltage_si = math.sqrt(max(0.0, 2 * electric_energy_si / c_obs.value_si))\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(voltage_si), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"U=sqrt(2*(W_total-W_magnetic)/C)\"})\n        voltage_si = math.sqrt(max(0.0, 2 * e_obs.value_si / c_obs.value_si))\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(voltage_si), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"U=sqrt(2*W/C)\"})\n\n    if wants_capacitance and e_obs and u_obs:\n        capacitance_si = 2 * e_obs.value_si / (u_obs.value_si**2)\n        unit = choose_capacitance_unit(capacitance_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(capacitance_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"C=2*W/U^2\"})\n\n    if wants_energy and q_obs and c_obs:\n        energy_si = q_obs.value_si**2 / (2 * c_obs.value_si)\n        unit = choose_energy_unit(energy_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(energy_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"W=Q^2/(2*C)\"})\n\n    voltage_function = re.search(rf\"\\bu(?:\\(t\\))?\\s*=\\s*({NUMBER_RE})\\s*(?:x|\\*)?\\s*(?:sin|cos)\\s*\\(\", text, re.I)\n    if wants_energy and c_obs and voltage_function and (\"maximum\" in text or \"max \" in text):\n        amplitude = parse_number(voltage_function.group(1))\n        if amplitude is not None:\n            energy_si = 0.5 * c_obs.value_si * amplitude**2\n            unit = choose_energy_unit(energy_si, question)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(energy_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"Wmax=0.5*C*U0^2\", \"amplitude\": amplitude})\n\n    if area and separation and wants_capacitance:\n        capacitance_si = EPSILON_0 * dielectric * area / separation\n        unit = choose_capacitance_unit(capacitance_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(capacitance_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"C=epsilon0*epsilon_r*S/d\", \"area_si\": area, \"separation_si\": separation, \"dielectric\": dielectric})\n\n    if area and separation and wants_charge and u_obs:\n        capacitance_si = EPSILON_0 * dielectric * area / separation\n        charge_si = capacitance_si * u_obs.value_si\n        unit = choose_charge_unit(charge_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(charge_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"Q=C*U; C=epsilon0*epsilon_r*S/d\", \"capacitance_si\": capacitance_si})\n\n    e_field_obs = next((obs for obs in observations if obs.unit_si == \"V/m\"), None)\n    if area and wants_charge and e_field_obs and (\"breakdown\" in text or \"maximum charge\" in text):\n        charge_si = EPSILON_0 * dielectric * area * e_field_obs.value_si\n        unit = choose_charge_unit(charge_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(charge_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"Q_max=epsilon0*epsilon_r*S*E_max\"})\n\n    if wants_energy and e_obs and len([obs for obs in observations if obs.quantity_type == \"voltage\"]) >= 2:\n        voltages = [obs for obs in observations if obs.quantity_type == \"voltage\"]\n        energy_si = e_obs.value_si * (voltages[-1].value_si / voltages[0].value_si) ** 2\n        unit = choose_energy_unit(energy_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(energy_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"W2=W1*(U2/U1)^2\"})\n\n    if area and separation and wants_energy and u_obs:\n        capacitance_si = EPSILON_0 * dielectric * area / separation\n        energy_si = 0.5 * capacitance_si * u_obs.value_si**2\n        unit = choose_energy_unit(energy_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(energy_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"W=0.5*C*U^2; C=epsilon0*epsilon_r*S/d\", \"capacitance_si\": capacitance_si})\n\n    if \"force\" in text and area and q_obs:\n        force_si = q_obs.value_si**2 / (2 * EPSILON_0 * dielectric * area)\n        unit = infer_requested_unit(question, [\"N\", \"mN\"]) or \"N\"\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(force_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"F=Q^2/(2*epsilon0*epsilon_r*S)\"})\n\n    if \"how does\" in text and \"voltage\" in text and \"charge\" in text and \"constant\" in text and len(capacitances) >= 2:\n        ratio = capacitances[0].value_si / capacitances[1].value_si\n        if abs(ratio - 0.5) <= 1e-6:\n            return SolveResult(family=family, answer_type=\"text\", pred_answer=\"the voltage is halved\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"V proportional to 1/C at constant Q\", \"ratio\": ratio})\n        if abs(ratio - 2.0) <= 1e-6:\n            return SolveResult(family=family, answer_type=\"text\", pred_answer=\"the voltage is doubled\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"V proportional to 1/C at constant Q\", \"ratio\": ratio})\n        return SolveResult(family=family, answer_type=\"math_expression\", pred_answer=f\"V_new = {format_number(ratio)} V_old\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"V_new/V_old=C_old/C_new\", \"ratio\": ratio})\n\n    if \"reduction in energy\" in text and len(capacitances) >= 2 and (\"same voltage\" in text or \"maintaining\" in text):\n        reduction = (capacitances[0].value_si - capacitances[1].value_si) / capacitances[0].value_si * 100\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(reduction), pred_unit=\"%\", status=\"ok\", trace={**trace, \"formula\": \"energy proportional to C at constant U\", \"reduction_percent\": reduction})\n\n    if (\"percentage\" in text or \"%\" in text) and \"energy\" in text and \"remains\" in text:\n        voltages = [obs for obs in observations if obs.quantity_type == \"voltage\"]\n        if len(voltages) >= 2:\n            percent = (voltages[1].value_si / voltages[0].value_si) ** 2 * 100\n            return SolveResult(\n                family=family,\n                answer_type=\"numeric\",\n                pred_answer=format_number(percent),\n                pred_unit=\"%\",\n                status=\"ok\",\n                trace={**trace, \"formula\": \"E proportional to U^2 for fixed C; percent remaining=(U2/U1)^2*100\"},\n            )\n\n    if \"lc circuit\" in text and \"magnetic\" in text and \"energy\" in text and \"total energy\" in text and c_obs and u_obs:\n        total_energy = first_value_by_type(observations, \"energy\")\n        if total_energy:\n            electric_si = 0.5 * c_obs.value_si * u_obs.value_si**2\n            magnetic_si = max(0.0, total_energy.value_si - electric_si)\n            unit = choose_energy_unit(magnetic_si, question)\n            return SolveResult(\n                family=family,\n                answer_type=\"numeric\",\n                pred_answer=format_number(convert_from_si(magnetic_si, unit)),\n                pred_unit=unit,\n                status=\"ok\",\n                trace={**trace, \"formula\": \"LC energy conservation; Wm=W_total-0.5*C*U^2\", \"electric_si\": electric_si, \"total_si\": total_energy.value_si},\n            )\n\n    if wants_energy and c_obs and u_obs and re.search(r\"connected with another uncharged|charge is equally shared|distributed equally|after sharing|after connection\", text):\n        charge_si = c_obs.value_si * u_obs.value_si\n        if len(capacitances) >= 2 and \"another uncharged\" in text:\n            c_total = sum(cap.value_si for cap in capacitances[:2])\n        else:\n            c_total = extract_count(text, default=2) * c_obs.value_si\n        energy_si = charge_si * charge_si / (2 * c_total)\n        unit = choose_energy_unit(energy_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(energy_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"Q conserved, E=Q^2/(2*C_total)\", \"charge_si\": charge_si, \"c_total\": c_total})\n\n    if wants_voltage and c_obs and u_obs and extract_dielectric_constant(question):\n        dielectric = extract_dielectric_constant(question) or 1.0\n        voltage = u_obs.value_si / dielectric if re.search(r\"disconnected|disconnect|isolated\", text) else u_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(voltage), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"dielectric voltage state\", \"dielectric\": dielectric})\n\n    if wants_capacitance and c_obs and re.search(r\"distance .* doubled|distance between .* doubled|plates .* doubled|moved apart\", text):\n        capacitance_si = c_obs.value_si / 2\n        unit = infer_requested_unit(question, [\"F\", \"μF\", \"nF\", \"pF\"]) or c_obs.unit\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(capacitance_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"C proportional to 1/d; doubled distance halves C\"})\n\n    if wants_voltage and c_obs and u_obs and re.search(r\"distance .* doubled|plates .* doubled|moved apart\", text):\n        voltage = u_obs.value_si * 2 if re.search(r\"disconnected|disconnect|isolated\", text) else u_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(voltage), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"distance doubled voltage state\"})\n\n    if \"c'\" in text and \"series\" in text and q_obs and c_obs and u_obs:\n        vc = q_obs.value_si / c_obs.value_si\n        vc2 = u_obs.value_si - vc\n        if vc2 > 0:\n            c2 = q_obs.value_si / vc2\n            unit = infer_requested_unit(question, [\"F\", \"μF\", \"nF\", \"pF\"]) or \"μF\"\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(c2, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"series capacitor C'=Q/(U_total-Q/C)\"})\n\n    if wants_energy and c_obs and u_obs and re.search(r\"distance .* doubled|plates .* doubled|moved apart\", text):\n        energy_si = 0.5 * c_obs.value_si * u_obs.value_si**2\n        energy_si = energy_si * 2 if re.search(r\"disconnected|disconnect|isolated\", text) else energy_si / 2\n        unit = choose_energy_unit(energy_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(energy_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"distance doubled energy state\"})\n\n    if wants_energy and wants_charge and c_obs and u_obs:\n        energy_si = 0.5 * c_obs.value_si * u_obs.value_si**2\n        charge_si = c_obs.value_si * u_obs.value_si\n        e_unit = choose_energy_unit(energy_si, question)\n        q_unit = choose_charge_unit(charge_si, question)\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=f\"{format_number(convert_from_si(energy_si, e_unit))}; {format_number(convert_from_si(charge_si, q_unit))}\",\n            pred_unit=f\"{e_unit}; {q_unit}\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"E=0.5*C*U^2; Q=C*U\", \"energy_si\": energy_si, \"charge_si\": charge_si},\n        )\n\n    if wants_dielectric and c_obs:\n        area = extract_area(question)\n        separation = extract_plate_separation(question)\n        if area and separation:\n            dielectric = c_obs.value_si * separation / (EPSILON_0 * area)\n            return SolveResult(\n                family=family,\n                answer_type=\"numeric\",\n                pred_answer=format_number(dielectric),\n                pred_unit=\"-\",\n                status=\"ok\",\n                trace={**trace, \"formula\": \"epsilon_r=C*d/(epsilon0*A)\", \"area_si\": area, \"separation_si\": separation},\n            )\n\n    if wants_voltage and q_obs:\n        capacitances = [obs for obs in observations if obs.quantity_type == \"capacitance\"]\n        limit = extract_voltage_upper_bound(question)\n        candidates = []\n        for cap in capacitances:\n            if cap.value_si:\n                voltage = q_obs.value_si / cap.value_si\n                candidates.append({\"capacitance_symbol\": cap.symbol, \"voltage\": voltage})\n        if candidates:\n            chosen = None\n            if limit is not None:\n                valid = [c for c in candidates if c[\"voltage\"] < limit]\n                if valid:\n                    chosen = valid[0]\n            chosen = chosen or candidates[0]\n            return SolveResult(\n                family=family,\n                answer_type=\"numeric\",\n                pred_answer=format_number(chosen[\"voltage\"]),\n                pred_unit=\"V\",\n                status=\"ok\",\n                trace={**trace, \"formula\": \"U=Q/C\", \"candidates\": candidates, \"limit\": limit},\n            )\n\n    if wants_charge and c_obs and u_obs:\n        charge_si = c_obs.value_si * u_obs.value_si\n        unit = choose_charge_unit(charge_si, question)\n        value = convert_from_si(charge_si, unit)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(value), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"Q=C*U\", \"value_si\": charge_si})\n\n    if wants_energy and c_obs and u_obs:\n        energy_si = 0.5 * c_obs.value_si * u_obs.value_si**2\n        dielectric = extract_dielectric_constant(question)\n        if dielectric and re.search(r\"disconnected|disconnect\", text):\n            energy_si = energy_si / dielectric\n        elif dielectric and re.search(r\"connected|remains connected\", text):\n            energy_si = energy_si * dielectric\n        unit = choose_energy_unit(energy_si, question)\n        value = convert_from_si(energy_si, unit)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(value), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"E=0.5*C*U^2 with dielectric state if present\", \"value_si\": energy_si, \"dielectric\": dielectric})\n\n    if wants_capacitance and q_obs and u_obs:\n        capacitance_si = q_obs.value_si / u_obs.value_si\n        unit = infer_requested_unit(question, [\"F\", \"μF\", \"nF\", \"pF\"]) or \"μF\"\n        value = convert_from_si(capacitance_si, unit)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(value), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"C=Q/U\", \"value_si\": capacitance_si})\n\n    if wants_voltage and q_obs and c_obs:\n        voltage_si = q_obs.value_si / c_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(voltage_si), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"U=Q/C\"})\n\n    if wants_capacitance and re.search(r\"parallel-plate|plate area|plate separation\", text):\n        area = extract_area(question)\n        separation = extract_plate_separation(question)\n        if area and separation:\n            capacitance_si = EPSILON_0 * area / separation\n            unit = infer_requested_unit(question, [\"F\", \"μF\", \"nF\", \"pF\"]) or \"pF\"\n            value = convert_from_si(capacitance_si, unit)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(value), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"C=epsilon0*S/d\", \"area_si\": area})\n\n    return SolveResult(family=family, status=\"unsupported\", failure_reason=\"capacitor formula not covered\", trace=trace)\n\n\ndef extract_area(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(rf\"(?:area|S)\\s*(?:=|of)?\\s*({NUMBER_RE})\\s*(cm|mm|m)\\s*(?:\\^2|²|2)\", text, re.I)\n    if m:\n        value = parse_number(m.group(1))\n        unit = normalize_unit(m.group(2))\n        if value is not None:\n            scale, _ = UNIT_SCALE_TO_SI.get(unit, (1.0, unit))\n            return value * scale * scale\n\n    radius_patterns = [\n        rf\"(?:radius|R)\\s*(?:=|of|is)?\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        rf\"circular plates?.{{0,80}}?({NUMBER_RE})\\s*({UNIT_RE})\",\n    ]\n    for pat in radius_patterns:\n        r = re.search(pat, text, re.I)\n        if not r:\n            continue\n        value = parse_number(r.group(1))\n        unit = normalize_unit(r.group(2))\n        if value is None:\n            continue\n        radius_si, unit_si = unit_to_si(value, unit)\n        if unit_si == \"m\":\n            return math.pi * radius_si * radius_si\n    return None\n\n\ndef extract_plate_separation(question: str) -> float | None:\n    text = normalize_text(question)\n    patterns = [\n        rf\"\\bd\\s*(?:=|is)?\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        rf\"(?:plate separation|separation|distance between the plates|distance between plates)\\s*(?:is|=|of)?\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        rf\"(?:distance between the two plates|distance between two plates)\\s*(?:is|=|of)?\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        rf\"(?:plates? are separated by)\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n        rf\"(?:separated by)\\s*({NUMBER_RE})\\s*({UNIT_RE})\",\n    ]\n    for pat in patterns:\n        m = re.search(pat, text, re.I)\n        if not m:\n            continue\n        value = parse_number(m.group(1))\n        unit = normalize_unit(m.group(2))\n        if value is None:\n            continue\n        value_si, unit_si = unit_to_si(value, unit)\n        if unit_si == \"m\":\n            return value_si\n    return None\n\n\ndef choose_capacitance_unit(value_si: float, question: str = \"\") -> str:\n    explicit = infer_requested_unit(question, [\"F\", \"μF\", \"nF\", \"pF\"])\n    if explicit:\n        return explicit\n    av = abs(value_si)\n    if av == 0:\n        return \"μF\"\n    if av < 1e-9:\n        return \"pF\"\n    if av < 1e-6:\n        return \"nF\"\n    if av < 1e-3:\n        return \"μF\"\n    return \"F\"\n\n\ndef choose_inductance_unit(value_si: float, question: str = \"\") -> str:\n    explicit = infer_requested_unit(question, [\"H\", \"mH\", \"uH\"])\n    if explicit:\n        return explicit\n    if abs(value_si) < 1:\n        return \"mH\"\n    return \"H\"\n\n\ndef extract_dielectric_constants(question: str) -> list[float]:\n    text = normalize_text(question)\n    values: list[float] = []\n    for m in re.finditer(r\"(?:dielectric constant|relative permittivity|epsilon|ε|ε_r)\\s*(?:of|is|=)?\\s*([0-9.]+)\", text, re.I):\n        value = parse_number(m.group(1))\n        if value is not None:\n            values.append(value)\n    return values\n\n\ndef extract_frequency_factor(question: str) -> float | None:\n    text = normalize_text(question).lower()\n    m = re.search(r\"frequency\\s+(?:f\\s+)?(?:is\\s+)?(?:increased by|multiplied by|changed by)\\s+({})\\s+times\".format(NUMBER_RE), text, re.I)\n    if not m:\n        m = re.search(r\"frequency\\s+is\\s+doubled\", text, re.I)\n        if m:\n            return 2.0\n    if not m:\n        m = re.search(r\"frequency\\s+is\\s+halved\", text, re.I)\n        if m:\n            return 0.5\n    if not m:\n        return None\n    value = parse_number(m.group(1))\n    return value\n\n\ndef infer_requested_unit(question: str, candidates: list[str]) -> str | None:\n    text = normalize_text(question)\n    for unit in candidates:\n        if unit in {\"C\", \"F\", \"H\", \"V\", \"A\", \"N\", \"J\", \"W\", \"m\", \"g\"}:\n            pattern = rf\"(?<![A-Za-z0-9μµ]){re.escape(unit)}(?![A-Za-z0-9])\"\n        else:\n            pattern = re.escape(unit)\n        if re.search(pattern, text):\n            return normalize_unit(unit)\n    return None\n\n\ndef solve_rlc(question: str, observations: list[Observation]) -> SolveResult:\n    text = normalize_text(question)\n    lower = text.lower()\n    obs_by_symbol = get_obs_by_symbol(observations)\n    trace = {\"observations\": [asdict(o) for o in observations], \"planner\": \"deterministic_rlc\"}\n    family = classify_family(question)\n    l_obs = obs_by_symbol.get(\"L\") or first_value_by_type(observations, \"inductance\")\n    c_obs = obs_by_symbol.get(\"C\") or first_value_by_type(observations, \"capacitance\")\n    f_obs = obs_by_symbol.get(\"f\") or first_value_by_type(observations, \"frequency\")\n    r_obs = obs_by_symbol.get(\"R\") or first_value_by_type(observations, \"resistance\")\n    u_obs = obs_by_symbol.get(\"U\") or first_value_by_type(observations, \"voltage\")\n    z_obs = obs_by_symbol.get(\"Z\")\n    xl_obs = obs_by_symbol.get(\"XL\") or obs_by_symbol.get(\"Xl\")\n    xc_obs = obs_by_symbol.get(\"XC\") or obs_by_symbol.get(\"Xc\")\n    is_resonant_prompt = \"resonan\" in lower and \"not in resonance\" not in lower\n\n    ohm_values = [obs.value_si for obs in observations if obs.unit_si == \"ohm\"]\n    if (\"multiple\" in lower or \"factor\" in lower or \"multiplied\" in lower or \"value of k\" in lower) and (\"resonan\" in lower or \"ω\" in text or \"omega\" in lower):\n        if xl_obs and xc_obs:\n            xl0, xc0 = xl_obs.value_si, xc_obs.value_si\n        elif len(ohm_values) >= 2 and (\"inductive reactance\" in lower or \"xl\" in lower) and (\"capacitive reactance\" in lower or \"xc\" in lower):\n            xl0, xc0 = ohm_values[0], ohm_values[1]\n        else:\n            xl0 = xc0 = None\n        if xl0 and xc0:\n            factor = math.sqrt(xc0 / xl0)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(factor), pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"k=sqrt(XC0/XL0) because XL'=kXL0 and XC'=XC0/k\"})\n\n    if (\"power factor\" in lower or \"cos\" in lower or \"lcω²\" in lower or \"lcω2\" in lower or \"lcw\" in lower) and is_resonant_prompt:\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=\"1\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"at resonance, cos(phi)=1\"})\n\n    if (family == \"rlc_resonance_yes_no\" or re.search(r\"\\bis\\b.*resonan(?:t|ce) frequency\", lower) or re.search(r\"\\bdoes\\b.*resonan\", lower)) and l_obs and c_obs and f_obs:\n        f0 = 1.0 / (2 * math.pi * math.sqrt(l_obs.value_si * c_obs.value_si))\n        rel_err = abs(f_obs.value_si - f0) / max(f0, 1e-12)\n        ans = \"Yes\" if rel_err <= 0.01 or abs(f_obs.value_si - f0) <= 0.5 else \"No\"\n        return SolveResult(family=family, answer_type=\"yes_no\", pred_answer=ans, pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"f0=1/(2*pi*sqrt(L*C)); compare with given f\", \"f0\": f0, \"relative_error\": rel_err})\n\n    if is_resonant_prompt and z_obs and (\"pure resistance\" in lower or \"determine r\" in lower or \"impedance\" in lower):\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(z_obs.value_si), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"at resonance, Z=R\"})\n\n    if is_resonant_prompt and r_obs and (\"determine r\" in lower or \"total impedance\" in lower):\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(r_obs.value_si), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"at resonance, Z=R\"})\n\n    if (\"resonance frequency\" in lower or \"resonant frequency\" in lower or \"electrical resonance frequency\" in lower or re.search(r\"\\bf0\\b\", lower)) and l_obs and c_obs:\n        f0 = 1.0 / (2 * math.pi * math.sqrt(l_obs.value_si * c_obs.value_si))\n        unit = infer_requested_unit(question, [\"Hz\", \"kHz\"]) or \"Hz\"\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(f0, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"f0=1/(2*pi*sqrt(L*C))\"})\n\n    if family == \"rlc_resonance_yes_no\" and l_obs and c_obs and f_obs:\n        f0 = 1.0 / (2 * math.pi * math.sqrt(l_obs.value_si * c_obs.value_si))\n        rel_err = abs(f_obs.value_si - f0) / max(f0, 1e-12)\n        ans = \"Yes\" if rel_err <= 0.01 or abs(f_obs.value_si - f0) <= 0.5 else \"No\"\n        return SolveResult(family=family, answer_type=\"yes_no\", pred_answer=ans, pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"f0=1/(2*pi*sqrt(L*C))\", \"f0\": f0, \"relative_error\": rel_err})\n\n    if \"capacitive reactance\" in lower and \"power factor\" in lower and z_obs and r_obs:\n        xc = math.sqrt(max(0.0, z_obs.value_si**2 - r_obs.value_si**2))\n        pf = r_obs.value_si / z_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"{format_number(xc)}; {format_number(pf)}\", pred_unit=\"Ω; -\", status=\"ok\", trace={**trace, \"formula\": \"Xc=sqrt(Z^2-R^2); cos_phi=R/Z\"})\n\n    if \"capacitive reactance\" in lower and c_obs and f_obs:\n        xc = 1.0 / (2 * math.pi * f_obs.value_si * c_obs.value_si)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(xc), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"Xc=1/(2*pi*f*C)\"})\n\n    if \"inductive reactance\" in lower and l_obs and f_obs:\n        xl = 2 * math.pi * f_obs.value_si * l_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(xl), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"XL=2*pi*f*L\"})\n\n    if (\"what inductance\" in lower or \"what inductor\" in lower or \"what is the inductance\" in lower or \"calculate the inductance\" in lower or \"determine the inductance\" in lower) and c_obs and f_obs:\n        l_si = 1.0 / ((2 * math.pi * f_obs.value_si) ** 2 * c_obs.value_si)\n        unit = infer_requested_unit(question, [\"H\", \"mH\"]) or \"H\"\n        value = convert_from_si(l_si, unit)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(value), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"L=1/((2*pi*f)^2*C)\"})\n\n    if (\"what capacitance\" in lower or \"calculate the capacitance\" in lower or re.search(r\"calculate\\s+c\\b\", lower) or \"should be chosen\" in lower) and l_obs and f_obs:\n        c_si = 1.0 / ((2 * math.pi * f_obs.value_si) ** 2 * l_obs.value_si)\n        unit = choose_capacitance_unit(c_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(c_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"C=1/((2*pi*f)^2*L)\"})\n\n    if (\"what l is needed\" in lower or re.search(r\"calculate\\s+l\\b\", lower)) and c_obs and f_obs:\n        l_si = 1.0 / ((2 * math.pi * f_obs.value_si) ** 2 * c_obs.value_si)\n        unit = choose_inductance_unit(l_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(l_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"L=1/((2*pi*f)^2*C)\"})\n\n    if \"lcω²\" in lower or \"lcω2\" in lower or \"lcw\" in lower:\n        resistors = [obs.value_si for obs in observations if obs.quantity_type == \"resistance\"]\n        if (\"current\" in lower or \"effective current\" in lower) and u_obs and len(resistors) >= 2:\n            total_r = sum(resistors[:2])\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(u_obs.value_si / total_r), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"LCω²=1 resonance-like condition; I=U/(R1+R2)\"})\n\n    if (\"total impedance\" in lower or re.search(r\"calculate .*impedance|impedance z\", lower)) and r_obs:\n        if xl_obs and xc_obs:\n            z = math.sqrt(r_obs.value_si**2 + (xl_obs.value_si - xc_obs.value_si) ** 2)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(z), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"Z=sqrt(R^2+(XL-XC)^2)\"})\n        if l_obs and c_obs and f_obs:\n            xl = 2 * math.pi * f_obs.value_si * l_obs.value_si\n            xc = 1.0 / (2 * math.pi * f_obs.value_si * c_obs.value_si)\n            z = math.sqrt(r_obs.value_si**2 + (xl - xc) ** 2)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(z), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"XL=2*pi*f*L; XC=1/(2*pi*f*C); Z=sqrt(R^2+(XL-XC)^2)\", \"XL\": xl, \"XC\": xc})\n\n    freq_factor = extract_frequency_factor(question)\n    if r_obs and xl_obs and xc_obs:\n        xl_value, xc_value = xl_obs.value_si, xc_obs.value_si\n        if freq_factor:\n            xl_value *= freq_factor\n            xc_value /= freq_factor\n        z = math.sqrt(r_obs.value_si**2 + (xl_value - xc_value) ** 2)\n        if (\"current\" in lower or \"effective current\" in lower) and u_obs:\n            current = u_obs.value_si / z\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(current), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"frequency-scaled reactances; Z=sqrt(R^2+(XL-XC)^2); I=U/Z\", \"Z\": z, \"frequency_factor\": freq_factor})\n        if \"power\" in lower and u_obs:\n            current = u_obs.value_si / z\n            power = current * current * r_obs.value_si\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(power), pred_unit=\"W\", status=\"ok\", trace={**trace, \"formula\": \"frequency-scaled reactances; P=I^2*R; I=U/Z\", \"Z\": z, \"I\": current, \"frequency_factor\": freq_factor})\n\n    if xl_obs and xc_obs and u_obs and freq_factor and (\"voltage across r\" in lower or \"voltage across the resistor\" in lower):\n        xl_value = xl_obs.value_si * freq_factor\n        xc_value = xc_obs.value_si / freq_factor\n        if abs(xl_value - xc_value) <= 1e-9 * max(1.0, abs(xl_value), abs(xc_value)):\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(u_obs.value_si), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"after frequency change XL=XC, so circuit is resonant and UR=U\"})\n\n    if r_obs and xl_obs and xc_obs:\n        z = math.sqrt(r_obs.value_si**2 + (xl_obs.value_si - xc_obs.value_si) ** 2)\n        if (\"current\" in lower or \"effective current\" in lower) and u_obs:\n            current = u_obs.value_si / z\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(current), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"Z=sqrt(R^2+(XL-XC)^2); I=U/Z\", \"Z\": z})\n        if \"power\" in lower and u_obs:\n            current = u_obs.value_si / z\n            power = current * current * r_obs.value_si\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(power), pred_unit=\"W\", status=\"ok\", trace={**trace, \"formula\": \"P=I^2*R; I=U/Z\", \"Z\": z, \"I\": current})\n\n    if (\"rms current\" in lower or \"effective current\" in lower or re.search(r\"\\bcurrent\\b\", lower)) and u_obs and z_obs:\n        current = u_obs.value_si / z_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(current), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"I=U/Z\"})\n\n    if is_resonant_prompt and r_obs:\n        if \"power\" in lower and u_obs:\n            p = u_obs.value_si**2 / r_obs.value_si\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(p), pred_unit=\"W\", status=\"ok\", trace={**trace, \"formula\": \"P=U^2/R at resonance\"})\n        if \"impedance\" in lower or \"pure resistance\" in lower:\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(r_obs.value_si), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"Z=R at resonance\"})\n        if re.search(r\"\\bcurrent\\b\", lower) and u_obs:\n            i = u_obs.value_si / r_obs.value_si\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(i), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"I=U/R at resonance\"})\n\n    if (\"rms current\" in lower or \"effective current\" in lower or re.search(r\"\\bcurrent\\b\", lower)) and u_obs and r_obs and is_resonant_prompt:\n        current = u_obs.value_si / r_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(current), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"at resonance, I=U/R\"})\n\n    if is_resonant_prompt and \"voltage across the inductor\" in lower and u_obs and r_obs and l_obs and c_obs:\n        omega0 = 1 / math.sqrt(l_obs.value_si * c_obs.value_si)\n        xl = omega0 * l_obs.value_si\n        current = u_obs.value_si / r_obs.value_si\n        ul = current * xl\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(ul), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"UL=I*XL; I=U/R at resonance; XL=omega0*L\"})\n\n    if is_resonant_prompt and \"voltage across the capacitor\" in lower:\n        voltages = [obs.value_si for obs in observations if obs.quantity_type == \"voltage\"]\n        if len(voltages) >= 2 and voltages[1] > voltages[0]:\n            uc = math.sqrt(max(0.0, voltages[1] ** 2 - voltages[0] ** 2))\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(uc), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"at resonance, U_RC^2=U_R^2+U_C^2 with U_R=U\"})\n\n    return SolveResult(family=family, status=\"unsupported\", failure_reason=\"RLC formula not covered\", trace=trace)\n\n\ndef solve_measurement(question: str, observations: list[Observation]) -> SolveResult:\n    text = normalize_text(question)\n    lower = text.lower()\n    trace = {\"observations\": [asdict(o) for o in observations], \"planner\": \"deterministic_measurement\"}\n    nums = [obs for obs in observations if obs.quantity_type in {\"unknown\", \"distance\", \"current\", \"voltage\", \"resistance\"} or obs.unit not in {\"-\"}]\n    family = \"measurement\"\n\n    uncertain = extract_uncertain_assignments(question)\n    generic_uncertain = extract_generic_uncertain_measurement(question)\n    if generic_uncertain and (\"relative\" in lower or \"percentage\" in lower or \"percent\" in lower):\n        value, error, unit = generic_uncertain\n        rel = abs(error / value) * 100 if value else 0.0\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=format_number(rel),\n            pred_unit=\"%\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"relative_error=absolute_error/value*100\", \"value\": value, \"absolute_error\": error, \"unit\": unit},\n        )\n\n    current_values = [obs.value_si for obs in observations if obs.quantity_type == \"current\"]\n    if \"current\" in lower and len(current_values) >= 2:\n        if \"total current\" in lower and \"removed\" not in lower:\n            total = sum(current_values[:2])\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"I_total = {format_number(total)}\", pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"parallel branch total current=sum(branch currents)\"})\n        if \"third branch\" in lower or \"third current\" in lower:\n            value = abs(current_values[0] - current_values[1])\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"I3 = {format_number(value)}\", pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"current balance for three-branch prompt\"})\n        if \"removed\" in lower:\n            value = current_values[-1]\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"I_total_new = {format_number(value)}\", pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"remaining branch current after one lamp removed\"})\n\n    if (\"R=U/I\" in text.replace(\" \", \"\") or \"R = U/I\" in text or \"resistance R is calculated\" in lower) and {\"U\", \"I\"} <= set(uncertain):\n        u, du, _ = uncertain[\"U\"]\n        i, di, _ = uncertain[\"I\"]\n        r = u / i\n        dr = r * (abs(du / u) + abs(di / i))\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(dr), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"R=U/I; dR/R=dU/U+dI/I\", \"R\": r})\n\n    if \"power\" in lower and {\"U\", \"I\"} <= set(uncertain):\n        u, du, _ = uncertain[\"U\"]\n        i, di, _ = uncertain[\"I\"]\n        p = u * i\n        rel = abs(du / u) + abs(di / i)\n        if \"relative\" in lower:\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(rel * 100), pred_unit=\"%\", status=\"ok\", trace={**trace, \"formula\": \"P=U*I; dP/P=dU/U+dI/I\", \"P\": p})\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(p * rel), pred_unit=\"W\", status=\"ok\", trace={**trace, \"formula\": \"P=U*I; dP=P*(dU/U+dI/I)\", \"P\": p})\n\n    if \"series\" in lower and \"resistance\" in lower and len(uncertain) >= 2:\n        total_error = sum(err for _, err, unit in uncertain.values() if unit in {\"Ω\", \"ohm\"})\n        if total_error:\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(total_error), pred_unit=\"Ω\", status=\"ok\", trace={**trace, \"formula\": \"series resistance absolute errors add\"})\n\n    if \"least count\" in lower:\n        least_direct = extract_least_count(question)\n        least_value, least_unit = least_direct if least_direct else (None, None)\n        if least_value is None and observations:\n            least_value, least_unit = observations[0].value, observations[0].unit\n        if \"relative error\" in lower and least_value is not None and len(observations) >= 2:\n            reading = observations[0] if abs(observations[0].value - least_value) > 1e-12 else observations[-1]\n            rel = abs(least_value / reading.value) * 100\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(rel), pred_unit=\"%\", status=\"ok\", trace={**trace, \"formula\": \"relative_error=least_count/reading*100\"})\n        if least_value is not None:\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(least_value), pred_unit=least_unit or \"-\", status=\"ok\", trace={**trace, \"formula\": \"absolute_error=least_count\"})\n\n    # True/measured absolute and relative error.\n    if (\"true value\" in lower or \"actual\" in lower or \"true\" in lower) and (\"measured\" in lower or \"student measured\" in lower):\n        pair = extract_actual_measured_pair(question)\n        values = extract_plain_measurement_values(question)\n        if pair or len(values) >= 2:\n            if pair:\n                true_val, measured_val, unit = pair\n            else:\n                true_val, unit = values[0]\n                measured_val, _ = values[1]\n            abs_err = abs(measured_val - true_val)\n            rel = abs_err / abs(true_val) * 100 if true_val else 0.0\n            if \"relative\" in lower and \"absolute\" not in lower:\n                return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(rel), pred_unit=\"%\", status=\"ok\", trace={**trace, \"formula\": \"relative_error=abs(measured-true)/true*100\"})\n            if \"absolute\" in lower and \"relative\" not in lower:\n                return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(abs_err), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"absolute_error=abs(measured-true)\"})\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"{format_number(abs_err)}; {format_number(rel)}\", pred_unit=f\"{unit}; %\", status=\"ok\", trace={**trace, \"formula\": \"absolute_and_relative_error\"})\n\n    if re.search(r\"measured .*times|measurements|readings\", lower):\n        values = extract_plain_measurement_values(question)\n        if len(values) >= 3:\n            unit = values[0][1]\n            xs = [v for v, _ in values]\n            mean = sum(xs) / len(xs)\n            mad = sum(abs(x - mean) for x in xs) / len(xs)\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"{format_number(mean)}; {format_number(mad, 3)}\", pred_unit=f\"{unit}; {unit}\", status=\"ok\", trace={**trace, \"formula\": \"mean_and_mean_absolute_error\", \"values\": xs})\n\n    if \"parallel\" in lower and \"current\" in lower and \"resistance\" in lower:\n        u = next((obs for obs in observations if obs.unit == \"V\"), None)\n        resistors = [obs for obs in observations if obs.unit in {\"Ω\", \"ohm\"}]\n        if u and len(resistors) >= 2:\n            currents = [u.value_si / r.value_si for r in resistors[:2]]\n            if \"total current\" in lower:\n                total = sum(currents)\n                return SolveResult(family=family, answer_type=\"numeric\", pred_answer=f\"I_total = {format_number(total)}\", pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"I_total=sum(U/Ri)\", \"branch_currents\": currents})\n            answer = \"; \".join(f\"I{i+1} = {format_number(cur)}\" for i, cur in enumerate(currents))\n            unit = \"; \".join([\"A\"] * len(currents))\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=answer, pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"Ii=U/Ri\"})\n\n    return SolveResult(family=family, status=\"unsupported\", failure_reason=\"measurement formula not covered\", trace=trace)\n\n\ndef extract_plain_measurement_values(question: str) -> list[tuple[float, str]]:\n    text = normalize_text(question)\n    values: list[tuple[float, str]] = []\n    for match in re.finditer(rf\"({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I):\n        value = parse_number(match.group(1))\n        unit = normalize_unit(match.group(2))\n        if value is not None:\n            values.append((value, unit))\n    return values\n\n\ndef extract_least_count(question: str) -> tuple[float, str] | None:\n    text = normalize_text(question)\n    m = re.search(rf\"least count(?:\\s*\\([^)]*\\))?(?:\\s+of| is| =)?\\s*({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I)\n    if not m:\n        return None\n    value = parse_number(m.group(1))\n    if value is None:\n        return None\n    return value, normalize_unit(m.group(2))\n\n\ndef extract_actual_measured_pair(question: str) -> tuple[float, float, str] | None:\n    text = normalize_text(question)\n    actual = re.search(rf\"(?:actual|true value|true(?:\\s+\\w+)?)\\D{{0,80}}?({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I)\n    measured = re.search(rf\"(?:measured value|student measured|measured|measures|measure[sd] .* as)\\D{{0,80}}?({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I)\n    if not actual or not measured:\n        return None\n    actual_value = parse_number(actual.group(1))\n    measured_value = parse_number(measured.group(1))\n    if actual_value is None or measured_value is None:\n        return None\n    return actual_value, measured_value, normalize_unit(actual.group(2))\n\n\ndef extract_uncertain_assignments(question: str) -> dict[str, tuple[float, float, str]]:\n    text = normalize_text(question)\n    values: dict[str, tuple[float, float, str]] = {}\n    for m in re.finditer(rf\"\\b([A-Za-z][A-Za-z0-9]*)\\s*=\\s*({NUMBER_RE})\\s*(?:±|\\+/-|\\+-)\\s*({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I):\n        value = parse_number(m.group(2))\n        error = parse_number(m.group(3))\n        if value is None or error is None:\n            continue\n        values[m.group(1).upper()] = (value, error, normalize_unit(m.group(4)))\n    return values\n\n\ndef extract_generic_uncertain_measurement(question: str) -> tuple[float, float, str] | None:\n    text = normalize_text(question)\n    m = re.search(rf\"({NUMBER_RE})\\s*(?:±|\\+/-|\\+-)\\s*({NUMBER_RE})\\s*({UNIT_RE})\", text, re.I)\n    if not m:\n        return None\n    value = parse_number(m.group(1))\n    error = parse_number(m.group(2))\n    if value is None or error is None:\n        return None\n    return value, error, normalize_unit(m.group(3))\n\n\ndef solve_energy_power(question: str, observations: list[Observation]) -> SolveResult:\n    text = normalize_text(question).lower()\n    trace = {\"observations\": [asdict(o) for o in observations], \"planner\": \"deterministic_energy_power\"}\n    family = classify_family(question)\n    l_obs = first_value_by_type(observations, \"inductance\")\n    i_obs = first_value_by_type(observations, \"current\")\n    e_obs = first_value_by_type(observations, \"energy\")\n    power_values = [obs.value_si for obs in observations if obs.quantity_type == \"power\"]\n\n    if (\"total power\" in text or \"power consumption\" in text) and len(power_values) >= 2:\n        total = sum(power_values)\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=f\"P_total = {format_number(total)}\",\n            pred_unit=\"W\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"P_total=sum(component powers)\"},\n        )\n\n    if \"lc circuit\" in text and re.search(r\"\\bi\\s*=\\s*0\\b\", text) and (\"where is the energy\" in text or \"energy stored\" in text):\n        return SolveResult(\n            family=family,\n            answer_type=\"text\",\n            pred_answer=\"all the energy is stored in the electric field of the capacitor.\",\n            pred_unit=\"-\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"LC energy exchange: when current is zero, magnetic energy is zero\"},\n        )\n\n    if \"total energy\" in text and \"magnetic energy is half\" in text:\n        return SolveResult(\n            family=family,\n            answer_type=\"text\",\n            pred_answer=\"Half of the total energy\",\n            pred_unit=\"J\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"W_total=W_electric+W_magnetic\"},\n        )\n\n    if \"total energy\" in text and (\"unchanged\" in text or \"vary over time\" in text or \"constant\" in text):\n        return SolveResult(\n            family=family,\n            answer_type=\"text\",\n            pred_answer=\"Equal, unchanged\",\n            pred_unit=\"J\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"ideal LC total energy is conserved\"},\n        )\n\n    if \"energy of oscillation\" in text and \"lc circuit\" in text:\n        return SolveResult(\n            family=family,\n            answer_type=\"math_expression\",\n            pred_answer=\"U = 0.5*L*I_max^2\",\n            pred_unit=\"J\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"LC total energy equals maximum magnetic energy\"},\n        )\n\n    if \"current\" in text and \"halved\" in text and \"energy\" in text:\n        return SolveResult(\n            family=family,\n            answer_type=\"text\",\n            pred_answer=\"Reduced to 1/4\",\n            pred_unit=\"-\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"W proportional to I^2\"},\n        )\n\n    if \"energy in the inductor\" in text and \"total energy\" in text and (\"⅓\" in text or \"1/3\" in text or \"one third\" in text):\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=\"67\",\n            pred_unit=\"%\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"capacitor energy fraction = 1 - 1/3 = 2/3\"},\n        )\n\n    if \"magnetic\" in text and \"energy\" in text and l_obs:\n        current_expr = re.search(rf\"\\bi(?:\\(t\\))?\\s*=\\s*({NUMBER_RE})\\s*(?:x|\\*)?\\s*(cos|sin)\\s*\\(\\s*({NUMBER_RE})\\s*t\\s*\\)\", text, re.I)\n        time_expr = re.search(rf\"\\bt\\s*=\\s*({NUMBER_RE})\\s*s\\b\", text, re.I)\n        if current_expr and (\"maximum\" in text or \"max \" in text):\n            amplitude = parse_number(current_expr.group(1))\n            if amplitude is not None:\n                e_si = 0.5 * l_obs.value_si * amplitude**2\n                unit = choose_energy_unit(e_si, question)\n                return SolveResult(\n                    family=family,\n                    answer_type=\"numeric\",\n                    pred_answer=format_number(convert_from_si(e_si, unit)),\n                    pred_unit=unit,\n                    status=\"ok\",\n                    trace={**trace, \"formula\": \"Wmax=0.5*L*I0^2\", \"amplitude\": amplitude},\n                )\n        if current_expr and time_expr:\n            amplitude = parse_number(current_expr.group(1))\n            func = current_expr.group(2).lower()\n            omega = parse_number(current_expr.group(3))\n            time_value = parse_number(time_expr.group(1))\n            if amplitude is not None and omega is not None and time_value is not None:\n                trig = math.cos if func == \"cos\" else math.sin\n                current = amplitude * trig(omega * time_value)\n                e_si = 0.5 * l_obs.value_si * current**2\n                unit = choose_energy_unit(e_si, question)\n                return SolveResult(\n                    family=family,\n                    answer_type=\"numeric\",\n                    pred_answer=format_number(convert_from_si(e_si, unit)),\n                    pred_unit=unit,\n                    status=\"ok\",\n                    trace={**trace, \"formula\": \"I(t)=I0*cos(omega*t); E=0.5*L*I(t)^2\", \"current\": current},\n                )\n    if (\"magnetic\" in text or \"inductor\" in text or \"coil\" in text) and \"energy\" in text and e_obs and i_obs and not l_obs:\n        l_si = 2 * e_obs.value_si / (i_obs.value_si**2)\n        unit = infer_requested_unit(question, [\"H\", \"mH\"]) or \"H\"\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=format_number(convert_from_si(l_si, unit)),\n            pred_unit=unit,\n            status=\"ok\",\n            trace={**trace, \"formula\": \"L=2*W/I^2\"},\n        )\n\n    if (\"magnetic\" in text or \"inductor\" in text or \"coil\" in text) and \"energy\" in text and e_obs and l_obs and not i_obs:\n        current = math.sqrt(max(0.0, 2 * e_obs.value_si / l_obs.value_si))\n        return SolveResult(\n            family=family,\n            answer_type=\"numeric\",\n            pred_answer=format_number_for_question(current, question),\n            pred_unit=\"A\",\n            status=\"ok\",\n            trace={**trace, \"formula\": \"I=sqrt(2*W/L)\"},\n        )\n\n    if \"magnetic\" in text and \"energy\" in text and l_obs and i_obs:\n        e_si = 0.5 * l_obs.value_si * i_obs.value_si**2\n        unit = choose_energy_unit(e_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(e_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"E=0.5*L*I^2\"})\n    return SolveResult(family=family, status=\"unsupported\", failure_reason=\"energy/power formula not covered\", trace=trace)\n\n\ndef extract_time_seconds(question: str) -> float | None:\n    text = normalize_text(question)\n    m = re.search(rf\"(?:over a period of|in|during|after)\\s*({NUMBER_RE})\\s*(?:s|second|seconds)\\b\", text, re.I)\n    if not m:\n        m = re.search(rf\"\\bt\\s*(?:=|is)?\\s*({NUMBER_RE})\\s*(?:s|second|seconds)\\b\", text, re.I)\n    if not m:\n        return None\n    return parse_number(m.group(1))\n\n\ndef extract_change_pair(question: str, unit: str) -> tuple[float, float] | None:\n    text = normalize_text(question)\n    pat = rf\"from\\s*({NUMBER_RE})\\s*(?:{re.escape(unit)})?\\s*to\\s*({NUMBER_RE})\\s*{re.escape(unit)}\"\n    m = re.search(pat, text, re.I)\n    if not m:\n        pat = rf\"(?:decreases|increases)\\s+from\\s*({NUMBER_RE})\\s*{re.escape(unit)}\\s*to\\s*({NUMBER_RE})\\s*(?:{re.escape(unit)})?\"\n        m = re.search(pat, text, re.I)\n    if not m:\n        return None\n    start = parse_number(m.group(1))\n    end = parse_number(m.group(2))\n    if start is None or end is None:\n        return None\n    return start, end\n\n\ndef extract_solenoid_turn_count(question: str) -> float | None:\n    text = normalize_text(question)\n    patterns = [\n        rf\"(?:has|consists of|with)\\s*({NUMBER_RE})\\s*turns\\b\",\n        rf\"(?:consisting of)\\s*({NUMBER_RE})\\s*turns\\b\",\n        rf\"\\bN\\s*(?:=|is)?\\s*({NUMBER_RE})\\s*turns\\b\",\n        rf\"({NUMBER_RE})\\s*turns\\b(?!/)\",\n    ]\n    for pat in patterns:\n        m = re.search(pat, text, re.I)\n        if m:\n            return parse_number(m.group(1))\n    return None\n\n\ndef extract_turn_density(question: str) -> float | None:\n    text = normalize_text(question)\n    patterns = [\n        rf\"\\bn\\s*(?:=|is)?\\s*({NUMBER_RE})\\s*turns/m\\b\",\n        rf\"({NUMBER_RE})\\s*turns/m\\b\",\n    ]\n    for pat in patterns:\n        m = re.search(pat, text, re.I)\n        if m:\n            return parse_number(m.group(1))\n    return None\n\n\ndef extract_solenoid_length(question: str, observations: list[Observation]) -> float | None:\n    text = normalize_text(question)\n    m = re.search(rf\"(?:length|long|is)\\s*(?:l\\s*)?(?:=|of|is)?\\s*({NUMBER_RE})\\s*(m|cm|mm)\\s*(?:long)?\", text, re.I)\n    if m:\n        value = parse_number(m.group(1))\n        if value is not None:\n            value_si, unit_si = unit_to_si(value, normalize_unit(m.group(2)))\n            if unit_si == \"m\":\n                return value_si\n    distance_values = [obs.value_si for obs in observations if obs.quantity_type == \"distance\" and obs.unit_si == \"m\"]\n    return distance_values[0] if distance_values else None\n\n\ndef solve_magnetic_induction(question: str, observations: list[Observation]) -> SolveResult:\n    text = normalize_text(question)\n    lower = text.lower()\n    trace = {\"observations\": [asdict(o) for o in observations], \"planner\": \"deterministic_magnetic_induction\"}\n    family = classify_family(question)\n    i_obs = first_value_by_type(observations, \"current\")\n    l_obs = first_value_by_type(observations, \"inductance\")\n    e_obs = first_value_by_type(observations, \"energy\")\n    voltage_obs = first_value_by_type(observations, \"voltage\")\n    flux_obs = first_value_by_type(observations, \"magnetic_flux\")\n    instantaneous_current = parse_instantaneous_cos_current(question)\n\n    if \"formula\" in lower and \"magnetic field energy\" in lower and \"inductor\" in lower:\n        return SolveResult(family=family, answer_type=\"math_expression\", pred_answer=\"W = 1/2 · L · I^2\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"symbolic inductor energy\"})\n\n    if e_obs and i_obs and not l_obs and (\"inductance\" in lower or \"coil\" in lower):\n        l_si = 2 * e_obs.value_si / (i_obs.value_si**2)\n        unit = infer_requested_unit(question, [\"H\", \"mH\"]) or \"H\"\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(l_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"L=2*W/I^2\"})\n\n    if e_obs and l_obs and not i_obs and (\"current\" in lower or \"instantaneous current\" in lower):\n        current = math.sqrt(max(0.0, 2 * e_obs.value_si / l_obs.value_si))\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number_for_question(current, question), pred_unit=\"A\", status=\"ok\", trace={**trace, \"formula\": \"I=sqrt(2*W/L)\"})\n\n    if l_obs and (i_obs or instantaneous_current is not None) and (\"magnetic field energy\" in lower or \"energy\" in lower):\n        current_si = instantaneous_current if instantaneous_current is not None else i_obs.value_si\n        e_si = 0.5 * l_obs.value_si * current_si**2\n        unit = choose_energy_unit(e_si, question)\n        formula = \"W=0.5*L*(I0*cos(omega*t))^2\" if instantaneous_current is not None else \"W=0.5*L*I^2\"\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(e_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": formula, \"current_si\": current_si})\n\n    turns = extract_solenoid_turn_count(question)\n    turn_density = extract_turn_density(question)\n    length = extract_solenoid_length(question, observations)\n    area = extract_area(question)\n    if not turn_density and turns and length:\n        turn_density = turns / length\n\n    if (\"magnetic field\" in lower or \"flux density\" in lower or \"field inside\" in lower) and \"energy\" not in lower and turn_density and i_obs:\n        b_si = MU_0 * turn_density * i_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(b_si), pred_unit=\"T\", status=\"ok\", trace={**trace, \"formula\": \"B=mu0*n*I\", \"turn_density\": turn_density})\n\n    if (\"inductance\" in lower or \"self-inductance\" in lower) and turns and area and length and not voltage_obs:\n        inductance_si = MU_0 * turns * turns * area / length\n        unit = choose_inductance_unit(inductance_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(inductance_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"L=mu0*N^2*S/l\", \"turns\": turns, \"area\": area, \"length\": length})\n\n    if (\"flux linkage\" in lower or \"total magnetic flux\" in lower) and flux_obs and turns:\n        linkage = turns * flux_obs.value_si\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(linkage), pred_unit=\"Wb\", status=\"ok\", trace={**trace, \"formula\": \"flux_linkage=N*phi\"})\n\n    if (\"magnetic flux\" in lower or \"flux through\" in lower) and turn_density and i_obs and area:\n        b_si = MU_0 * turn_density * i_obs.value_si\n        flux_si = b_si * area\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(flux_si), pred_unit=\"Wb\", status=\"ok\", trace={**trace, \"formula\": \"Phi=B*S; B=mu0*n*I\", \"B\": b_si})\n\n    if (\"magnetic field energy\" in lower or \"energy\" in lower) and turns and area and length and i_obs:\n        inductance_si = MU_0 * turns * turns * area / length\n        e_si = 0.5 * inductance_si * i_obs.value_si**2\n        unit = choose_energy_unit(e_si, question)\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(e_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"W=0.5*L*I^2; L=mu0*N^2*S/l\", \"L\": inductance_si})\n\n    time_s = extract_time_seconds(question)\n    current_values = [obs.value_si for obs in observations if obs.quantity_type == \"current\"]\n    current_pair = extract_change_pair(question, \"A\")\n    if voltage_obs and time_s and (len(current_values) >= 2 or current_pair) and (\"inductance\" in lower or \"self-inductance\" in lower):\n        delta_i = abs(current_pair[1] - current_pair[0]) if current_pair else abs(current_values[-1] - current_values[0])\n        if delta_i:\n            inductance_si = voltage_obs.value_si * time_s / delta_i\n            unit = infer_requested_unit(question, [\"H\", \"mH\"]) or \"H\"\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(convert_from_si(inductance_si, unit)), pred_unit=unit, status=\"ok\", trace={**trace, \"formula\": \"L=|emf|*dt/dI\"})\n\n    if l_obs and time_s and (len(current_values) >= 2 or current_pair) and (\"electromotive force\" in lower or \"emf\" in lower):\n        delta_i = abs(current_pair[1] - current_pair[0]) if current_pair else abs(current_values[-1] - current_values[0])\n        emf = l_obs.value_si * delta_i / time_s\n        return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(emf), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"emf=L*dI/dt\"})\n\n    flux_numbers = re.findall(rf\"({NUMBER_RE})\\s*Wb\", text, re.I)\n    flux_pair = extract_change_pair(question, \"Wb\")\n    if (\"electromotive force\" in lower or \"emf\" in lower) and time_s and (len(flux_numbers) >= 2 or flux_pair):\n        if flux_pair:\n            phi0, phi1 = flux_pair\n        else:\n            phi0 = parse_number(flux_numbers[0])\n            phi1 = parse_number(flux_numbers[1])\n        if phi0 is not None and phi1 is not None:\n            emf = abs(phi1 - phi0) / time_s\n            return SolveResult(family=family, answer_type=\"numeric\", pred_answer=format_number(emf), pred_unit=\"V\", status=\"ok\", trace={**trace, \"formula\": \"average emf=|dPhi|/dt\"})\n\n    if (\"depends linearly on\" in lower or \"depend linearly on\" in lower) and \"solenoid\" in lower:\n        return SolveResult(family=family, answer_type=\"text\", pred_answer=\"Current through the solenoid\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"B=mu0*n*I\"})\n\n    if \"self-inductance\" in lower and \"does not depend\" in lower and \"solenoid\" in lower:\n        return SolveResult(family=family, answer_type=\"text\", pred_answer=\"Current intensity\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"L=mu0*N^2*S/l, independent of current\"})\n\n    if \"kind of oscillation\" in lower and \"lc circuit\" in lower:\n        return SolveResult(family=family, answer_type=\"text\", pred_answer=\"Simple Harmonic Motion (SHM)\", pred_unit=\"-\", status=\"ok\", trace={**trace, \"formula\": \"ideal LC oscillator\"})\n\n    return SolveResult(family=family, status=\"unsupported\", failure_reason=\"magnetic induction formula not covered\", trace=trace)\n\n\ndef route_and_solve(row: dict[str, str]) -> SolveResult:\n    question = row.get(\"question\", \"\")\n    observations = extract_quantity_observations(question)\n    relations = extract_relations(question, observations)\n    family = classify_family(question)\n\n    solvers: list[Callable[[], SolveResult]] = []\n    if family in {\"capacitor\", \"energy_power\"}:\n        solvers.append(lambda: solve_capacitor_energy(question, observations))\n        solvers.append(lambda: solve_energy_power(question, observations))\n    if family == \"magnetic_induction\":\n        solvers.append(lambda: solve_magnetic_induction(question, observations))\n        solvers.append(lambda: solve_energy_power(question, observations))\n    if family in {\"rlc_resonance_yes_no\", \"ac_rlc\"}:\n        solvers.append(lambda: solve_rlc(question, observations))\n    if family == \"measurement\":\n        solvers.append(lambda: solve_measurement(question, observations))\n    if family in {\"electrostatics_force\", \"electrostatics_field\"}:\n        solvers.append(lambda: solve_electrostatics(question, observations, relations))\n    # Cross-family fallback for DDT/NL rows with recognizable formulas.\n    solvers.extend(\n        [\n            lambda: solve_rlc(question, observations),\n            lambda: solve_measurement(question, observations),\n            lambda: solve_capacitor_energy(question, observations),\n            lambda: solve_energy_power(question, observations),\n            lambda: solve_magnetic_induction(question, observations),\n        ]\n    )\n\n    first_specific_failure = None\n    last = None\n    for solver in solvers:\n        result = solver()\n        last = result\n        if result.status == \"ok\":\n            return result\n        if first_specific_failure is None and result.family == family:\n            first_specific_failure = result\n    return first_specific_failure or last or SolveResult(family=family, status=\"unsupported\", failure_reason=\"no solver matched\")\n\n\ndef normalize_numeric_values(answer: str) -> list[float]:\n    values = []\n    parts = split_multi_values(answer)\n    if not parts:\n        parts = [answer]\n    for part in parts:\n        # Strip assignment labels, keep the right side.\n        if \"=\" in part:\n            part = part.split(\"=\")[-1]\n        value = parse_number(part)\n        if value is not None:\n            values.append(value)\n    return values\n\n\ndef normalize_numeric_units(answer: str, fallback_unit: str) -> list[str]:\n    parts = split_multi_values(answer)\n    if not parts:\n        parts = [answer]\n    fallback_parts = split_multi_values(fallback_unit)\n    if not fallback_parts:\n        fallback_parts = [fallback_unit]\n    units = []\n    for i, part in enumerate(parts):\n        part = normalize_text(part)\n        match = re.search(rf\"{NUMBER_RE}\\s*({UNIT_RE})\\b\", part, flags=re.I)\n        if match:\n            units.append(normalize_unit(match.group(1)))\n        else:\n            units.append(normalize_unit(fallback_parts[min(i, len(fallback_parts) - 1)]))\n    return units\n\n\ndef compare_result(pred_answer: str, pred_unit: str, gold_answer: str, gold_unit: str, answer_type: str) -> tuple[bool, str]:\n    pred_unit = normalize_unit(pred_unit)\n    gold_unit = normalize_unit(gold_unit)\n    if answer_type == \"yes_no\":\n        return pred_answer.strip().lower() == gold_answer.strip().lower(), \"yes_no_exact\"\n    if answer_type in {\"text\", \"math_expression\"} and not normalize_numeric_values(gold_answer):\n        return normalize_answer_text_for_compare(pred_answer) == normalize_answer_text_for_compare(gold_answer), \"text_exact\"\n    pred_vals = normalize_numeric_values(pred_answer)\n    gold_vals = normalize_numeric_values(gold_answer)\n    if pred_vals and gold_vals and len(pred_vals) == len(gold_vals):\n        pred_units = normalize_numeric_units(pred_answer, pred_unit)\n        gold_units = normalize_numeric_units(gold_answer, gold_unit)\n        ok_vals = []\n        for i, (pv, gv) in enumerate(zip(pred_vals, gold_vals)):\n            pu = pred_units[min(i, len(pred_units) - 1)]\n            gu = gold_units[min(i, len(gold_units) - 1)]\n            pv_si, pu_si = unit_to_si(pv, pu)\n            gv_si, gu_si = unit_to_si(gv, gu)\n            if pu_si != gu_si and not ({pu_si, gu_si} <= {\"V/m\", \"N/C\"}):\n                ok_vals.append(False)\n                continue\n            tol = max(1e-6, 0.03 * abs(gv_si))\n            ok_vals.append(abs(pv_si - gv_si) <= tol)\n        return all(ok_vals), \"numeric_unit_tolerant\"\n    return False, \"unparsed_or_shape_mismatch\"\n\n\ndef build_dataset_profile(rows: list[dict[str, str]]) -> list[dict[str, Any]]:\n    profile = []\n    for row in rows:\n        flags = detect_reasoning_flags(row.get(\"question\", \"\"), row.get(\"cot\", \"\"))\n        observations = extract_quantity_observations(row.get(\"question\", \"\"))\n        relations = extract_relations(row.get(\"question\", \"\"), observations)\n        profile.append(\n            {\n                \"id\": row[\"id\"],\n                \"prefix\": row_prefix(row[\"id\"]),\n                \"family\": classify_family(row.get(\"question\", \"\")),\n                \"gold_answer_type\": classify_answer_type(row.get(\"answer\", \"\"), row.get(\"unit\", \"\")),\n                \"gold_subshape\": answer_subshape(row.get(\"answer\", \"\"), row.get(\"unit\", \"\")),\n                \"gold_answer\": row.get(\"answer\", \"\"),\n                \"gold_unit\": row.get(\"unit\", \"\"),\n                \"n_observations\": len(observations),\n                \"n_relations\": len(relations),\n                **{k: int(v) for k, v in flags.items()},\n            }\n        )\n    return profile\n\n\ndef build_family_summary(profile: list[dict[str, Any]], results: list[dict[str, Any]] | None = None) -> list[dict[str, Any]]:\n    grouped: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)\n    for row in profile:\n        grouped[(row[\"prefix\"], row[\"family\"])].append(row)\n    result_by_id = {r[\"id\"]: r for r in results or []}\n    summary = []\n    for (prefix, family), items in sorted(grouped.items()):\n        base = {\n            \"prefix\": prefix,\n            \"family\": family,\n            \"rows\": len(items),\n            \"yes_no\": sum(1 for x in items if x[\"gold_answer_type\"] == \"yes_no\"),\n            \"numeric\": sum(1 for x in items if x[\"gold_answer_type\"] == \"numeric\"),\n            \"text\": sum(1 for x in items if x[\"gold_answer_type\"] == \"text\"),\n            \"math_expression\": sum(1 for x in items if x[\"gold_answer_type\"] == \"math_expression\"),\n            \"needs_geometry_relation\": sum(int(x[\"needs_geometry_relation\"]) for x in items),\n            \"needs_vector_composition\": sum(int(x[\"needs_vector_composition\"]) for x in items),\n            \"needs_circuit_relation\": sum(int(x[\"needs_circuit_relation\"]) for x in items),\n            \"needs_measurement_error\": sum(int(x[\"needs_measurement_error\"]) for x in items),\n        }\n        if results is not None:\n            subset_results = [result_by_id[x[\"id\"]] for x in items if x[\"id\"] in result_by_id]\n            base.update(\n                {\n                    \"ok\": sum(1 for r in subset_results if r[\"status\"] == \"ok\"),\n                    \"wrong\": sum(1 for r in subset_results if r[\"status\"] == \"wrong\"),\n                    \"unsupported\": sum(1 for r in subset_results if r[\"status\"] == \"unsupported\"),\n                    \"validator_failed\": sum(1 for r in subset_results if r[\"status\"] == \"validator_failed\"),\n                    \"executor_failed\": sum(1 for r in subset_results if r[\"status\"] == \"executor_failed\"),\n                }\n            )\n        summary.append(base)\n    return summary\n\n\ndef write_csv(path: Path, rows: list[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        path.write_text(\"\", encoding=\"utf-8\")\n        return\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef write_failure_taxonomy(path: Path, profile: list[dict[str, Any]], results: list[dict[str, Any]]) -> None:\n    by_reason: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for row in results:\n        if row[\"status\"] != \"ok\":\n            by_reason[row[\"failure_reason\"] or row[\"status\"]].append(row)\n    status_counts = Counter(row[\"status\"] for row in results)\n    lines = [\n        \"# Physics Baseline Failure Taxonomy\",\n        \"\",\n        \"## Status Counts\",\n        \"\",\n    ]\n    for status, count in status_counts.most_common():\n        lines.append(f\"- `{status}`: {count}\")\n    lines.extend([\"\", \"## Failure Buckets\", \"\"])\n    for reason, items in sorted(by_reason.items(), key=lambda kv: (-len(kv[1]), kv[0])):\n        lines.append(f\"### {reason}\")\n        lines.append(\"\")\n        lines.append(f\"Rows: {len(items)}\")\n        lines.append(\"\")\n        for item in items[:8]:\n            lines.append(\n                f\"- `{item['id']}` family=`{item['family']}` gold=`{item['gold_answer']} {item['gold_unit']}` \"\n                f\"pred=`{item['pred_answer']} {item['pred_unit']}`\"\n            )\n        lines.append(\"\")\n    lines.extend(\n        [\n            \"## Known Architectural Bottlenecks\",\n            \"\",\n            \"- Free-form PoT can produce executable but physically wrong formulas; keep it behind IR validation.\",\n            \"- Geometry facts such as right angle, between, and included angle must come from deterministic relations when possible.\",\n            \"- Unsupported rows are explicit baseline gaps, not silent model guesses.\",\n            \"\",\n        ]\n    )\n    path.write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n\n\ndef compact_snippet(text: str, limit: int = 180) -> str:\n    snippet = re.sub(r\"\\s+\", \" \", text or \"\").strip()\n    if len(snippet) <= limit:\n        return snippet\n    return snippet[: limit - 3].rstrip() + \"...\"\n\n\ndef trace_summary(trace_text: str) -> str:\n    try:\n        trace = json.loads(trace_text or \"{}\")\n    except json.JSONDecodeError:\n        return compact_snippet(trace_text, 120)\n    keys = []\n    for key in [\n        \"target\",\n        \"primitive\",\n        \"formula\",\n        \"relations\",\n        \"sources\",\n        \"missing\",\n        \"value_si\",\n        \"resultant_si\",\n        \"reason\",\n    ]:\n        if key in trace:\n            keys.append(f\"{key}={compact_snippet(json.dumps(trace[key], ensure_ascii=False), 70)}\")\n    return \" | \".join(keys[:5])\n\n\ndef review_bucket(record: dict[str, Any], row: dict[str, str], profile_row: dict[str, Any] | None = None) -> str:\n    text = normalize_text(row.get(\"question\", \"\")).lower()\n    family = record.get(\"family\", \"\")\n    status = record.get(\"status\", \"\")\n    reason = record.get(\"failure_reason\", \"\")\n\n    if status == \"wrong\":\n        if family == \"capacitor\":\n            if \"%\" in row.get(\"answer\", \"\") or \"reduction\" in text or \"percentage\" in text:\n                return \"capacitor_percentage_or_change_target\"\n            if \"uncharged\" in text or \"shared\" in text or \"connected to another\" in text:\n                return \"capacitor_charge_sharing\"\n            if \"halved\" in text or \"doubled\" in text or \"dielectric\" in text:\n                return \"capacitor_state_change\"\n            if \"voltage\" in text and \"capacitance\" in text:\n                return \"capacitor_voltage_capacitance_target_ambiguity\"\n            return \"capacitor_target_or_formula\"\n        if family in {\"ac_rlc\", \"rlc_resonance_yes_no\"}:\n            if \"power factor\" in text:\n                return \"rlc_power_factor_target\"\n            if re.search(r\"\\bpower\\b\", text):\n                return \"rlc_power_target\"\n            if \"impedance\" in text:\n                return \"rlc_impedance_target\"\n            if \"current\" in text:\n                return \"rlc_current_target\"\n            return \"rlc_target_selection\"\n        if family == \"measurement\":\n            if \"least count\" in text or \"smallest division\" in text:\n                return \"measurement_least_count\"\n            if any(tok in text for tok in [\"relative error\", \"percentage error\", \"absolute error\"]):\n                return \"measurement_error_contract\"\n            return \"measurement_output_contract\"\n        if family in {\"electrostatics_force\", \"electrostatics_field\"}:\n            return \"electrostatics_target_geometry_or_vector\"\n        return f\"{family}_wrong\"\n\n    if status == \"validator_failed\":\n        if any(tok in text for tok in [\"midpoint\", \"middle\"]):\n            return \"geometry_midpoint_grounding\"\n        if any(tok in text for tok in [\"perpendicular bisector\", \"equidistant\"]):\n            return \"geometry_perpendicular_bisector_or_equidistant\"\n        if \"equilateral\" in text:\n            return \"geometry_equilateral_grounding\"\n        if \"right\" in text or \"perpendicular\" in text:\n            return \"geometry_right_triangle_or_perpendicular\"\n        if profile_row and int(profile_row.get(\"needs_geometry_relation\", 0)):\n            return \"geometry_relation_missing\"\n        return \"validator_grounding_missing\"\n\n    if status == \"unsupported\":\n        if family in {\"electrostatics_force\", \"electrostatics_field\"}:\n            if \"midpoint\" in text:\n                return \"electrostatics_midpoint_target\"\n            if \"equilateral\" in text:\n                return \"electrostatics_equilateral_vertices\"\n            if \"perpendicular bisector\" in text or \"away from\" in text:\n                return \"electrostatics_perpendicular_distance\"\n            if \"test charge\" in text or \"charge q\" in text or \"at c\" in text:\n                return \"electrostatics_charge_target_mapping\"\n            return \"electrostatics_relation_or_vector_gap\"\n        if family in {\"ac_rlc\", \"rlc_resonance_yes_no\"}:\n            if \"power factor\" in text:\n                return \"rlc_power_factor\"\n            if \"impedance\" in text:\n                return \"rlc_impedance\"\n            if \"resonance\" in text:\n                return \"rlc_resonance\"\n            if \"current\" in text:\n                return \"rlc_current\"\n            if re.search(r\"\\bpower\\b\", text):\n                return \"rlc_power\"\n            return \"rlc_formula_family\"\n        if family == \"measurement\":\n            if \"least count\" in text:\n                return \"measurement_least_count\"\n            if any(tok in text for tok in [\"relative error\", \"percentage error\", \"absolute error\"]):\n                return \"measurement_error_propagation\"\n            return \"measurement_statistic_or_contract\"\n        if family == \"capacitor\":\n            if \"dielectric\" in text:\n                return \"capacitor_dielectric_state\"\n            if \"series\" in text or \"parallel\" in text:\n                return \"capacitor_network\"\n            if \"energy\" in text:\n                return \"capacitor_energy_state\"\n            return \"capacitor_target_or_relation\"\n        if family == \"magnetic_induction\":\n            return \"magnetic_induction_primitives\"\n        if family == \"energy_power\":\n            return \"energy_power_primitives\"\n        return f\"{family}_unsupported\"\n\n    return reason or status or \"unclassified\"\n\n\ndef build_review_rows(\n    source_rows: list[dict[str, str]],\n    profile: list[dict[str, Any]],\n    results: list[dict[str, Any]],\n    status: str,\n) -> list[dict[str, Any]]:\n    source_by_id = {row[\"id\"]: row for row in source_rows}\n    profile_by_id = {row[\"id\"]: row for row in profile}\n    review_rows = []\n    for record in results:\n        if record[\"status\"] != status:\n            continue\n        source = source_by_id.get(record[\"id\"], {})\n        profile_row = profile_by_id.get(record[\"id\"])\n        review_rows.append(\n            {\n                \"id\": record[\"id\"],\n                \"prefix\": record[\"prefix\"],\n                \"family\": record[\"family\"],\n                \"answer_type\": record[\"answer_type\"],\n                \"review_bucket\": review_bucket(record, source, profile_row),\n                \"pred_answer\": record[\"pred_answer\"],\n                \"pred_unit\": record[\"pred_unit\"],\n                \"gold_answer\": record[\"gold_answer\"],\n                \"gold_unit\": record[\"gold_unit\"],\n                \"failure_reason\": record[\"failure_reason\"],\n                \"question_snippet\": compact_snippet(source.get(\"question\", \"\")),\n                \"trace_summary\": trace_summary(record.get(\"trace\", \"\")),\n            }\n        )\n    return review_rows\n\n\ndef build_unsupported_clusters(\n    source_rows: list[dict[str, str]],\n    profile: list[dict[str, Any]],\n    results: list[dict[str, Any]],\n) -> list[dict[str, Any]]:\n    source_by_id = {row[\"id\"]: row for row in source_rows}\n    profile_by_id = {row[\"id\"]: row for row in profile}\n    grouped: dict[tuple[str, str, str], list[dict[str, Any]]] = defaultdict(list)\n    for record in results:\n        if record[\"status\"] != \"unsupported\":\n            continue\n        source = source_by_id.get(record[\"id\"], {})\n        bucket = review_bucket(record, source, profile_by_id.get(record[\"id\"]))\n        key = (record[\"family\"], bucket, record[\"failure_reason\"] or \"unsupported\")\n        grouped[key].append(record)\n\n    clusters = []\n    for index, ((family, bucket, reason), items) in enumerate(\n        sorted(grouped.items(), key=lambda kv: (-len(kv[1]), kv[0])), start=1\n    ):\n        sample_ids = [item[\"id\"] for item in items[:8]]\n        sample_questions = [\n            f\"{item['id']}: {compact_snippet(source_by_id.get(item['id'], {}).get('question', ''), 100)}\"\n            for item in items[:3]\n        ]\n        clusters.append(\n            {\n                \"cluster_id\": f\"U{index:03d}\",\n                \"family\": family,\n                \"reasoning_shape\": bucket,\n                \"failure_reason\": reason,\n                \"count\": len(items),\n                \"sample_ids\": \"; \".join(sample_ids),\n                \"sample_gold\": \"; \".join(f\"{item['id']}={item['gold_answer']} {item['gold_unit']}\" for item in items[:5]),\n                \"sample_questions\": \" || \".join(sample_questions),\n            }\n        )\n    return clusters\n\n\ndef write_next_abstractions(\n    path: Path,\n    results: list[dict[str, Any]],\n    wrong_review: list[dict[str, Any]],\n    validator_review: list[dict[str, Any]],\n    unsupported_clusters: list[dict[str, Any]],\n) -> None:\n    status_counts = Counter(row[\"status\"] for row in results)\n    wrong_buckets = Counter(row[\"review_bucket\"] for row in wrong_review)\n    validator_buckets = Counter(row[\"review_bucket\"] for row in validator_review)\n    lines = [\n        \"# V2 Next Abstractions\",\n        \"\",\n        \"This file is generated from `baseline_results.csv`; it is a taxonomy review, not a row-ID solver plan.\",\n        \"\",\n        \"## Current Status\",\n        \"\",\n    ]\n    for status, count in status_counts.most_common():\n        lines.append(f\"- `{status}`: {count}\")\n\n    lines.extend([\"\", \"## Wrong Rows First\", \"\"])\n    if wrong_buckets:\n        for bucket, count in wrong_buckets.most_common():\n            samples = [row[\"id\"] for row in wrong_review if row[\"review_bucket\"] == bucket][:6]\n            lines.append(f\"- `{bucket}`: {count} rows; samples: {', '.join(samples)}\")\n    else:\n        lines.append(\"- No wrong rows in this run.\")\n\n    lines.extend([\"\", \"## Validator Failures\", \"\"])\n    if validator_buckets:\n        for bucket, count in validator_buckets.most_common():\n            samples = [row[\"id\"] for row in validator_review if row[\"review_bucket\"] == bucket][:6]\n            lines.append(f\"- `{bucket}`: {count} rows; samples: {', '.join(samples)}\")\n    else:\n        lines.append(\"- No validator failures in this run.\")\n\n    lines.extend([\"\", \"## Unsupported Clusters To Unlock\", \"\"])\n    for cluster in unsupported_clusters[:20]:\n        lines.append(\n            f\"- `{cluster['cluster_id']}` `{cluster['reasoning_shape']}`: {cluster['count']} rows \"\n            f\"({cluster['family']}); samples: {cluster['sample_ids']}\"\n        )\n\n    lines.extend(\n        [\n            \"\",\n            \"## Implementation Guardrails\",\n            \"\",\n            \"- Keep Qwen outside trusted execution until the controlled IR schema is stable.\",\n            \"- Add primitives by reasoning shape, not by row ID.\",\n            \"- Every newly solved row should expose target, primitive, grounded observations, relations, and formula in `trace`.\",\n            \"- Wrong rows are higher priority than raising raw coverage.\",\n            \"\",\n        ]\n    )\n    path.write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n\n\ndef write_taxonomy_reviews(output_dir: Path, source_rows: list[dict[str, str]], profile: list[dict[str, Any]], results: list[dict[str, Any]]) -> None:\n    wrong_review = build_review_rows(source_rows, profile, results, \"wrong\")\n    validator_review = build_review_rows(source_rows, profile, results, \"validator_failed\")\n    unsupported_clusters = build_unsupported_clusters(source_rows, profile, results)\n\n    write_csv(output_dir / \"wrong_review.csv\", wrong_review)\n    write_csv(output_dir / \"validator_failed_review.csv\", validator_review)\n    write_csv(output_dir / \"unsupported_clusters.csv\", unsupported_clusters)\n    write_next_abstractions(output_dir / \"next_abstractions.md\", results, wrong_review, validator_review, unsupported_clusters)\n\n\ndef run_dataset(data_path: Path, output_dir: Path) -> dict[str, Any]:\n    with data_path.open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as f:\n        rows = list(csv.DictReader(f))\n    profile = build_dataset_profile(rows)\n    results = []\n    for row in rows:\n        try:\n            result = route_and_solve(row)\n        except Exception as exc:  # keep dataset runs total-order and auditable\n            result = SolveResult(status=\"executor_failed\", failure_reason=repr(exc), family=classify_family(row.get(\"question\", \"\")))\n        if result.status == \"ok\":\n            expected_type = classify_answer_type(row.get(\"answer\", \"\"), row.get(\"unit\", \"\"))\n            match, match_method = compare_result(\n                result.pred_answer,\n                result.pred_unit,\n                row.get(\"answer\", \"\"),\n                row.get(\"unit\", \"\"),\n                expected_type,\n            )\n            if not match:\n                result.status = \"wrong\"\n                result.failure_reason = f\"prediction did not match ground truth ({match_method})\"\n            result.answer_type = expected_type\n        record = {\n            \"id\": row[\"id\"],\n            \"prefix\": row_prefix(row[\"id\"]),\n            \"family\": result.family,\n            \"answer_type\": result.answer_type,\n            \"pred_answer\": result.pred_answer,\n            \"pred_unit\": result.pred_unit,\n            \"gold_answer\": row.get(\"answer\", \"\"),\n            \"gold_unit\": row.get(\"unit\", \"\"),\n            \"status\": result.status,\n            \"failure_reason\": result.failure_reason,\n            \"trace\": json.dumps(result.trace, ensure_ascii=False, sort_keys=True),\n        }\n        results.append(record)\n    summary = build_family_summary(profile, results)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    write_csv(output_dir / \"dataset_profile.csv\", profile)\n    write_csv(output_dir / \"baseline_results.csv\", results)\n    write_csv(output_dir / \"family_summary.csv\", summary)\n    write_failure_taxonomy(output_dir / \"failure_taxonomy.md\", profile, results)\n    write_taxonomy_reviews(output_dir, rows, profile, results)\n    return {\n        \"rows\": len(rows),\n        \"status_counts\": Counter(r[\"status\"] for r in results),\n        \"output_dir\": str(output_dir),\n    }\n\n\ndef assert_close(value: float, expected: float, rel: float = 1e-3) -> None:\n    if abs(value - expected) > max(1e-9, rel * abs(expected)):\n        raise AssertionError(f\"{value} != {expected}\")\n\n\ndef self_test() -> None:\n    assert_close(parse_number(\"3 × 10^-8\") or 0, 3e-8)\n    assert_close(parse_number(\"9√3 × 10^-27\") or 0, 9 * math.sqrt(3) * 1e-27)\n    assert normalize_unit(\"µF\") == \"μF\"\n    assert classify_answer_type(\"Yes\", \"-\") == \"yes_no\"\n    assert classify_answer_type(\"Towards q2\", \"-\") == \"text\"\n    assert classify_answer_type(\"E = 3/2E1\", \"-\") == \"math_expression\"\n\n    rows = {}\n    with resolve_data_path(DEFAULT_DATA_PATH).open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as f:\n        for row in csv.DictReader(f):\n            rows.setdefault(row[\"id\"], row)\n    golden_ids = [\n        \"TD401\",\n        \"TD402\",\n        \"LD004\",\n        \"LD002\",\n        \"CHLT006\",\n        \"THCB088\",\n        \"LD001\",\n        \"LD005\",\n        \"LD060\",\n        \"TD373\",\n        \"TD377\",\n        \"CH041\",\n        \"CH167\",\n        \"DDT321\",\n        \"THCB001\",\n        \"THCB003\",\n        \"TD003\",\n        \"TD004\",\n        \"TD006\",\n        \"NL091\",\n        \"NL367\",\n        \"NL379\",\n    ]\n    for rid in golden_ids:\n        result = route_and_solve(rows[rid])\n        if result.status != \"ok\":\n            raise AssertionError(f\"{rid} did not solve: {result.status} {result.failure_reason}\")\n        expected_type = classify_answer_type(rows[rid][\"answer\"], rows[rid][\"unit\"])\n        ok, method = compare_result(result.pred_answer, result.pred_unit, rows[rid][\"answer\"], rows[rid][\"unit\"], expected_type)\n        if not ok:\n            raise AssertionError(f\"{rid} mismatch by {method}: got {result.pred_answer} {result.pred_unit}\")\n\n    # Direct check for the bottleneck sample: AC must be sqrt(BC^2 - AB^2), not sqrt(AB^2 + BC^2).\n    ld002 = route_and_solve(rows[\"LD002\"])\n    assert_close(float(ld002.pred_answer), 0.02445, rel=5e-3)\n    print(\"self-test ok\")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=\"Run baseline physics dataset audit/solver.\")\n    parser.add_argument(\"data_path\", nargs=\"?\", default=str(DEFAULT_DATA_PATH))\n    parser.add_argument(\"--output-dir\", default=str(DEFAULT_OUTPUT_DIR))\n    parser.add_argument(\"--self-test\", action=\"store_true\")\n    args = parser.parse_args()\n    if args.self_test:\n        self_test()\n        return\n    info = run_dataset(resolve_data_path(args.data_path), Path(args.output_dir))\n    print(json.dumps({\"rows\": info[\"rows\"], \"status_counts\": dict(info[\"status_counts\"]), \"output_dir\": info[\"output_dir\"]}, ensure_ascii=False, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n"
# Kaggle /kaggle/input is read-only. Always materialize the embedded solver in /kaggle/working.
solver_file = Path("/kaggle/working/physics_baseline_solver.py")
solver_file.parent.mkdir(parents=True, exist_ok=True)
solver_file.write_text(SOLVER_CODE, encoding="utf-8")

spec = importlib.util.spec_from_file_location("physics_baseline_solver", solver_file)
pbs = importlib.util.module_from_spec(spec)
sys.modules["physics_baseline_solver"] = pbs
spec.loader.exec_module(pbs)
print("Loaded embedded deterministic solver:", solver_file)


# ============================================================
# Planner/executor fallback
# ============================================================
# Stage order:
# 1. deterministic_solver: high-confidence symbolic/rule solver.
# 2. planner_executor: Qwen emits a JSON calculation plan; Python validates and executes it.
# 3. direct_fallback: Qwen answers directly only if the plan cannot be executed after repair.

HYBRID_USE_PLANNER_EXECUTOR = True
PLANNER_REPAIR_MAX_RETRIES = 2
PLANNER_MAX_NEW_TOKENS = 1200
DIRECT_FALLBACK_MAX_NEW_TOKENS = GEN_MAX_NEW_TOKENS

try:
    import sympy as sp
    from sympy.parsing.sympy_parser import (
        parse_expr,
        standard_transformations,
        implicit_multiplication_application,
        convert_xor,
    )
    PLANNER_EXECUTOR_AVAILABLE = True
    PARSE_TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)
except Exception as exc:
    PLANNER_EXECUTOR_AVAILABLE = False
    PLANNER_IMPORT_ERROR = repr(exc)
    print("Planner executor disabled because SymPy import failed:", PLANNER_IMPORT_ERROR)


PHYSICS_FORMULA_BANK = """
Use these as retrieval hints, not as mandatory rules.
Electrostatics:
- Coulomb force magnitude: F = k*abs(q1*q2)/r**2.
- Electric field of a point charge: E = k*abs(q)/r**2.
- Force on charge q in field E: F = abs(q)*E.
- Superposition: add vector components: Fx = sum(Fi*cos(theta_i)), Fy = sum(Fi*sin(theta_i)), F = sqrt(Fx**2 + Fy**2).
- Electric-field zero on a line: solve k*abs(q1)/x**2 = k*abs(q2)/(d-x)**2 or the outside-region equivalent.
Capacitors/LC:
- Parallel-plate capacitance: C = eps0*epsr*S/d.
- Capacitor charge: Q = C*U.
- Energy: W = 0.5*C*U**2 = Q**2/(2*C) = 0.5*Q*U.
- LC energy: W = 0.5*C*Umax**2 = 0.5*L*Imax**2.
AC/RLC:
- XL = omega*L, XC = 1/(omega*C), omega = 2*pi*f.
- Series impedance: Z = sqrt(R**2 + (XL-XC)**2).
- Current: I = U/Z. Power: P = I**2*R = U*I*cos_phi.
- Power factor: cos_phi = R/Z.
- Resonance: XL = XC, omega0 = 1/sqrt(L*C), f0 = 1/(2*pi*sqrt(L*C)).
Magnetism/induction:
- Solenoid field: B = mu0*n*I = mu0*N*I/l.
- Magnetic flux: Phi = B*S. Flux linkage: lambda = N*Phi.
- Induced emf magnitude: e = abs(dPhi/dt), or e = L*abs(dI/dt).
- Inductor energy: W = 0.5*L*I**2.
Measurement:
- Absolute error = abs(measured - true). Relative error = absolute_error/abs(true). Percentage error = 100*relative_error.
"""


def select_formula_hints(question, limit=1800):
    q = str(question).lower()
    chunks = []
    for block in PHYSICS_FORMULA_BANK.strip().splitlines():
        lower = block.lower()
        if block.endswith(":"):
            current_header = block
            continue
        if (
            ("charge" in q or "electric field" in q or "force" in q or "coulomb" in q) and any(x in lower for x in ["coulomb", "electric", "superposition", "zero"])
        ) or (
            ("capacitor" in q or "capacitance" in q or "lc" in q) and any(x in lower for x in ["capacit", "energy", "lc"])
        ) or (
            ("rlc" in q or "resonance" in q or "reactance" in q or "impedance" in q or "power factor" in q) and any(x in lower for x in ["xl", "series", "current", "power", "resonance"])
        ) or (
            ("solenoid" in q or "magnetic" in q or "flux" in q or "induced" in q or "inductor" in q) and any(x in lower for x in ["solenoid", "flux", "emf", "inductor"])
        ) or (
            ("error" in q or "measured" in q or "measurement" in q) and "error" in lower
        ):
            chunks.append(block)
    if not chunks:
        chunks = PHYSICS_FORMULA_BANK.strip().splitlines()
    return "\n".join(chunks)[:limit]


def generate_chat_text(system_content, user_content, max_new_tokens):
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


def extract_json_object(text):
    text = str(text or "").strip()
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S | re.I)
    if fence:
        text = fence.group(1)
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object found in model output.")
    depth = 0
    in_string = False
    escape = False
    for idx in range(start, len(text)):
        ch = text[idx]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:idx + 1]
    raise ValueError("Unbalanced JSON object in model output.")


def load_plan_from_text(text):
    raw = extract_json_object(text)
    return json.loads(raw)


def planner_error(stage, error_type, message, step_id=None, missing_symbols=None, unresolved_symbols=None):
    return {
        "stage": stage,
        "error_type": error_type,
        "message": message,
        "step_id": step_id,
        "missing_symbols": sorted(list(missing_symbols or [])),
        "unresolved_symbols": sorted(list(unresolved_symbols or [])),
    }


def error_signature(error):
    return (
        str(error.get("stage")),
        str(error.get("error_type")),
        str(error.get("step_id")),
        tuple(error.get("missing_symbols") or []),
        tuple(error.get("unresolved_symbols") or []),
    )


def planner_system_prompt():
    return """You are a physics planner for a neuro-symbolic solver.
Return JSON only. Do not include markdown.
Use ASCII variable names only.
For numeric and boolean problems, create executable equation/compare steps.
For text or symbolic_expression problems, set answer_type accordingly and include final_answer.
Use SI-compatible equations. The Python executor converts givens to SI.
Use only these step types:
- equation: {"id":"s1","type":"equation","equation":"U = I*R","solves_for":"U","unit":"V"}
- compare: {"id":"s2","type":"compare","left":"XL","operator":"approximately_equal","right":"XC","output":"resonance"}
Allowed answer_type: numeric, boolean, symbolic_expression, text.
Schema:
{
  "problem_type": "short topic",
  "answer_type": "numeric | boolean | symbolic_expression | text",
  "target": {"name": "string", "symbol": "string", "unit": "string"},
  "givens": [{"name": "string", "symbol": "string", "value": number_or_string, "unit": "string"}],
  "steps": [{"id": "s1", "type": "equation | compare", "...": "..."}],
  "qualitative_reasoning": ["optional"],
  "final_answer": "optional for text/symbolic_expression"
}"""


def build_planner_user_prompt(row, repair_context=None):
    question = str(row.get("question", "")).strip()
    hints = select_formula_hints(question)
    parts = [
        "Question:",
        question,
        "",
        "Formula hints:",
        hints,
        "",
        "Make an executable plan. Do not skip intermediate variables.",
        "If geometry/vector composition is needed, express it with ordinary equations using components or cosine law.",
    ]
    if repair_context:
        parts.extend([
            "",
            "Previous plan failed. Repair it without changing the target.",
            "Structured error:",
            json.dumps(repair_context.get("error", {}), ensure_ascii=False),
            "Previous plan:",
            json.dumps(repair_context.get("plan", {}), ensure_ascii=False),
        ])
    return "\n".join(parts)


def normalize_plan_answer_type(answer_type):
    raw = str(answer_type or "numeric").strip().lower()
    aliases = {"yes_no": "boolean", "bool": "boolean", "symbolic": "symbolic_expression", "math_expression": "symbolic_expression"}
    return aliases.get(raw, raw)


PLANNER_FUNCTIONS = {
    "sqrt", "sin", "cos", "tan", "asin", "acos", "atan", "abs", "Abs", "pi", "exp", "log"
}
PLANNER_CONSTANTS = {
    "k": 9e9,
    "eps0": 8.854e-12,
    "epsilon0": 8.854e-12,
    "mu0": 4 * math.pi * 1e-7,
    "pi": math.pi,
    "g": 9.8,
}


def names_in_expr(expr):
    names = set(re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", str(expr)))
    return names - PLANNER_FUNCTIONS


def normalize_equation_text(expr):
    expr = str(expr or "").strip()
    expr = expr.replace("×", "*").replace("·", "*").replace("^", "**")
    expr = expr.replace("−", "-").replace("π", "pi")
    expr = re.sub(r"\bAbs\s*\(", "abs(", expr)
    return expr


def parse_planner_expr(expr, symbol_names):
    expr = normalize_equation_text(expr)
    local_dict = {name: sp.Symbol(name) for name in symbol_names if name not in PLANNER_CONSTANTS}
    local_dict.update({
        "sqrt": sp.sqrt,
        "sin": sp.sin,
        "cos": sp.cos,
        "tan": sp.tan,
        "asin": sp.asin,
        "acos": sp.acos,
        "atan": sp.atan,
        "abs": sp.Abs,
        "Abs": sp.Abs,
        "exp": sp.exp,
        "log": sp.log,
        "pi": sp.pi,
    })
    return parse_expr(expr, local_dict=local_dict, transformations=PARSE_TRANSFORMS)


def coerce_given_value(value):
    if isinstance(value, (int, float)) and math.isfinite(float(value)):
        return float(value)
    parsed = pbs.parse_number(str(value))
    if parsed is None:
        raise ValueError(f"Cannot parse numeric given value: {value!r}")
    return float(parsed)


def load_plan_state(plan):
    state = dict(PLANNER_CONSTANTS)
    units = {name: "-" for name in PLANNER_CONSTANTS}
    givens = plan.get("givens") or []
    if not isinstance(givens, list):
        raise ValueError("givens must be a list")
    for item in givens:
        symbol = str(item.get("symbol", "")).strip()
        if not symbol or not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", symbol):
            raise ValueError(f"Invalid given symbol: {symbol!r}")
        unit = pbs.normalize_unit(str(item.get("unit", "-")))
        value = coerce_given_value(item.get("value"))
        value_si, unit_si = pbs.unit_to_si(value, unit)
        state[symbol] = value_si
        units[symbol] = unit_si
    return state, units


def evaluate_expr_numeric(expr, state, extra_names=None):
    symbol_names = set(state) | set(extra_names or []) | names_in_expr(expr)
    parsed = parse_planner_expr(expr, symbol_names)
    subs = {sp.Symbol(k): v for k, v in state.items() if isinstance(v, (int, float))}
    value = sp.N(parsed.subs(subs))
    unresolved = {str(s) for s in getattr(value, "free_symbols", set())}
    if unresolved:
        raise ValueError(f"Unresolved symbols: {sorted(unresolved)}")
    return float(value)


def execute_equation_step(step, state, units):
    step_id = step.get("id")
    equation = normalize_equation_text(step.get("equation", ""))
    target = str(step.get("solves_for", "")).strip()
    if not equation or "=" not in equation:
        return None, planner_error("executor", "invalid_equation", "Equation step must contain '='.", step_id)
    if not target:
        return None, planner_error("executor", "missing_target", "Equation step lacks solves_for.", step_id)
    lhs_text, rhs_text = equation.split("=", 1)
    all_names = names_in_expr(lhs_text) | names_in_expr(rhs_text) | {target} | set(state)
    if target not in all_names:
        return None, planner_error("plan_validation", "target_not_in_equation", "solves_for symbol is not in equation.", step_id)
    missing = (all_names - {target} - set(state) - set(PLANNER_CONSTANTS))
    if missing:
        return None, planner_error("plan_validation", "missing_symbol", "Equation uses symbols not given or computed.", step_id, missing_symbols=missing)
    try:
        lhs = parse_planner_expr(lhs_text, all_names)
        rhs = parse_planner_expr(rhs_text, all_names)
        target_sym = sp.Symbol(target)
        subs = {sp.Symbol(k): v for k, v in state.items() if isinstance(v, (int, float))}
        equation_obj = sp.Eq(lhs, rhs)
        solutions = sp.solve(equation_obj, target_sym)
        if not solutions:
            return None, planner_error("executor", "no_solution", "SymPy could not solve equation for target.", step_id)
        numeric_solutions = []
        for candidate in solutions:
            value = sp.N(candidate.subs(subs))
            unresolved = {str(s) for s in getattr(value, "free_symbols", set())}
            if unresolved:
                continue
            try:
                numeric_solutions.append(float(value))
            except Exception:
                pass
        if not numeric_solutions:
            unresolved = set()
            for candidate in solutions:
                unresolved |= {str(s) for s in getattr(candidate.subs(subs), "free_symbols", set())}
            return None, planner_error("executor", "unresolved_solution", "Solution still has unresolved symbols.", step_id, unresolved_symbols=unresolved)
        chosen = next((x for x in numeric_solutions if x >= 0), numeric_solutions[0])
        state[target] = chosen
        units[target] = pbs.normalize_unit(str(step.get("unit", units.get(target, "-"))))
        return {"symbol": target, "value": chosen, "unit": units[target]}, None
    except Exception as exc:
        return None, planner_error("executor", "equation_execution_error", repr(exc), step_id)


def execute_compare_step(step, state, units):
    step_id = step.get("id")
    left = step.get("left", "")
    right = step.get("right", "")
    op = str(step.get("operator", "approximately_equal")).strip().lower()
    output = str(step.get("output", "comparison")).strip()
    try:
        left_val = evaluate_expr_numeric(left, state)
        right_val = evaluate_expr_numeric(right, state)
        tol = max(1e-6, 0.03 * max(abs(left_val), abs(right_val), 1.0))
        if op in {"approximately_equal", "approx_equal", "equal", "=="}:
            result = abs(left_val - right_val) <= tol
        elif op in {"not_equal", "!="}:
            result = abs(left_val - right_val) > tol
        elif op in {"greater_than", ">"}:
            result = left_val > right_val
        elif op in {"less_than", "<"}:
            result = left_val < right_val
        elif op in {"greater_equal", ">="}:
            result = left_val >= right_val
        elif op in {"less_equal", "<="}:
            result = left_val <= right_val
        else:
            return None, planner_error("executor", "unknown_compare_operator", f"Unsupported compare operator: {op}", step_id)
        state[output] = bool(result)
        units[output] = "-"
        return {"symbol": output, "value": bool(result), "unit": "-"}, None
    except Exception as exc:
        missing = (names_in_expr(left) | names_in_expr(right)) - set(state) - set(PLANNER_CONSTANTS)
        return None, planner_error("executor", "compare_execution_error", repr(exc), step_id, missing_symbols=missing)


def format_planner_numeric(value_si, unit):
    unit = pbs.normalize_unit(str(unit or "-"))
    if unit in {"", "-", "nan", "None"}:
        return pbs.format_number(float(value_si)), "-"
    try:
        value = pbs.convert_from_si(float(value_si), unit)
    except Exception:
        value = float(value_si)
    return pbs.format_number(value), unit


def execute_plan(plan):
    if not PLANNER_EXECUTOR_AVAILABLE:
        return {"status": "failed", "error": planner_error("executor", "sympy_unavailable", PLANNER_IMPORT_ERROR)}
    if not isinstance(plan, dict):
        return {"status": "failed", "error": planner_error("plan_validation", "invalid_plan", "Plan is not a dict.")}

    answer_type = normalize_plan_answer_type(plan.get("answer_type", "numeric"))
    target = plan.get("target") or {}
    target_symbol = str(target.get("symbol", "")).strip()
    target_unit = pbs.normalize_unit(str(target.get("unit", "-")))

    if answer_type in {"text", "symbolic_expression"}:
        final = str(plan.get("final_answer", "")).strip()
        if final:
            return {
                "status": "ok",
                "source": "planner_direct",
                "pred_answer": final,
                "pred_unit": target_unit if target_unit != "" else "-",
                "answer_type": answer_type,
                "trace": {"plan": plan, "mode": "planner_direct_non_numeric"},
            }
        return {"status": "failed", "error": planner_error("plan_validation", "missing_final_answer", "Text/symbolic plan needs final_answer.")}

    try:
        state, units = load_plan_state(plan)
    except Exception as exc:
        return {"status": "failed", "error": planner_error("plan_validation", "invalid_givens", repr(exc))}

    steps = plan.get("steps") or []
    if not isinstance(steps, list) or not steps:
        return {"status": "failed", "error": planner_error("plan_validation", "missing_steps", "Numeric/boolean plan needs executable steps.")}

    trace_steps = []
    for step in steps:
        step_type = str(step.get("type", "equation")).strip().lower()
        if step_type == "equation":
            step_trace, error = execute_equation_step(step, state, units)
        elif step_type == "compare":
            step_trace, error = execute_compare_step(step, state, units)
        else:
            error = planner_error("plan_validation", "unknown_step_type", f"Unknown step type: {step_type}", step.get("id"))
            step_trace = None
        if error:
            return {"status": "failed", "error": error, "partial_trace": trace_steps}
        trace_steps.append({"id": step.get("id"), "type": step_type, "result": step_trace})

    if answer_type == "boolean":
        symbol = target_symbol or (trace_steps[-1]["result"] or {}).get("symbol", "")
        value = state.get(symbol, None)
        if isinstance(value, bool):
            return {
                "status": "ok",
                "source": "planner_executor",
                "pred_answer": "Yes" if value else "No",
                "pred_unit": "-",
                "answer_type": "yes_no",
                "trace": {"plan": plan, "steps": trace_steps},
            }
        return {"status": "failed", "error": planner_error("result_validation", "boolean_target_missing", "Boolean target was not produced.")}

    if not target_symbol:
        target_symbol = (trace_steps[-1]["result"] or {}).get("symbol", "")
    if target_symbol not in state:
        return {"status": "failed", "error": planner_error("result_validation", "target_missing", "Target was not produced.", missing_symbols={target_symbol})}

    answer, unit = format_planner_numeric(state[target_symbol], target_unit or units.get(target_symbol, "-"))
    return {
        "status": "ok",
        "source": "planner_executor",
        "pred_answer": answer,
        "pred_unit": unit,
        "answer_type": "numeric",
        "trace": {"plan": plan, "steps": trace_steps, "target_value_si": state[target_symbol]},
    }


def run_planner_executor(row):
    if not HYBRID_USE_PLANNER_EXECUTOR:
        return {"status": "skipped", "error": planner_error("planner", "disabled", "Planner executor disabled.")}
    previous_signature = None
    repair_context = None
    generated_history = []
    last_error = None
    last_plan = None

    for attempt in range(PLANNER_REPAIR_MAX_RETRIES + 1):
        generated = generate_chat_text(
            planner_system_prompt(),
            build_planner_user_prompt(row, repair_context),
            max_new_tokens=PLANNER_MAX_NEW_TOKENS,
        )
        generated_history.append(generated)
        try:
            plan = load_plan_from_text(generated)
        except Exception as exc:
            plan = {}
            result = {"status": "failed", "error": planner_error("planner", "invalid_json", repr(exc))}
        else:
            last_plan = plan
            result = execute_plan(plan)

        if result.get("status") == "ok":
            result["attempts"] = attempt + 1
            result["generated_history"] = generated_history
            return result

        last_error = result.get("error") or planner_error("executor", "unknown_error", "Planner execution failed.")
        sig = error_signature(last_error)
        if previous_signature == sig:
            last_error = dict(last_error)
            last_error["circuit_breaker"] = True
            break
        previous_signature = sig
        repair_context = {"error": last_error, "plan": last_plan or {}}

    return {
        "status": "failed",
        "error": last_error or planner_error("planner", "unknown_error", "Planner failed."),
        "attempts": len(generated_history),
        "generated_history": generated_history,
        "last_plan": last_plan,
    }


def build_direct_fallback_prompt(row, planner_failure=None):
    question = str(row.get("question", "")).strip()
    hints = select_formula_hints(question)
    error_text = ""
    if planner_failure:
        error_text = "\nThe symbolic planner failed with this structured error. Avoid repeating it:\n" + json.dumps(
            planner_failure.get("error", planner_failure), ensure_ascii=False
        )
    messages = [
        {
            "role": "system",
            "content": ANSWER_ONLY_SYSTEM_PROMPT + "\nUse concise Program-of-Thought internally, but output only `Final answer: <value> <unit>`.",
        },
        {
            "role": "user",
            "content": f"Question:\n{question}\n\nUseful formula hints:\n{hints}{error_text}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_from_prompt(prompt, max_new_tokens):
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


def hybrid_extract_final_answer(generated_text):
    text = str(generated_text).strip()
    matches = re.findall(r"final answer\s*:\s*(.+)", text, flags=re.I)
    if matches:
        return matches[-1].strip().splitlines()[0].strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else text


def format_pred_answer(value, unit):
    unit = str(unit or "-").strip()
    if unit in {"", "-", "nan", "None"}:
        return str(value)
    return f"{value} {unit}"


def metric_answer_unit_flags(pair_ok, pred_final, pred_answer, pred_unit, row_dict):
    if pair_ok:
        return True, True
    if "answer_matches" in globals():
        answer_ok = answer_matches(pred_final, row_dict.get("answer", ""))
    else:
        answer_ok = False
    if "unit_matches" in globals():
        unit_ok = unit_matches(pred_final, row_dict.get("unit", ""))
    else:
        unit_ok = False
    return bool(answer_ok), bool(unit_ok)


hybrid_source_df = test_df
if HYBRID_EVAL_MAX_SAMPLES is not None:
    hybrid_source_df = test_df.sample(
        n=min(HYBRID_EVAL_MAX_SAMPLES, len(test_df)),
        random_state=SEED,
    ).reset_index(drop=True)

hybrid_rows = []
source_counts = {"deterministic_solver": 0, "planner_executor": 0, "planner_direct": 0, "direct_fallback": 0}

FastLanguageModel.for_inference(model)

for idx, row in hybrid_source_df.iterrows():
    row_dict = row.to_dict()
    expected_type = pbs.classify_answer_type(row_dict.get("answer", ""), row_dict.get("unit", ""))
    solver_result = pbs.route_and_solve(row_dict)
    deterministic_status = solver_result.status
    planner_failure = None
    planner_plan = ""
    planner_error_text = ""
    planner_attempts = 0

    if solver_result.status == "ok":
        source = "deterministic_solver"
        generated = ""
        pred_answer = solver_result.pred_answer
        pred_unit = solver_result.pred_unit
        pred_final = format_pred_answer(pred_answer, pred_unit)
    else:
        planner_result = run_planner_executor(row_dict)
        planner_attempts = planner_result.get("attempts", 0)
        planner_failure = planner_result if planner_result.get("status") != "ok" else None
        if planner_result.get("trace", {}).get("plan") is not None:
            planner_plan = json.dumps(planner_result["trace"]["plan"], ensure_ascii=False)
        elif planner_result.get("last_plan") is not None:
            planner_plan = json.dumps(planner_result["last_plan"], ensure_ascii=False)

        if planner_result.get("status") == "ok":
            source = planner_result.get("source", "planner_executor")
            generated = "\n\n".join(planner_result.get("generated_history", []))
            pred_answer = planner_result.get("pred_answer", "")
            pred_unit = planner_result.get("pred_unit", "-")
            pred_final = format_pred_answer(pred_answer, pred_unit)
        else:
            source = "direct_fallback"
            planner_error_text = json.dumps(planner_result.get("error", {}), ensure_ascii=False)
            prompt = build_direct_fallback_prompt(row_dict, planner_result)
            generated = generate_from_prompt(prompt, DIRECT_FALLBACK_MAX_NEW_TOKENS)
            pred_final = hybrid_extract_final_answer(generated)
            pred_answer = pred_final
            pred_unit = "-"

    source_counts[source] = source_counts.get(source, 0) + 1

    pair_ok, match_method = pbs.compare_result(
        str(pred_answer),
        str(pred_unit),
        row_dict.get("answer", ""),
        row_dict.get("unit", ""),
        expected_type,
    )
    answer_ok, unit_ok = metric_answer_unit_flags(pair_ok, pred_final, pred_answer, pred_unit, row_dict)

    hybrid_rows.append({
        "id": row_dict.get("id"),
        "prediction_source": source,
        "deterministic_status": deterministic_status,
        "planner_attempts": planner_attempts,
        "planner_error": planner_error_text,
        "gold_answer": row_dict.get("answer"),
        "gold_unit": row_dict.get("unit"),
        "pred_final_answer": pred_final,
        "answer_ok": bool(answer_ok),
        "unit_ok": bool(unit_ok),
        "pair_ok": bool(pair_ok),
        "match_method": match_method,
        "planner_plan": planner_plan,
        "generated": generated,
    })

    if (idx + 1) % 10 == 0:
        print(f"Hybrid evaluated {idx + 1}/{len(hybrid_source_df)}")

hybrid_results_df = pd.DataFrame(hybrid_rows)
hybrid_results_df.to_csv(HYBRID_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved hybrid eval results:", HYBRID_OUTPUT_PATH)
print("Rows evaluated:", len(hybrid_results_df))
print("Source counts:", source_counts)
print("Answer accuracy:", hybrid_results_df["answer_ok"].mean() if len(hybrid_results_df) else None)
print("Unit accuracy:", hybrid_results_df["unit_ok"].mean() if len(hybrid_results_df) else None)
print("Pair accuracy:", hybrid_results_df["pair_ok"].mean() if len(hybrid_results_df) else None)

display(hybrid_results_df[[
    "id", "prediction_source", "deterministic_status", "planner_attempts",
    "gold_answer", "gold_unit", "pred_final_answer",
    "answer_ok", "unit_ok", "pair_ok", "planner_error"
]].head(100))


Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded embedded deterministic solver: /kaggle/working/physics_baseline_solver.py
Hybrid evaluated 10/270


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of sh

Hybrid evaluated 20/270


Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 1337]) with length 1337 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 1336]) 

Hybrid evaluated 260/270


Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 1092]) with length 1092 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hybrid evaluated 270/270
Saved hybrid eval results: /kaggle/working/qwen25_coder_7b_lora_hybrid_eval_20.csv
Rows evaluated: 270
Source counts: {'deterministic_solver': 259, 'planner_executor': 5, 'planner_direct': 0, 'direct_fallback': 6}
Answer accuracy: 0.9592592592592593
Unit accuracy: 0.9851851851851852
Pair accuracy: 0.9555555555555556


,id,prediction_source,deterministic_status,planner_attempts,gold_answer,gold_unit,pred_final_answer,answer_ok,unit_ok,pair_ok,planner_error
0,TD371,deterministic_solver,ok,0,9,times,9 times,True,True,True,
1,TD386,deterministic_solver,ok,0,decreases by half,-,decreases by half,True,True,True,
2,TD062,deterministic_solver,ok,0,22.74,pF,22.737072 pF,True,True,True,
3,TD053,deterministic_solver,ok,0,60.21,pF,60.2072 pF,True,True,True,
4,TD166,deterministic_solver,ok,0,3.005,nC,3.005056 nC,True,True,True,
...,...,...,...,...,...,...,...,...,...,...,...
95,LD311,deterministic_solver,ok,0,1.76 × 10^6,V/m,1.76323e+06 V/m,True,True,True,
96,LD152,deterministic_solver,ok,0,3.82,N,3.818377 N,True,True,True,
97,LD133,deterministic_solver,ok,0,0.434,N,0.433103 N,True,True,True,
98,LD154,deterministic_solver,ok,0,0.098,N,0.098 N,True,True,True,
